In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

print("Python version:", sys.version)
print("Project directory exists:", ROOT.exists())
print()

for name in [
    "calc_a",
    "calc_b",
    "calc_c",
    "calc_d",
    "calc_e",
    "tests",
    "scripts"
]:
    path = ROOT / name
    print(f"{name:10s} : {path.exists()}")

In [ ]:
import subprocess

cmd = [
    sys.executable,
    "-m",
    "pytest",
    "tests/test_calc_a_historical.py",
    "-q"
]

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

In [ ]:
%pip install pytest pytest-cov coverage pandas matplotlib

In [ ]:
import sys
import pytest
import coverage
import pandas as pd
import matplotlib

print("Python:", sys.version)
print("pytest:", pytest.__version__)
print("coverage:", coverage.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)

In [ ]:
import subprocess
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_calc_a_historical.py",
        "-q"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

coverage_json = ROOT / "coverage_calc_a.json"

if coverage_json.exists():
    coverage_json.unlink()

cmd = [
    sys.executable,
    "-m",
    "pytest",
    "tests/test_calc_a_historical.py",
    "--cov=calc_a",
    "--cov-branch",
    "--cov-report=term-missing",
    f"--cov-report=json:{coverage_json.name}",
    "-q",
]

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)
print("Coverage JSON created:", coverage_json.exists())
print("Coverage JSON path:", coverage_json)

In [ ]:
import json

with open(ROOT / "coverage_calc_a.json", "r", encoding="utf-8") as f:
    cov = json.load(f)

totals = cov["totals"]

print("Covered lines:", totals["covered_lines"])
print("Total statements:", totals["num_statements"])
print("Statement coverage %:", totals["percent_covered"])

print("Covered branches:", totals.get("covered_branches"))
print("Total branches:", totals.get("num_branches"))

if totals.get("num_branches"):
    branch_pct = totals["covered_branches"] / totals["num_branches"] * 100
    print("Branch coverage %:", round(branch_pct, 2))

In [ ]:
import subprocess
import sys
import json
import csv
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")
TEST_FILE = "tests/test_calc_a_historical.py"

# ------------------------------------------------------------
# 1. Collect all Pytest node IDs
# ------------------------------------------------------------
collect = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        TEST_FILE,
        "--collect-only",
        "-q",
    ],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)

node_ids = [
    line.strip()
    for line in collect.stdout.splitlines()
    if "::" in line and not line.startswith("<")
]

print("Collected tests:", len(node_ids))

# ------------------------------------------------------------
# 2. Run each test independently and save JSON coverage
# ------------------------------------------------------------
records = []

for i, node_id in enumerate(node_ids, start=1):
    json_name = f"_tmp_cov_{i:04d}.json"
    json_path = ROOT / json_name

    if json_path.exists():
        json_path.unlink()

    cmd = [
        sys.executable,
        "-m",
        "pytest",
        node_id,
        "--cov=calc_a",
        "--cov-branch",
        "--cov-report=",
        f"--cov-report=json:{json_name}",
        "-q",
    ]

    run = subprocess.run(
        cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
    )

    if run.returncode != 0:
        print("FAILED:", node_id)
        print(run.stdout)
        print(run.stderr)
        raise RuntimeError(f"Test failed during profiling: {node_id}")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # combine coverage from calc_a source files
    covered_lines = set()
    executed_branches = set()

    for filename, filedata in data["files"].items():
        if "calc_a" not in filename.replace("\\", "/"):
            continue

        for line in filedata.get("executed_lines", []):
            covered_lines.add((filename, int(line)))

        for branch in filedata.get("executed_branches", []):
            if len(branch) == 2:
                executed_branches.add(
                    (filename, int(branch[0]), int(branch[1]))
                )

    records.append({
        "test_id": node_id,
        "covered_lines": sorted(covered_lines),
        "covered_branches": sorted(executed_branches),
    })

    json_path.unlink(missing_ok=True)

    if i % 25 == 0 or i == len(node_ids):
        print(f"Profiled {i}/{len(node_ids)} tests")

print("Per-test profiling complete.")

In [ ]:
import subprocess
import sys
import json
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")
TEST_FILE = "tests/test_calc_a_historical.py"

# ---------------------------------------------------------
# Collect all individual Pytest test IDs
# ---------------------------------------------------------
collect = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        TEST_FILE,
        "--collect-only",
        "-q"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True
)

node_ids = [
    line.strip()
    for line in collect.stdout.splitlines()
    if "::" in line and not line.startswith("<")
]

print("Collected tests:", len(node_ids))

# ---------------------------------------------------------
# Execute every test separately and collect its coverage
# ---------------------------------------------------------
records = []

for i, node_id in enumerate(node_ids, start=1):

    temp_json = ROOT / f"_tmp_cov_{i:04d}.json"

    if temp_json.exists():
        temp_json.unlink()

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            node_id,
            "--cov=calc_a",
            "--cov-branch",
            "--cov-report=",
            f"--cov-report=json:{temp_json.name}",
            "-q"
        ],
        cwd=ROOT,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("\nFAILED TEST:")
        print(node_id)
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Per-test coverage profiling stopped.")

    with open(temp_json, "r", encoding="utf-8") as f:
        coverage_data = json.load(f)

    covered_lines = set()
    covered_branches = set()

    for filename, filedata in coverage_data["files"].items():

        normalized = filename.replace("\\", "/")

        if "calc_a/" not in normalized:
            continue

        for line in filedata.get("executed_lines", []):
            covered_lines.add(
                (normalized, int(line))
            )

        for branch in filedata.get("executed_branches", []):
            if len(branch) == 2:
                covered_branches.add(
                    (
                        normalized,
                        int(branch[0]),
                        int(branch[1])
                    )
                )

    records.append({
        "test_id": node_id,
        "covered_lines": sorted(covered_lines),
        "covered_branches": sorted(covered_branches)
    })

    temp_json.unlink(missing_ok=True)

    if i % 25 == 0 or i == len(node_ids):
        print(f"Profiled {i}/{len(node_ids)} tests")

print("\nPer-test profiling complete.")

In [ ]:
all_lines = set()
all_branches = set()

for record in records:
    all_lines.update(record["covered_lines"])
    all_branches.update(record["covered_branches"])

print("Number of tests:", len(records))
print("Unique covered lines:", len(all_lines))
print("Unique covered branches:", len(all_branches))
print(
    "Total structural elements:",
    len(all_lines) + len(all_branches)
)

In [ ]:
import pandas as pd
import json

rows = []

for record in records:
    rows.append({
        "test_id": record["test_id"],
        "n_covered_lines": len(record["covered_lines"]),
        "n_covered_branches": len(record["covered_branches"]),
        "covered_lines": json.dumps(record["covered_lines"]),
        "covered_branches": json.dumps(record["covered_branches"])
    })

df_profile = pd.DataFrame(rows)

output_file = ROOT / "per_test_coverage.csv"
df_profile.to_csv(output_file, index=False)

print("Saved to:", output_file)
print("Number of rows:", len(df_profile))

display(df_profile.head(10))

In [ ]:
TARGET_FILE = "calc_a/functions.py"

clean_records = []

for record in records:

    clean_lines = [
        item for item in record["covered_lines"]
        if item[0].replace("\\", "/").endswith(TARGET_FILE)
    ]

    clean_branches = [
        item for item in record["covered_branches"]
        if item[0].replace("\\", "/").endswith(TARGET_FILE)
    ]

    clean_records.append({
        "test_id": record["test_id"],
        "covered_lines": clean_lines,
        "covered_branches": clean_branches
    })

print("Clean records:", len(clean_records))

In [ ]:
clean_all_lines = set()
clean_all_branches = set()

for record in clean_records:
    clean_all_lines.update(record["covered_lines"])
    clean_all_branches.update(record["covered_branches"])

print("Number of tests:", len(clean_records))
print("Unique implementation lines:", len(clean_all_lines))
print("Unique implementation branches:", len(clean_all_branches))
print(
    "Total implementation structural elements:",
    len(clean_all_lines) + len(clean_all_branches)
)

In [ ]:
import json

with open(ROOT / "coverage_calc_a.json", "r", encoding="utf-8") as f:
    baseline_cov = json.load(f)

for filename, filedata in baseline_cov["files"].items():
    normalized = filename.replace("\\", "/")

    if normalized.endswith("calc_a/functions.py"):
        print("Implementation file:", normalized)
        print("Whole-suite executed lines:",
              len(filedata.get("executed_lines", [])))
        print("Whole-suite executed branches:",
              len(filedata.get("executed_branches", [])))
        print("Missing lines:",
              filedata.get("missing_lines", []))
        print("Missing branches:",
              filedata.get("missing_branches", []))

In [ ]:
clean_all_lines = set()
clean_all_branches = set()

for record in clean_records:
    clean_all_lines.update(record["covered_lines"])
    clean_all_branches.update(record["covered_branches"])

print("Per-test union lines:", len(clean_all_lines))
print("Per-test union branches:", len(clean_all_branches))
print("Total structural elements:",
      len(clean_all_lines) + len(clean_all_branches))

assert len(clean_all_lines) == 21
assert len(clean_all_branches) == 6

print("Verification PASSED")

In [ ]:
# ---------------------------------------------------------
# Greedy coverage-guided test-suite reduction
# ---------------------------------------------------------

# Represent lines and branches as distinct structural elements
def structural_elements(record):
    elements = set()

    for item in record["covered_lines"]:
        elements.add(("LINE",) + tuple(item))

    for item in record["covered_branches"]:
        elements.add(("BRANCH",) + tuple(item))

    return elements


test_coverage = {
    record["test_id"]: structural_elements(record)
    for record in clean_records
}

# Universe of all observed structural elements
universe = set()

for elems in test_coverage.values():
    universe.update(elems)

print("Observed structural universe:", len(universe))

selected_tests = []
covered = set()
remaining_tests = set(test_coverage.keys())

while covered != universe:

    best_test = None
    best_gain = -1

    # deterministic tie-breaking
    for test_id in sorted(remaining_tests):
        gain = len(test_coverage[test_id] - covered)

        if gain > best_gain:
            best_gain = gain
            best_test = test_id

    if best_test is None or best_gain <= 0:
        raise RuntimeError(
            "Reduction stopped before full coverage was preserved."
        )

    selected_tests.append(best_test)
    covered.update(test_coverage[best_test])
    remaining_tests.remove(best_test)

    print(
        f"{len(selected_tests):2d}. "
        f"gain={best_gain:2d}, "
        f"covered={len(covered):2d}/{len(universe):2d} | "
        f"{best_test}"
    )

print("\nReduction complete.")

In [ ]:
original_count = len(clean_records)
reduced_count = len(selected_tests)

reduction_ratio = (
    (original_count - reduced_count) / original_count
) * 100

coverage_preservation = (
    len(covered) / len(universe)
) * 100

print("Original tests:", original_count)
print("Reduced tests:", reduced_count)
print("Removed tests:", original_count - reduced_count)
print("Reduction ratio (%):", round(reduction_ratio, 2))
print("Coverage preservation (%):",
      round(coverage_preservation, 2))
print("Structural elements preserved:",
      len(covered), "/", len(universe))

In [ ]:
from pathlib import Path

selected_file = ROOT / "selected_tests.txt"

selected_file.write_text(
    "\n".join(selected_tests),
    encoding="utf-8"
)

print("Saved selected tests to:")
print(selected_file)

print("\nSelected tests:")
for i, test_id in enumerate(selected_tests, start=1):
    print(f"{i:2d}. {test_id}")

In [ ]:
import subprocess
import sys
import json

reduced_cov_json = ROOT / "coverage_calc_a_reduced.json"

if reduced_cov_json.exists():
    reduced_cov_json.unlink()

cmd = [
    sys.executable,
    "-m",
    "pytest",
    *selected_tests,
    "--cov=calc_a",
    "--cov-branch",
    "--cov-report=term-missing",
    f"--cov-report=json:{reduced_cov_json.name}",
    "-q",
]

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)
print("Coverage JSON created:", reduced_cov_json.exists())

In [ ]:
with open(reduced_cov_json, "r", encoding="utf-8") as f:
    reduced_cov = json.load(f)

for filename, filedata in reduced_cov["files"].items():
    normalized = filename.replace("\\", "/")

    if normalized.endswith("calc_a/functions.py"):
        print("Implementation file:", normalized)
        print("Executed lines:",
              len(filedata.get("executed_lines", [])))
        print("Executed branches:",
              len(filedata.get("executed_branches", [])))
        print("Missing lines:",
              filedata.get("missing_lines", []))
        print("Missing branches:",
              filedata.get("missing_branches", []))

In [ ]:
import json

summary = {
    "experiment": "Calculator A coverage-guided reduction",
    "original_tests": 278,
    "reduced_tests": len(selected_tests),
    "removed_tests": 278 - len(selected_tests),
    "reduction_ratio_percent": round(
        (278 - len(selected_tests)) / 278 * 100, 2
    ),
    "implementation_lines_preserved": 21,
    "implementation_branches_preserved": 6,
    "structural_elements_preserved": 27,
    "coverage_preservation_percent": 100.0
}

summary_path = ROOT / "reduction_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", summary_path)

In [ ]:
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

ADAPTER_DIR = ROOT / "adapters"
ADAPTER_DIR.mkdir(exist_ok=True)

(ADAPTER_DIR / "__init__.py").write_text(
    "",
    encoding="utf-8"
)

adapter_code = '''\
from calc_b.operations import (
    sum_values,
    difference,
    product,
    quotient,
    remainder,
    raise_to_power,
    root,
)

# Semantic adaptation from Calculator A API to Calculator B API

def add(a, b):
    return sum_values(a, b)

def subtract(a, b):
    return difference(a, b)

def multiply(a, b):
    return product(a, b)

def divide(a, b):
    return quotient(a, b)

def modulus(a, b):
    return remainder(a, b)

def power(a, b):
    return raise_to_power(a, b)

def square_root(a):
    return root(a)
'''

adapter_path = ADAPTER_DIR / "calc_b_adapter.py"
adapter_path.write_text(adapter_code, encoding="utf-8")

print("Created:", adapter_path)

In [ ]:
source_test = ROOT / "tests" / "test_calc_a_historical.py"
target_test = ROOT / "tests" / "test_calc_b_reuse.py"

source_text = source_test.read_text(encoding="utf-8")

print("Original references to calc_a:")
print(source_text.count("calc_a"))

In [ ]:
adapted_text = source_text.replace(
    "calc_a.functions",
    "adapters.calc_b_adapter"
)

target_test.write_text(
    adapted_text,
    encoding="utf-8"
)

print("Created:", target_test)
print("Remaining calc_a.functions references:",
      adapted_text.count("calc_a.functions"))
print("Adapter references:",
      adapted_text.count("adapters.calc_b_adapter"))

In [ ]:
selected_tests_b = [
    test_id.replace(
        "tests/test_calc_a_historical.py",
        "tests/test_calc_b_reuse.py"
    )
    for test_id in selected_tests
]

print("Number of adapted tests:", len(selected_tests_b))

for i, test_id in enumerate(selected_tests_b, start=1):
    print(f"{i:2d}. {test_id}")

In [ ]:
import subprocess
import sys

result_b = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        *selected_tests_b,
        "-q"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result_b.stdout)
print(result_b.stderr)
print("Return code:", result_b.returncode)

In [ ]:
import subprocess
import sys
import json
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

coverage_b_json = ROOT / "coverage_calc_b_reuse.json"

if coverage_b_json.exists():
    coverage_b_json.unlink()

cmd = [
    sys.executable,
    "-m",
    "pytest",
    *selected_tests_b,
    "--cov=calc_b",
    "--cov-branch",
    "--cov-report=term-missing",
    f"--cov-report=json:{coverage_b_json.name}",
    "-q",
]

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)
print("Coverage JSON created:", coverage_b_json.exists())

In [ ]:
with open(coverage_b_json, "r", encoding="utf-8") as f:
    cov_b = json.load(f)

for filename, filedata in cov_b["files"].items():
    normalized = filename.replace("\\", "/")

    if normalized.endswith("calc_b/operations.py"):
        print("Implementation file:", normalized)
        print("Executed lines:",
              len(filedata.get("executed_lines", [])))
        print("Total statements:",
              filedata["summary"]["num_statements"])
        print("Statement coverage %:",
              filedata["summary"]["percent_covered"])

        print("Executed branches:",
              len(filedata.get("executed_branches", [])))
        print("Total branches:",
              filedata["summary"].get("num_branches"))

        if filedata["summary"].get("num_branches"):
            branch_pct = (
                filedata["summary"]["covered_branches"]
                / filedata["summary"]["num_branches"]
                * 100
            )
            print("Branch coverage %:", round(branch_pct, 2))

        print("Missing lines:",
              filedata.get("missing_lines", []))
        print("Missing branches:",
              filedata.get("missing_branches", []))

In [ ]:
import json

reuse_b_summary = {
    "experiment": "Cross-application reuse A_to_B",
    "source_application": "Calculator A",
    "target_application": "Calculator B",
    "adapted_tests": len(selected_tests_b),
    "executed_successfully": 10,
    "passed_tests": 10,
    "failed_tests": 0,
    "transfer_success_rate_percent": 100.0,
    "target_statements": 21,
    "covered_statements": 21,
    "statement_coverage_percent": 100.0,
    "target_branches": 6,
    "covered_branches": 6,
    "branch_coverage_percent": 100.0,
    "missing_lines": [],
    "missing_branches": []
}

reuse_b_path = ROOT / "reuse_A_to_B_summary.json"

with open(reuse_b_path, "w", encoding="utf-8") as f:
    json.dump(reuse_b_summary, f, indent=2)

print("Saved:", reuse_b_path)

In [ ]:
from pathlib import Path

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

calc_c_file = ROOT / "calc_c" / "calculator.py"

print(calc_c_file.read_text(encoding="utf-8"))

In [ ]:
ADAPTER_DIR = ROOT / "adapters"
ADAPTER_DIR.mkdir(exist_ok=True)

adapter_c_code = '''\
from calc_c.calculator import Calculator

def add(a, b):
    return Calculator.add(a, b)

def subtract(a, b):
    return Calculator.subtract(a, b)

def multiply(a, b):
    return Calculator.multiply(a, b)

def divide(a, b):
    return Calculator.divide(a, b)

def modulus(a, b):
    return Calculator.modulus(a, b)

def power(a, b):
    return Calculator.power(a, b)

def square_root(a):
    return Calculator.square_root(a)
'''

adapter_c_path = ADAPTER_DIR / "calc_c_adapter.py"
adapter_c_path.write_text(adapter_c_code, encoding="utf-8")

print("Created:", adapter_c_path)

In [ ]:
source_test = ROOT / "tests" / "test_calc_a_historical.py"
target_test_c = ROOT / "tests" / "test_calc_c_reuse.py"

source_text = source_test.read_text(encoding="utf-8")

adapted_text_c = source_text.replace(
    "calc_a.functions",
    "adapters.calc_c_adapter"
)

target_test_c.write_text(
    adapted_text_c,
    encoding="utf-8"
)

print("Created:", target_test_c)
print(
    "Remaining calc_a.functions references:",
    adapted_text_c.count("calc_a.functions")
)

In [ ]:
selected_tests_c = [
    test_id.replace(
        "tests/test_calc_a_historical.py",
        "tests/test_calc_c_reuse.py"
    )
    for test_id in selected_tests
]

print("Number of adapted tests:", len(selected_tests_c))

for i, test_id in enumerate(selected_tests_c, start=1):
    print(f"{i:2d}. {test_id}")

In [ ]:
import subprocess
import sys

result_c = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        *selected_tests_c,
        "-q"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result_c.stdout)
print(result_c.stderr)
print("Return code:", result_c.returncode)

In [ ]:
import subprocess
import sys
import json

coverage_c_json = ROOT / "coverage_calc_c_reuse.json"

if coverage_c_json.exists():
    coverage_c_json.unlink()

cmd = [
    sys.executable,
    "-m",
    "pytest",
    *selected_tests_c,
    "--cov=calc_c",
    "--cov-branch",
    "--cov-report=term-missing",
    f"--cov-report=json:{coverage_c_json.name}",
    "-q",
]

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)
print("Coverage JSON created:", coverage_c_json.exists())

In [ ]:
with open(coverage_c_json, "r", encoding="utf-8") as f:
    cov_c = json.load(f)

for filename, filedata in cov_c["files"].items():
    normalized = filename.replace("\\", "/")

    if normalized.endswith("calc_c/calculator.py"):
        print("Implementation file:", normalized)
        print("Executed lines:",
              len(filedata.get("executed_lines", [])))
        print("Total statements:",
              filedata["summary"]["num_statements"])
        print("Statement coverage %:",
              filedata["summary"]["percent_covered"])

        print("Executed branches:",
              len(filedata.get("executed_branches", [])))
        print("Total branches:",
              filedata["summary"].get("num_branches"))

        if filedata["summary"].get("num_branches"):
            branch_pct = (
                filedata["summary"]["covered_branches"]
                / filedata["summary"]["num_branches"] * 100
            )
            print("Branch coverage %:", round(branch_pct, 2))

        print("Missing lines:",
              filedata.get("missing_lines", []))
        print("Missing branches:",
              filedata.get("missing_branches", []))

In [ ]:
# ============================================================
# EMSE EXPERIMENT 2
# Cross-application reuse:
# Calculator A reduced suite -> C, D, E
# ============================================================

from pathlib import Path
import subprocess
import sys
import json
import re
import pandas as pd

# ------------------------------------------------------------
# 0. Project paths
# ------------------------------------------------------------

ROOT = Path(r"<LOCAL_CALCULATOR_WORKSPACE>")

TEST_DIR = ROOT / "tests"
ADAPTER_DIR = ROOT / "adapters"
RESULT_DIR = ROOT / "results" / "experiment_02_transfer"

ADAPTER_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

(ADAPTER_DIR / "__init__.py").touch()

print("Project root:", ROOT)
print("Result directory:", RESULT_DIR)


# ------------------------------------------------------------
# 1. Load the 10 reduced Calculator A tests
# ------------------------------------------------------------

selected_file = ROOT / "selected_tests.txt"

if "selected_tests" not in globals():

    if not selected_file.exists():
        raise FileNotFoundError(
            "selected_tests.txt was not found. "
            "The reduction experiment must be completed first."
        )

    selected_tests = [
        line.strip()
        for line in selected_file.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

print("\nReduced source-suite size:", len(selected_tests))

if len(selected_tests) != 10:
    print(
        "WARNING: Expected 10 reduced tests, "
        f"but found {len(selected_tests)}."
    )


# ------------------------------------------------------------
# 2. Source historical test file
# ------------------------------------------------------------

source_test_file = (
    TEST_DIR / "test_calc_a_historical.py"
)

if not source_test_file.exists():
    raise FileNotFoundError(source_test_file)

source_test_text = source_test_file.read_text(
    encoding="utf-8"
)


# ============================================================
# 3. CREATE ADAPTERS
# ============================================================

# ------------------------------------------------------------
# Calculator C adapter
# Function API -> static class methods
# ------------------------------------------------------------

adapter_c_code = """\
from calc_c.calculator import Calculator

def add(a, b):
    return Calculator.add(a, b)

def subtract(a, b):
    return Calculator.subtract(a, b)

def multiply(a, b):
    return Calculator.multiply(a, b)

def divide(a, b):
    return Calculator.divide(a, b)

def modulus(a, b):
    return Calculator.modulus(a, b)

def power(a, b):
    return Calculator.power(a, b)

def square_root(a):
    return Calculator.square_root(a)
"""

(ADAPTER_DIR / "calc_c_adapter.py").write_text(
    adapter_c_code,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Calculator D adapter
# Function API -> command-dispatch engine
# ------------------------------------------------------------

adapter_d_code = """\
from calc_d.engine import CalculationEngine

_engine = CalculationEngine()

def add(a, b):
    return _engine.compute("add", a, b)

def subtract(a, b):
    return _engine.compute("subtract", a, b)

def multiply(a, b):
    return _engine.compute("multiply", a, b)

def divide(a, b):
    return _engine.compute("divide", a, b)

def modulus(a, b):
    return _engine.compute("modulus", a, b)

def power(a, b):
    return _engine.compute("power", a, b)

def square_root(a):
    return _engine.compute("square_root", a)
"""

(ADAPTER_DIR / "calc_d_adapter.py").write_text(
    adapter_d_code,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Calculator E adapter
# Function API -> request-dictionary service
# ------------------------------------------------------------

adapter_e_code = """\
from calc_e.service import CalculatorService

_service = CalculatorService()

def add(a, b):
    return _service.execute({
        "operation": "add",
        "left": a,
        "right": b
    })

def subtract(a, b):
    return _service.execute({
        "operation": "subtract",
        "left": a,
        "right": b
    })

def multiply(a, b):
    return _service.execute({
        "operation": "multiply",
        "left": a,
        "right": b
    })

def divide(a, b):
    return _service.execute({
        "operation": "divide",
        "left": a,
        "right": b
    })

def modulus(a, b):
    return _service.execute({
        "operation": "modulus",
        "left": a,
        "right": b
    })

def power(a, b):
    return _service.execute({
        "operation": "power",
        "left": a,
        "right": b
    })

def square_root(a):
    return _service.execute({
        "operation": "square_root",
        "left": a
    })
"""

(ADAPTER_DIR / "calc_e_adapter.py").write_text(
    adapter_e_code,
    encoding="utf-8"
)

print("\nAdapters created:")
print("  calc_c_adapter.py")
print("  calc_d_adapter.py")
print("  calc_e_adapter.py")


# ============================================================
# 4. CREATE ADAPTED TEST FILES
# ============================================================

targets = {
    "C": {
        "adapter": "adapters.calc_c_adapter",
        "test_file": "test_calc_c_reuse.py",
        "coverage_package": "calc_c",
        "implementation_file": "calc_c/calculator.py",
    },

    "D": {
        "adapter": "adapters.calc_d_adapter",
        "test_file": "test_calc_d_reuse.py",
        "coverage_package": "calc_d",
        "implementation_file": "calc_d/engine.py",
    },

    "E": {
        "adapter": "adapters.calc_e_adapter",
        "test_file": "test_calc_e_reuse.py",
        "coverage_package": "calc_e",
        "implementation_file": "calc_e/service.py",
    },
}


for target_name, cfg in targets.items():

    adapted_text = source_test_text.replace(
        "calc_a.functions",
        cfg["adapter"]
    )

    if "calc_a.functions" in adapted_text:
        raise RuntimeError(
            f"Calculator {target_name}: "
            "source import was not fully replaced."
        )

    target_test_path = TEST_DIR / cfg["test_file"]

    target_test_path.write_text(
        adapted_text,
        encoding="utf-8"
    )

    cfg["test_path"] = target_test_path

    cfg["selected_tests"] = [
        test_id.replace(
            "tests/test_calc_a_historical.py",
            f"tests/{cfg['test_file']}"
        )
        for test_id in selected_tests
    ]

    print(
        f"Created adapted test file for {target_name}:",
        target_test_path
    )


# ============================================================
# 5. HELPER: PARSE PYTEST RESULT COUNTS
# ============================================================

def parse_pytest_counts(output):

    counts = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
    }

    for key, pattern in patterns.items():

        match = re.search(pattern, output)

        if match:
            counts[key] = int(match.group(1))

    return counts


# ============================================================
# 6. RUN TRANSFER + COVERAGE FOR EACH TARGET
# ============================================================

all_results = []

for target_name, cfg in targets.items():

    print("\n")
    print("=" * 70)
    print(
        f"CALCULATOR A -> CALCULATOR {target_name}"
    )
    print("=" * 70)

    adapted_tests = cfg["selected_tests"]

    # --------------------------------------------------------
    # 6.1 Functional execution
    # --------------------------------------------------------

    functional_cmd = [
        sys.executable,
        "-m",
        "pytest",
        *adapted_tests,
        "-q",
    ]

    functional_result = subprocess.run(
        functional_cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
    )

    print("\nFUNCTIONAL EXECUTION")
    print(functional_result.stdout)

    if functional_result.stderr.strip():
        print(functional_result.stderr)

    print(
        "Return code:",
        functional_result.returncode
    )

    counts = parse_pytest_counts(
        functional_result.stdout
        + "\n"
        + functional_result.stderr
    )

    passed = counts["passed"]
    failed = counts["failed"]
    errors = counts["errors"]

    n_adapted = len(adapted_tests)

    transfer_success_rate = (
        passed / n_adapted * 100
        if n_adapted
        else 0.0
    )

    # --------------------------------------------------------
    # 6.2 Target coverage
    # --------------------------------------------------------

    coverage_json = (
        RESULT_DIR
        / f"coverage_A_to_{target_name}.json"
    )

    if coverage_json.exists():
        coverage_json.unlink()

    coverage_cmd = [
        sys.executable,
        "-m",
        "pytest",
        *adapted_tests,
        f"--cov={cfg['coverage_package']}",
        "--cov-branch",
        "--cov-report=term-missing",
        f"--cov-report=json:{coverage_json}",
        "-q",
    ]

    coverage_result = subprocess.run(
        coverage_cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
    )

    print("\nCOVERAGE EXECUTION")
    print(coverage_result.stdout)

    if coverage_result.stderr.strip():
        print(coverage_result.stderr)

    print(
        "Coverage return code:",
        coverage_result.returncode
    )

    if not coverage_json.exists():
        raise RuntimeError(
            f"Coverage JSON not created for "
            f"Calculator {target_name}."
        )

    with open(
        coverage_json,
        "r",
        encoding="utf-8"
    ) as f:
        coverage_data = json.load(f)

    target_file_data = None

    for filename, filedata in (
        coverage_data["files"].items()
    ):

        normalized = filename.replace("\\", "/")

        if normalized.endswith(
            cfg["implementation_file"]
        ):
            target_file_data = filedata
            break

    if target_file_data is None:
        raise RuntimeError(
            f"Could not find "
            f"{cfg['implementation_file']} "
            "in coverage JSON."
        )

    summary = target_file_data["summary"]

    total_statements = int(
        summary["num_statements"]
    )

    covered_statements = int(
        summary["covered_lines"]
    )

    missing_lines = (
        target_file_data.get(
            "missing_lines",
            []
        )
    )

    statement_pct = (
        covered_statements
        / total_statements
        * 100
        if total_statements
        else 100.0
    )

    total_branches = int(
        summary.get(
            "num_branches",
            0
        )
    )

    covered_branches = int(
        summary.get(
            "covered_branches",
            0
        )
    )

    missing_branches = (
        target_file_data.get(
            "missing_branches",
            []
        )
    )

    branch_pct = (
        covered_branches
        / total_branches
        * 100
        if total_branches
        else 100.0
    )

    result_record = {
        "source_application": "Calculator A",
        "target_application":
            f"Calculator {target_name}",

        "adapted_tests": n_adapted,
        "passed_tests": passed,
        "failed_tests": failed,
        "error_tests": errors,

        "transfer_success_rate_percent":
            round(
                transfer_success_rate,
                2
            ),

        "total_statements":
            total_statements,

        "covered_statements":
            covered_statements,

        "statement_coverage_percent":
            round(
                statement_pct,
                2
            ),

        "total_branches":
            total_branches,

        "covered_branches":
            covered_branches,

        "branch_coverage_percent":
            round(
                branch_pct,
                2
            ),

        "missing_lines":
            missing_lines,

        "missing_branches":
            missing_branches,

        "functional_return_code":
            functional_result.returncode,

        "coverage_return_code":
            coverage_result.returncode,
    }

    all_results.append(
        result_record
    )

    # --------------------------------------------------------
    # 6.3 Save individual JSON
    # --------------------------------------------------------

    individual_summary_path = (
        RESULT_DIR
        / f"reuse_A_to_{target_name}_summary.json"
    )

    with open(
        individual_summary_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result_record,
            f,
            indent=2
        )

    # Also retain a copy at project root
    root_summary_path = (
        ROOT
        / f"reuse_A_to_{target_name}_summary.json"
    )

    with open(
        root_summary_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result_record,
            f,
            indent=2
        )

    print("\nSUMMARY")
    print(
        "Adapted tests:",
        n_adapted
    )
    print(
        "Passed:",
        passed
    )
    print(
        "Failed:",
        failed
    )
    print(
        "Errors:",
        errors
    )
    print(
        "Transfer success rate (%):",
        round(
            transfer_success_rate,
            2
        )
    )

    print(
        "Statements:",
        f"{covered_statements}/"
        f"{total_statements}"
    )

    print(
        "Statement coverage (%):",
        round(
            statement_pct,
            2
        )
    )

    print(
        "Branches:",
        f"{covered_branches}/"
        f"{total_branches}"
    )

    print(
        "Branch coverage (%):",
        round(
            branch_pct,
            2
        )
    )

    print(
        "Missing lines:",
        missing_lines
    )

    print(
        "Missing branches:",
        missing_branches
    )

    print(
        "Saved:",
        individual_summary_path
    )


# ============================================================
# 7. INCLUDE THE COMPLETED A -> B RESULT
# ============================================================

b_summary_file = (
    ROOT / "reuse_A_to_B_summary.json"
)

combined_results = []

if b_summary_file.exists():

    with open(
        b_summary_file,
        "r",
        encoding="utf-8"
    ) as f:
        b_result = json.load(f)

    combined_results.append(
        {
            "source_application":
                "Calculator A",

            "target_application":
                "Calculator B",

            "adapted_tests":
                b_result.get(
                    "adapted_tests",
                    10
                ),

            "passed_tests":
                b_result.get(
                    "passed_tests",
                    10
                ),

            "failed_tests":
                b_result.get(
                    "failed_tests",
                    0
                ),

            "error_tests":
                0,

            "transfer_success_rate_percent":
                b_result.get(
                    "transfer_success_rate_percent",
                    100.0
                ),

            "total_statements":
                b_result.get(
                    "target_statements",
                    21
                ),

            "covered_statements":
                b_result.get(
                    "covered_statements",
                    21
                ),

            "statement_coverage_percent":
                b_result.get(
                    "statement_coverage_percent",
                    100.0
                ),

            "total_branches":
                b_result.get(
                    "target_branches",
                    6
                ),

            "covered_branches":
                b_result.get(
                    "covered_branches",
                    6
                ),

            "branch_coverage_percent":
                b_result.get(
                    "branch_coverage_percent",
                    100.0
                ),

            "missing_lines":
                b_result.get(
                    "missing_lines",
                    []
                ),

            "missing_branches":
                b_result.get(
                    "missing_branches",
                    []
                )
        }
    )


combined_results.extend(
    all_results
)


# ============================================================
# 8. SAVE COMBINED TRANSFER TABLE
# ============================================================

df_transfer = pd.DataFrame(
    combined_results
)

csv_path = (
    RESULT_DIR
    / "cross_application_transfer_summary.csv"
)

df_transfer.to_csv(
    csv_path,
    index=False
)

json_path = (
    RESULT_DIR
    / "cross_application_transfer_summary.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        combined_results,
        f,
        indent=2
    )

print("\n")
print("=" * 70)
print("EXPERIMENT 2 CROSS-APPLICATION SUMMARY")
print("=" * 70)

display(
    df_transfer[
        [
            "target_application",
            "adapted_tests",
            "passed_tests",
            "transfer_success_rate_percent",
            "covered_statements",
            "total_statements",
            "statement_coverage_percent",
            "covered_branches",
            "total_branches",
            "branch_coverage_percent",
        ]
    ]
)

print("\nSaved combined CSV:")
print(csv_path)

print("\nSaved combined JSON:")
print(json_path)

print("\nExperiment 2 run completed.")

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(r"D:\EMSE_Realistic_Benchmark")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_r1_historical.py",
        "-q"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

In [ ]:
from pathlib import Path
import subprocess
import sys
import json

ROOT = Path(r"D:\EMSE_Realistic_Benchmark")

coverage_json = ROOT / "coverage_r1_baseline.json"

if coverage_json.exists():
    coverage_json.unlink()

cmd = [
    sys.executable,
    "-m",
    "pytest",
    "tests/test_r1_historical.py",
    "--cov=r1",
    "--cov-branch",
    "--cov-report=term-missing",
    f"--cov-report=json:{coverage_json.name}",
    "-q",
]

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)
print("Coverage JSON created:", coverage_json.exists())

In [ ]:
with open(ROOT / "coverage_r1_baseline.json", "r", encoding="utf-8") as f:
    cov = json.load(f)

print("=== R1 WHOLE-SUITE COVERAGE ===")

total_statements = 0
covered_statements = 0
total_branches = 0
covered_branches = 0

for filename, filedata in cov["files"].items():
    normalized = filename.replace("\\", "/")

    if "/r1/" in f"/{normalized}" and not normalized.endswith("__init__.py"):
        summary = filedata["summary"]

        total_statements += summary["num_statements"]
        covered_statements += summary["covered_lines"]
        total_branches += summary.get("num_branches", 0)
        covered_branches += summary.get("covered_branches", 0)

        print("\nFile:", normalized)
        print(" Statements:",
              summary["covered_lines"],
              "/",
              summary["num_statements"])

        print(" Branches:",
              summary.get("covered_branches", 0),
              "/",
              summary.get("num_branches", 0))

        print(" Missing lines:",
              filedata.get("missing_lines", []))

        print(" Missing branches:",
              filedata.get("missing_branches", []))

print("\n=== TOTAL R1 IMPLEMENTATION COVERAGE ===")

statement_pct = (
    covered_statements / total_statements * 100
    if total_statements else 0
)

branch_pct = (
    covered_branches / total_branches * 100
    if total_branches else 0
)

print("Covered statements:", covered_statements)
print("Total statements:", total_statements)
print("Statement coverage %:", round(statement_pct, 2))

print("Covered branches:", covered_branches)
print("Total branches:", total_branches)
print("Branch coverage %:", round(branch_pct, 2))

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/measure_complexity.py"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/measure_complexity.py"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(r"D:\EMSE_Realistic_Benchmark")

result = subprocess.run(
    [
        sys.executable,
        "scripts/measure_complexity.py"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

In [ ]:
%pip install radon

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(r"D:\EMSE_Realistic_Benchmark")

result = subprocess.run(
    [
        sys.executable,
        "scripts/measure_complexity.py"
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import urllib.request
import urllib.error
import zipfile
import hashlib
import json
import shutil
import sys
import os

# ==========================================================
# EMSE TIER-3 OSS BENCHMARK ACQUISITION
# ==========================================================

ROOT = Path(r"<LOCAL_WORKSPACE>")
DOWNLOAD_DIR = ROOT / "_downloads"

ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Workspace:", ROOT)

# ----------------------------------------------------------
# Frozen candidate versions
# ----------------------------------------------------------

PROJECTS = {
    "attrs": {
        "owner": "python-attrs",
        "repo": "attrs",
        "versions": ["24.1.0", "25.3.0", "26.1.0"],
    },

    "cattrs": {
        "owner": "python-attrs",
        "repo": "cattrs",
        "versions": ["24.1.0", "25.3.0", "26.1.0"],
    },

    "boltons": {
        "owner": "mahmoud",
        "repo": "boltons",
        "versions": ["24.1.0", "25.0.0", "26.1.0"],
    },

    "more_itertools": {
        "owner": "more-itertools",
        "repo": "more-itertools",
        "versions": ["10.5.0", "10.8.0", "11.1.0"],
    },
}

# ----------------------------------------------------------
# Utility functions
# ----------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def download_file(url, destination):
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent":
                "Mozilla/5.0 EMSE-regression-testing-benchmark"
        },
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        data = response.read()

    destination.write_bytes(data)


def try_download_tag(owner, repo, version, destination):

    # Some GitHub projects use "v10.8.0"; others use "10.8.0".
    candidate_tags = [
        version,
        f"v{version}",
    ]

    errors = []

    for tag in candidate_tags:

        url = (
            f"https://codeload.github.com/"
            f"{owner}/{repo}/zip/refs/tags/{tag}"
        )

        print("Trying:", url)

        try:
            download_file(url, destination)

            if destination.stat().st_size < 1000:
                raise RuntimeError(
                    "Downloaded archive unexpectedly small."
                )

            return tag, url

        except Exception as exc:

            errors.append(
                {
                    "tag": tag,
                    "url": url,
                    "error": str(exc),
                }
            )

            if destination.exists():
                destination.unlink()

    raise RuntimeError(
        f"No downloadable tag found for {repo} {version}: "
        f"{errors}"
    )


def extract_archive(archive, destination):

    temp = destination.parent / (
        destination.name + "_EXTRACT_TEMP"
    )

    if temp.exists():
        shutil.rmtree(temp)

    if destination.exists():
        shutil.rmtree(destination)

    temp.mkdir(parents=True)

    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(temp)

    extracted = [
        p for p in temp.iterdir()
        if p.is_dir()
    ]

    if len(extracted) != 1:
        raise RuntimeError(
            f"Unexpected GitHub archive structure: {archive}"
        )

    shutil.move(
        str(extracted[0]),
        str(destination)
    )

    shutil.rmtree(temp)


def count_python_files(directory):

    production = []
    tests = []

    for path in directory.rglob("*.py"):

        parts_lower = [
            part.lower()
            for part in path.parts
        ]

        relative = path.relative_to(directory)

        if (
            "test" in parts_lower
            or "tests" in parts_lower
            or path.name.startswith("test_")
        ):
            tests.append(relative)
        else:
            production.append(relative)

    return len(production), len(tests)


# ----------------------------------------------------------
# Acquire all 12 frozen releases
# ----------------------------------------------------------

manifest = {
    "benchmark":
        "EMSE Tier-3 Real Open-Source Benchmark",
    "purpose":
        "Coverage-guided regression-test reduction, "
        "cross-version reuse, residual-gap augmentation, "
        "mutation adequacy and execution-cost evaluation",
    "acquisition_time_utc":
        datetime.now(timezone.utc).isoformat(),
    "python_version":
        sys.version,
    "workspace":
        str(ROOT),
    "subjects": [],
}

failures = []


for project_name, cfg in PROJECTS.items():

    print("\n")
    print("=" * 76)
    print("PROJECT:", project_name)
    print("=" * 76)

    project_root = ROOT / project_name
    project_root.mkdir(exist_ok=True)

    for version in cfg["versions"]:

        print("\n---", project_name, version, "---")

        archive = (
            DOWNLOAD_DIR
            / f"{project_name}_{version}.zip"
        )

        destination = (
            project_root
            / version
        )

        try:

            tag, url = try_download_tag(
                cfg["owner"],
                cfg["repo"],
                version,
                archive,
            )

            digest = sha256_file(archive)

            print(
                "Downloaded:",
                archive.name
            )

            print(
                "SHA-256:",
                digest
            )

            extract_archive(
                archive,
                destination
            )

            prod_py, test_py = (
                count_python_files(destination)
            )

            record = {
                "project": project_name,
                "repository":
                    f"https://github.com/"
                    f"{cfg['owner']}/{cfg['repo']}",
                "requested_version":
                    version,
                "resolved_tag":
                    tag,
                "archive_url":
                    url,
                "archive_filename":
                    archive.name,
                "sha256":
                    digest,
                "local_path":
                    str(destination),
                "production_python_files":
                    prod_py,
                "test_python_files":
                    test_py,
                "status":
                    "ACQUIRED",
            }

            manifest[
                "subjects"
            ].append(record)

            print(
                "Extracted to:",
                destination
            )

            print(
                "Production Python files:",
                prod_py
            )

            print(
                "Test Python files:",
                test_py
            )

        except Exception as exc:

            failure = {
                "project":
                    project_name,
                "version":
                    version,
                "error":
                    str(exc),
            }

            failures.append(failure)

            manifest[
                "subjects"
            ].append(
                {
                    "project":
                        project_name,
                    "requested_version":
                        version,
                    "status":
                        "FAILED",
                    "error":
                        str(exc),
                }
            )

            print(
                "FAILED:",
                project_name,
                version
            )

            print(exc)


# ----------------------------------------------------------
# Save reproducibility manifest
# ----------------------------------------------------------

manifest["failures"] = failures

manifest_path = (
    ROOT
    / "tier3_acquisition_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8",
)


# ----------------------------------------------------------
# Human-readable acquisition table
# ----------------------------------------------------------

print("\n")
print("=" * 76)
print("TIER-3 ACQUISITION SUMMARY")
print("=" * 76)

successful = [
    x for x in manifest["subjects"]
    if x["status"] == "ACQUIRED"
]

for x in successful:

    print(
        f"{x['project']:18s} "
        f"{x['requested_version']:10s} "
        f"tag={x['resolved_tag']:10s} "
        f"prod_py={x['production_python_files']:4d} "
        f"test_py={x['test_python_files']:4d}"
    )


print("\nSuccessfully acquired:",
      len(successful),
      "/ 12")

print("Failures:",
      len(failures))

print(
    "\nManifest saved to:"
)

print(
    manifest_path
)


if failures:

    print("\nFAILED SUBJECTS:")

    for f in failures:
        print(f)


print("\nTier-3 acquisition completed.")

In [ ]:
from pathlib import Path
import ast
import csv
import hashlib
import json
import re
import sys

from radon.complexity import cc_visit

ROOT = Path(r"<LOCAL_WORKSPACE>")

SUBJECTS = {
    "attrs": {
        "versions": ["24.1.0", "25.3.0", "26.1.0"],
        "production_roots": [
            Path("src/attr"),
            Path("src/attrs"),
        ],
        "test_roots": [Path("tests")],
    },
    "cattrs": {
        "versions": ["24.1.0", "25.3.0", "26.1.0"],
        "production_roots": [
            Path("src/cattrs"),
        ],
        "test_roots": [Path("tests")],
    },
    "boltons": {
        "versions": ["24.1.0", "25.0.0", "26.1.0"],
        "production_roots": [
            Path("boltons"),
        ],
        "test_roots": [Path("tests")],
    },
    "more_itertools": {
        "versions": ["10.5.0", "10.8.0", "11.1.0"],
        "production_roots": [
            Path("more_itertools"),
            Path("src/more_itertools"),
        ],
        "test_roots": [Path("tests")],
    },
}


def valid_roots(subject_dir, candidates):
    return [
        subject_dir / p
        for p in candidates
        if (subject_dir / p).exists()
    ]


def python_files(roots):
    files = []
    for root in roots:
        files.extend(root.rglob("*.py"))
    return sorted(set(files))


def count_loc(path):
    count = 0
    try:
        for line in path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).splitlines():
            stripped = line.strip()
            if stripped and not stripped.startswith("#"):
                count += 1
    except Exception:
        pass
    return count


def ast_counts(path):
    try:
        source = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
        tree = ast.parse(source)
    except Exception:
        return 0, 0, 0

    functions = 0
    classes = 0
    tests = 0

    for node in ast.walk(tree):
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ):
            functions += 1

            if node.name.startswith("test"):
                tests += 1

        elif isinstance(node, ast.ClassDef):
            classes += 1

    return functions, classes, tests


def complexity_counts(path):
    try:
        source = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        blocks = cc_visit(source)

        values = []

        for block in blocks:
            values.append(block.complexity)

            methods = getattr(block, "methods", [])
            for method in methods:
                values.append(method.complexity)

        return values

    except Exception:
        return []


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def detect_python_requirement(subject_dir):
    candidates = [
        subject_dir / "pyproject.toml",
        subject_dir / "setup.cfg",
        subject_dir / "setup.py",
    ]

    patterns = [
        r'requires-python\s*=\s*"([^"]+)"',
        r"requires-python\s*=\s*'([^']+)'",
        r'python_requires\s*=\s*"([^"]+)"',
        r"python_requires\s*=\s*'([^']+)'",
    ]

    for file in candidates:
        if not file.exists():
            continue

        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        for pattern in patterns:
            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )

            if match:
                return match.group(1)

    return None


def config_presence(subject_dir):
    names = [
        "pyproject.toml",
        "setup.cfg",
        "setup.py",
        "tox.ini",
        "pytest.ini",
        "requirements.txt",
        "requirements-dev.txt",
    ]

    return [
        name
        for name in names
        if (subject_dir / name).exists()
    ]


records = []
file_hash_index = {}

print("Python:", sys.version)
print("Workspace:", ROOT)

for project, cfg in SUBJECTS.items():

    print("\n" + "=" * 76)
    print(project.upper())
    print("=" * 76)

    file_hash_index[project] = {}

    for version in cfg["versions"]:

        subject_dir = ROOT / project / version

        prod_roots = valid_roots(
            subject_dir,
            cfg["production_roots"],
        )

        test_roots = valid_roots(
            subject_dir,
            cfg["test_roots"],
        )

        prod_files = python_files(prod_roots)
        test_files = python_files(test_roots)

        prod_loc = sum(
            count_loc(p)
            for p in prod_files
        )

        test_loc = sum(
            count_loc(p)
            for p in test_files
        )

        prod_functions = 0
        prod_classes = 0

        static_test_functions = 0

        complexity_values = []

        for p in prod_files:

            f, c, _ = ast_counts(p)

            prod_functions += f
            prod_classes += c

            complexity_values.extend(
                complexity_counts(p)
            )

        for p in test_files:

            _, _, t = ast_counts(p)

            static_test_functions += t

        mean_cc = (
            sum(complexity_values)
            / len(complexity_values)
            if complexity_values
            else 0.0
        )

        max_cc = (
            max(complexity_values)
            if complexity_values
            else 0
        )

        python_requirement = (
            detect_python_requirement(
                subject_dir
            )
        )

        configs = config_presence(
            subject_dir
        )

        hashes = {}

        for p in prod_files:

            rel = p.relative_to(subject_dir)

            hashes[
                str(rel).replace("\\", "/")
            ] = sha256(p)

        file_hash_index[
            project
        ][version] = hashes

        record = {
            "project": project,
            "version": version,
            "production_python_files":
                len(prod_files),
            "production_loc":
                prod_loc,
            "test_python_files":
                len(test_files),
            "test_loc":
                test_loc,
            "production_functions_methods":
                prod_functions,
            "production_classes":
                prod_classes,
            "static_test_functions":
                static_test_functions,
            "mean_cyclomatic_complexity":
                round(mean_cc, 3),
            "max_cyclomatic_complexity":
                max_cc,
            "python_requirement":
                python_requirement,
            "configuration_files":
                "; ".join(configs),
        }

        records.append(record)

        print(
            f"{version:10s}  "
            f"prod_LOC={prod_loc:6d}  "
            f"test_LOC={test_loc:6d}  "
            f"prod_py={len(prod_files):3d}  "
            f"test_py={len(test_files):3d}  "
            f"functions={prod_functions:4d}  "
            f"classes={prod_classes:3d}  "
            f"static_tests={static_test_functions:4d}  "
            f"mean_CC={mean_cc:5.2f}  "
            f"max_CC={max_cc:3d}"
        )


# ==========================================================
# Release-to-release production-code change analysis
# ==========================================================

change_records = []

print("\n")
print("=" * 76)
print("RELEASE-TO-RELEASE SOURCE CHANGE SUMMARY")
print("=" * 76)

for project, cfg in SUBJECTS.items():

    versions = cfg["versions"]

    for old, new in zip(
        versions[:-1],
        versions[1:]
    ):

        old_files = file_hash_index[
            project
        ][old]

        new_files = file_hash_index[
            project
        ][new]

        old_names = set(old_files)
        new_names = set(new_files)

        added = new_names - old_names
        removed = old_names - new_names

        common = old_names & new_names

        modified = {
            p for p in common
            if old_files[p] != new_files[p]
        }

        unchanged = common - modified

        total_changed = (
            len(added)
            + len(removed)
            + len(modified)
        )

        row = {
            "project": project,
            "from_version": old,
            "to_version": new,
            "added_python_files":
                len(added),
            "removed_python_files":
                len(removed),
            "modified_python_files":
                len(modified),
            "unchanged_python_files":
                len(unchanged),
            "total_changed_python_files":
                total_changed,
        }

        change_records.append(row)

        print(
            f"{project:18s} "
            f"{old:10s} -> {new:10s}  "
            f"added={len(added):3d}  "
            f"removed={len(removed):3d}  "
            f"modified={len(modified):3d}  "
            f"unchanged={len(unchanged):3d}"
        )


# ==========================================================
# Save CSV / JSON
# ==========================================================

screening_csv = (
    ROOT
    / "tier3_static_screening.csv"
)

screening_json = (
    ROOT
    / "tier3_static_screening.json"
)

changes_csv = (
    ROOT
    / "tier3_release_changes.csv"
)

changes_json = (
    ROOT
    / "tier3_release_changes.json"
)


with open(
    screening_csv,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=records[0].keys(),
    )

    writer.writeheader()
    writer.writerows(records)


screening_json.write_text(
    json.dumps(
        records,
        indent=2,
    ),
    encoding="utf-8",
)


with open(
    changes_csv,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=change_records[0].keys(),
    )

    writer.writeheader()
    writer.writerows(change_records)


changes_json.write_text(
    json.dumps(
        change_records,
        indent=2,
    ),
    encoding="utf-8",
)


print("\nSaved:")
print(screening_csv)
print(screening_json)
print(changes_csv)
print(changes_json)

print("\nStatic Tier-3 feasibility screening completed.")

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import csv
import os
import shutil
import time
import tomllib

ROOT = Path(r"<LOCAL_WORKSPACE>")
VENV_ROOT = ROOT / "_venvs"
RESULT_ROOT = ROOT / "dynamic_feasibility"

VENV_ROOT.mkdir(exist_ok=True)
RESULT_ROOT.mkdir(exist_ok=True)

SUBJECTS = {
    "attrs": ["24.1.0", "25.3.0", "26.1.0"],
    "cattrs": ["24.1.0", "25.3.0", "26.1.0"],
    "boltons": ["24.1.0", "25.0.0", "26.1.0"],
    "more_itertools": ["10.5.0", "10.8.0", "11.1.0"],
}

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def run(cmd, cwd=None, timeout=600):
    """
    Run command and return structured result.
    """
    start = time.time()

    try:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout,
            shell=False,
        )

        return {
            "returncode": proc.returncode,
            "output": proc.stdout,
            "elapsed_seconds": round(time.time() - start, 3),
            "timeout": False,
        }

    except subprocess.TimeoutExpired as exc:
        output = ""

        if exc.stdout:
            if isinstance(exc.stdout, bytes):
                output += exc.stdout.decode(
                    "utf-8",
                    errors="replace"
                )
            else:
                output += exc.stdout

        return {
            "returncode": -999,
            "output": output,
            "elapsed_seconds": round(time.time() - start, 3),
            "timeout": True,
        }


def venv_python(venv_dir):
    return venv_dir / "Scripts" / "python.exe"


def parse_pyproject(project_dir):
    """
    Inspect pyproject.toml for:
      - optional dependency groups
      - PEP 735 dependency-groups
      - Python requirement
    """
    path = project_dir / "pyproject.toml"

    result = {
        "python_requirement": None,
        "optional_dependency_groups": [],
        "dependency_groups": [],
    }

    if not path.exists():
        return result

    try:
        data = tomllib.loads(
            path.read_text(
                encoding="utf-8",
                errors="replace"
            )
        )
    except Exception:
        return result

    project = data.get("project", {})

    result["python_requirement"] = (
        project.get("requires-python")
    )

    optional = project.get(
        "optional-dependencies",
        {}
    )

    result["optional_dependency_groups"] = (
        sorted(optional.keys())
        if isinstance(optional, dict)
        else []
    )

    dep_groups = data.get(
        "dependency-groups",
        {}
    )

    result["dependency_groups"] = (
        sorted(dep_groups.keys())
        if isinstance(dep_groups, dict)
        else []
    )

    return result


def choose_extra(groups):
    """
    Select a likely test/development extra if present.
    """
    preferred = [
        "tests",
        "test",
        "testing",
        "dev",
        "development",
    ]

    lower_map = {
        x.lower(): x
        for x in groups
    }

    for name in preferred:
        if name in lower_map:
            return lower_map[name]

    return None


def extract_dependency_group(project_dir, group_name):
    """
    Extract simple string dependencies from a PEP 735
    dependency group.

    Handles nested include-group entries recursively.
    """
    path = project_dir / "pyproject.toml"

    if not path.exists():
        return []

    try:
        data = tomllib.loads(
            path.read_text(
                encoding="utf-8",
                errors="replace"
            )
        )
    except Exception:
        return []

    groups = data.get(
        "dependency-groups",
        {}
    )

    seen = set()

    def resolve(name):
        if name in seen:
            return []

        seen.add(name)

        values = groups.get(name, [])
        deps = []

        if not isinstance(values, list):
            return deps

        for entry in values:

            if isinstance(entry, str):
                deps.append(entry)

            elif isinstance(entry, dict):
                included = entry.get(
                    "include-group"
                )

                if included:
                    deps.extend(
                        resolve(included)
                    )

        return deps

    return resolve(group_name)


def count_collected_tests(text):
    """
    pytest typically prints:
       123 tests collected
       1 test collected

    Return count if identifiable.
    """
    import re

    patterns = [
        r"(\d+)\s+tests?\s+collected",
        r"collected\s+(\d+)\s+items?",
    ]

    for pattern in patterns:
        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if matches:
            return int(matches[-1])

    return None


# ---------------------------------------------------------
# Dynamic feasibility
# ---------------------------------------------------------

records = []

print("=" * 78)
print("EMSE TIER-3 DYNAMIC FEASIBILITY")
print("=" * 78)
print("Driver Python:", sys.version)
print("Root:", ROOT)

for project, versions in SUBJECTS.items():

    print("\n" + "=" * 78)
    print(project.upper())
    print("=" * 78)

    for version in versions:

        project_dir = (
            ROOT
            / project
            / version
        )

        venv_dir = (
            VENV_ROOT
            / f"{project}_{version}"
        )

        log_dir = (
            RESULT_ROOT
            / project
            / version
        )

        log_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        print(
            f"\n--- {project} {version} ---"
        )

        metadata = parse_pyproject(
            project_dir
        )

        optional_groups = (
            metadata[
                "optional_dependency_groups"
            ]
        )

        dependency_groups = (
            metadata[
                "dependency_groups"
            ]
        )

        chosen_extra = choose_extra(
            optional_groups
        )

        chosen_dep_group = choose_extra(
            dependency_groups
        )

        # ---------------------------------------------
        # Create clean isolated environment
        # ---------------------------------------------

        if venv_dir.exists():
            shutil.rmtree(
                venv_dir,
                ignore_errors=True
            )

        create_result = run(
            [
                sys.executable,
                "-m",
                "venv",
                str(venv_dir),
            ],
            timeout=180,
        )

        py = venv_python(
            venv_dir
        )

        if (
            create_result["returncode"] != 0
            or not py.exists()
        ):
            print("VENV CREATION FAILED")

            records.append({
                "project": project,
                "version": version,
                "python_requirement":
                    metadata[
                        "python_requirement"
                    ],
                "venv_created": False,
                "install_status":
                    "NOT_RUN",
                "collection_status":
                    "NOT_RUN",
                "collected_tests": None,
            })

            continue

        # ---------------------------------------------
        # Upgrade packaging infrastructure
        # ---------------------------------------------

        bootstrap = run(
            [
                str(py),
                "-m",
                "pip",
                "install",
                "--upgrade",
                "pip",
                "setuptools",
                "wheel",
            ],
            timeout=300,
        )

        (
            log_dir
            / "01_bootstrap.txt"
        ).write_text(
            bootstrap["output"],
            encoding="utf-8"
        )

        # ---------------------------------------------
        # Install common test infrastructure
        # ---------------------------------------------

        common = run(
            [
                str(py),
                "-m",
                "pip",
                "install",
                "pytest",
                "pytest-cov",
                "coverage",
                "hypothesis",
                "packaging",
            ],
            timeout=600,
        )

        (
            log_dir
            / "02_common_test_tools.txt"
        ).write_text(
            common["output"],
            encoding="utf-8"
        )

        # ---------------------------------------------
        # Install project itself
        #
        # Priority:
        # 1. project-defined test extra if available
        # 2. project editable install
        # ---------------------------------------------

        if chosen_extra:
            install_target = (
                f".[ {chosen_extra} ]"
                .replace(" ", "")
            )
        else:
            install_target = "."

        install_result = run(
            [
                str(py),
                "-m",
                "pip",
                "install",
                "-e",
                install_target,
            ],
            cwd=project_dir,
            timeout=900,
        )

        (
            log_dir
            / "03_project_install.txt"
        ).write_text(
            install_result["output"],
            encoding="utf-8"
        )

        # ---------------------------------------------
        # If there is a PEP 735 dev/test dependency
        # group, install its simple dependency entries.
        # ---------------------------------------------

        dependency_group_install = None
        group_dependencies = []

        if chosen_dep_group:

            group_dependencies = (
                extract_dependency_group(
                    project_dir,
                    chosen_dep_group,
                )
            )

            if group_dependencies:

                dependency_group_install = run(
                    [
                        str(py),
                        "-m",
                        "pip",
                        "install",
                        *group_dependencies,
                    ],
                    cwd=project_dir,
                    timeout=900,
                )

                (
                    log_dir
                    / "04_dependency_group.txt"
                ).write_text(
                    dependency_group_install[
                        "output"
                    ],
                    encoding="utf-8"
                )

        # ---------------------------------------------
        # Collection only
        #
        # Important:
        # No test execution yet.
        # ---------------------------------------------

        collect_result = run(
            [
                str(py),
                "-m",
                "pytest",
                "--collect-only",
                "-q",
            ],
            cwd=project_dir,
            timeout=900,
        )

        (
            log_dir
            / "05_pytest_collect.txt"
        ).write_text(
            collect_result["output"],
            encoding="utf-8"
        )

        collected = count_collected_tests(
            collect_result["output"]
        )

        # ---------------------------------------------
        # pip check
        # ---------------------------------------------

        pip_check = run(
            [
                str(py),
                "-m",
                "pip",
                "check",
            ],
            timeout=180,
        )

        (
            log_dir
            / "06_pip_check.txt"
        ).write_text(
            pip_check["output"],
            encoding="utf-8"
        )

        # ---------------------------------------------
        # Status
        # ---------------------------------------------

        if install_result["returncode"] == 0:
            install_status = "PASS"
        else:
            install_status = "FAIL"

        if collect_result["returncode"] == 0:
            collection_status = "PASS"
        elif collect_result["timeout"]:
            collection_status = "TIMEOUT"
        else:
            collection_status = "FAIL"

        record = {
            "project":
                project,

            "version":
                version,

            "python_requirement":
                metadata[
                    "python_requirement"
                ],

            "optional_dependency_groups":
                ";".join(
                    optional_groups
                ),

            "chosen_optional_extra":
                chosen_extra,

            "dependency_groups":
                ";".join(
                    dependency_groups
                ),

            "chosen_dependency_group":
                chosen_dep_group,

            "venv_created":
                True,

            "bootstrap_returncode":
                bootstrap["returncode"],

            "common_tools_returncode":
                common["returncode"],

            "install_status":
                install_status,

            "install_returncode":
                install_result[
                    "returncode"
                ],

            "collection_status":
                collection_status,

            "collection_returncode":
                collect_result[
                    "returncode"
                ],

            "collection_seconds":
                collect_result[
                    "elapsed_seconds"
                ],

            "collected_tests":
                collected,

            "pip_check_returncode":
                pip_check[
                    "returncode"
                ],
        }

        records.append(record)

        print(
            f"Python requirement : "
            f"{metadata['python_requirement']}"
        )

        print(
            f"Optional extras    : "
            f"{optional_groups}"
        )

        print(
            f"Chosen extra       : "
            f"{chosen_extra}"
        )

        print(
            f"Dependency groups  : "
            f"{dependency_groups}"
        )

        print(
            f"Install            : "
            f"{install_status}"
        )

        print(
            f"pytest collection  : "
            f"{collection_status}"
        )

        print(
            f"Collected tests    : "
            f"{collected}"
        )

        print(
            f"Collection time    : "
            f"{collect_result['elapsed_seconds']} s"
        )


# ---------------------------------------------------------
# Save results
# ---------------------------------------------------------

json_path = (
    RESULT_ROOT
    / "tier3_dynamic_feasibility.json"
)

csv_path = (
    RESULT_ROOT
    / "tier3_dynamic_feasibility.csv"
)

json_path.write_text(
    json.dumps(
        records,
        indent=2
    ),
    encoding="utf-8"
)

all_fields = sorted(
    {
        key
        for row in records
        for key in row.keys()
    }
)

with csv_path.open(
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=all_fields
    )

    writer.writeheader()
    writer.writerows(records)


# ---------------------------------------------------------
# Final compact table
# ---------------------------------------------------------

print("\n")
print("=" * 78)
print("DYNAMIC FEASIBILITY SUMMARY")
print("=" * 78)

for r in records:

    tests = r.get(
        "collected_tests"
    )

    print(
        f"{r['project']:18s} "
        f"{r['version']:10s} "
        f"install={r['install_status']:4s} "
        f"collect={r['collection_status']:7s} "
        f"tests={str(tests):>7s}"
    )

print("\nSaved:")
print(csv_path)
print(json_path)

print(
    "\nDynamic feasibility screening completed."
)

In [ ]:
from pathlib import Path

ROOT = Path(r"<LOCAL_WORKSPACE>\dynamic_feasibility")

FAILED = {
    "cattrs": {
        "24.1.0": ["03_project_install.txt", "05_pytest_collect.txt"],
        "25.3.0": ["03_project_install.txt", "05_pytest_collect.txt"],
        "26.1.0": ["03_project_install.txt", "05_pytest_collect.txt"],
    },
    "boltons": {
        "24.1.0": ["05_pytest_collect.txt"],
        "25.0.0": ["05_pytest_collect.txt"],
    },
}

TAIL_LINES = 80

print("=" * 80)
print("EMSE TIER-3 FAILURE DIAGNOSTICS")
print("=" * 80)

for project, versions in FAILED.items():

    for version, filenames in versions.items():

        print("\n" + "#" * 80)
        print(f"{project.upper()} {version}")
        print("#" * 80)

        folder = ROOT / project / version

        for filename in filenames:

            path = folder / filename

            print("\n" + "-" * 80)
            print(filename)
            print("-" * 80)

            if not path.exists():
                print("LOG FILE NOT FOUND")
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            lines = text.splitlines()

            tail = lines[-TAIL_LINES:]

            for line in tail:
                print(line)

print("\n" + "=" * 80)
print("END OF FAILURE DIAGNOSTICS")
print("=" * 80)

In [ ]:
from pathlib import Path
import subprocess
import os
import re
import json
import time
import sys

ROOT = Path(r"<LOCAL_WORKSPACE>")
VENV_ROOT = ROOT / "_venvs"
OUT = ROOT / "compatibility_resolution"
OUT.mkdir(exist_ok=True)

def py(project, version):
    return VENV_ROOT / f"{project}_{version}" / "Scripts" / "python.exe"

def run(cmd, cwd=None, env=None, timeout=900):
    start = time.time()
    try:
        p = subprocess.run(
            [str(x) for x in cmd],
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout,
        )
        return p.returncode, p.stdout, round(time.time()-start, 3)
    except subprocess.TimeoutExpired as e:
        text = e.stdout or ""
        if isinstance(text, bytes):
            text = text.decode("utf-8", errors="replace")
        return -999, text, round(time.time()-start, 3)

def count_tests(text):
    pats = [
        r"(\d+)\s+tests?\s+collected",
        r"collected\s+(\d+)\s+items?",
    ]
    for pat in pats:
        m = re.findall(pat, text, re.I)
        if m:
            return int(m[-1])
    return None

results = []

# =========================================================
# A. CATTRS
# =========================================================

print("=" * 78)
print("A. CATTRS — VCS VERSION-METADATA RESOLUTION")
print("=" * 78)

for version in ["24.1.0", "25.3.0", "26.1.0"]:

    project = "cattrs"
    project_dir = ROOT / project / version
    python = py(project, version)
    log_dir = OUT / project / version
    log_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n--- cattrs {version} ---")

    env = os.environ.copy()

    # setuptools-scm documented escape hatch for archives
    env["SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS"] = version

    # Ensure core test dependencies that collection requires.
    rc_dep, out_dep, _ = run(
        [
            python, "-m", "pip", "install",
            "typing_extensions",
            "pytest",
            "pytest-cov",
            "coverage",
            "hypothesis",
        ],
        cwd=project_dir,
        env=env,
    )

    (log_dir / "01_dependencies.txt").write_text(
        out_dep, encoding="utf-8"
    )

    # Install frozen source without changing it.
    rc_install, out_install, sec_install = run(
        [
            python, "-m", "pip", "install",
            "-e", "."
        ],
        cwd=project_dir,
        env=env,
    )

    (log_dir / "02_install.txt").write_text(
        out_install, encoding="utf-8"
    )

    if rc_install == 0:
        rc_collect, out_collect, sec_collect = run(
            [
                python, "-m", "pytest",
                "--collect-only", "-q"
            ],
            cwd=project_dir,
            env=env,
        )
    else:
        rc_collect = None
        out_collect = "Collection not attempted: installation failed."
        sec_collect = 0

    (log_dir / "03_collect.txt").write_text(
        out_collect, encoding="utf-8"
    )

    n = count_tests(out_collect)

    print("Install :", "PASS" if rc_install == 0 else "FAIL")
    print("Collect :", "PASS" if rc_collect == 0 else "FAIL")
    print("Tests   :", n)

    results.append({
        "project": project,
        "version": version,
        "resolution":
            "SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS",
        "install_returncode": rc_install,
        "collection_returncode": rc_collect,
        "collected_tests": n,
        "install_seconds": sec_install,
        "collection_seconds": sec_collect,
    })


# =========================================================
# B. BOLTONS HISTORICAL PYTEST COMPATIBILITY
# =========================================================

print("\n" + "=" * 78)
print("B. BOLTONS — HISTORICAL PYTEST COMPATIBILITY")
print("=" * 78)

# Try newest plausible historical pytest first.
# Stop at first version that collects successfully.
PYTEST_CANDIDATES = [
    "pytest==7.4.4",
    "pytest==7.3.2",
    "pytest==7.2.2",
    "pytest==7.1.3",
]

for version in ["24.1.0", "25.0.0"]:

    project = "boltons"
    project_dir = ROOT / project / version
    python = py(project, version)
    log_dir = OUT / project / version
    log_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n--- boltons {version} ---")

    success = False

    for candidate in PYTEST_CANDIDATES:

        print("Trying", candidate)

        rc_install, install_output, _ = run(
            [
                python, "-m", "pip", "install",
                "--upgrade",
                "--force-reinstall",
                candidate,
            ],
            cwd=project_dir,
        )

        safe_name = candidate.replace("==", "_")

        (log_dir / f"install_{safe_name}.txt").write_text(
            install_output,
            encoding="utf-8"
        )

        rc_collect, collect_output, sec = run(
            [
                python, "-m", "pytest",
                "--collect-only", "-q"
            ],
            cwd=project_dir,
        )

        (log_dir / f"collect_{safe_name}.txt").write_text(
            collect_output,
            encoding="utf-8"
        )

        n = count_tests(collect_output)

        print(
            "   ",
            "PASS" if rc_collect == 0 else "FAIL",
            "| tests =", n
        )

        results.append({
            "project": project,
            "version": version,
            "resolution": candidate,
            "install_returncode": rc_install,
            "collection_returncode": rc_collect,
            "collected_tests": n,
            "collection_seconds": sec,
        })

        if rc_collect == 0:
            success = True
            print("Selected compatibility version:", candidate)
            break

    if not success:
        print("No candidate pytest version succeeded.")


# =========================================================
# SAVE
# =========================================================

result_file = OUT / "compatibility_resolution.json"

result_file.write_text(
    json.dumps(results, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 78)
print("COMPATIBILITY RESOLUTION SUMMARY")
print("=" * 78)

for row in results:
    if (
        row["project"] == "cattrs"
        or row["collection_returncode"] == 0
    ):
        print(
            f"{row['project']:12s} "
            f"{row['version']:10s} "
            f"{row['resolution']:45s} "
            f"install={str(row['install_returncode']):>3s} "
            f"collect={str(row['collection_returncode']):>4s} "
            f"tests={str(row['collected_tests']):>6s}"
        )

print("\nSaved:", result_file)
print("Compatibility-resolution stage completed.")

In [ ]:
from pathlib import Path

ROOT = Path(r"<LOCAL_WORKSPACE>\compatibility_resolution\cattrs")

VERSIONS = ["24.1.0", "25.3.0", "26.1.0"]

print("=" * 80)
print("CATTRS COMPATIBILITY INSTALL DIAGNOSTICS")
print("=" * 80)

for version in VERSIONS:

    path = ROOT / version / "02_install.txt"

    print("\n" + "#" * 80)
    print(f"CATTRS {version}")
    print("#" * 80)

    if not path.exists():
        print("INSTALL LOG NOT FOUND")
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    lines = text.splitlines()

    # Show only the final 100 lines, where the root error normally appears
    for line in lines[-100:]:
        print(line)

print("\n" + "=" * 80)
print("END CATTRS INSTALL DIAGNOSTICS")
print("=" * 80)

In [ ]:
from pathlib import Path
import re

ROOT = Path(r"<LOCAL_WORKSPACE>\compatibility_resolution\cattrs")
VERSIONS = ["24.1.0", "25.3.0", "26.1.0"]

PATTERNS = [
    r"traceback",
    r"lookuperror",
    r"valueerror",
    r"typeerror",
    r"error:",
    r"exception",
    r"setuptools[-_]scm",
    r"pretend",
    r"hatch",
    r"version",
    r"metadata",
]

CONTEXT = 8

print("=" * 84)
print("CATTRS ROOT-CAUSE EXTRACTION")
print("=" * 84)

for version in VERSIONS:

    path = ROOT / version / "02_install.txt"

    print("\n" + "#" * 84)
    print(f"CATTRS {version}")
    print("#" * 84)

    if not path.exists():
        print("LOG FILE NOT FOUND")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    hits = set()

    for i, line in enumerate(lines):
        lower = line.lower()

        if any(
            re.search(p, lower)
            for p in PATTERNS
        ):
            start = max(0, i - CONTEXT)
            end = min(len(lines), i + CONTEXT + 1)

            for j in range(start, end):
                hits.add(j)

    if not hits:
        print("No diagnostic patterns found.")
        continue

    previous = None

    for j in sorted(hits):

        if previous is not None and j > previous + 1:
            print("\n...")

        print(f"{j+1:04d}: {lines[j]}")

        previous = j

print("\n" + "=" * 84)
print("END ROOT-CAUSE EXTRACTION")
print("=" * 84)

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import shutil
import sys
import os
import re
import time
import tomllib
import venv

ROOT = Path(r"<LOCAL_WORKSPACE>")

FROZEN_ROOT = ROOT / "cattrs"
EXEC_ROOT = ROOT / "_git_exec" / "cattrs"
VENV_ROOT = ROOT / "_git_venvs"
RESULT_ROOT = ROOT / "git_execution_validation" / "cattrs"

EXEC_ROOT.mkdir(parents=True, exist_ok=True)
VENV_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

REPO = "https://github.com/python-attrs/cattrs.git"

VERSIONS = {
    "24.1.0": "v24.1.0",
    "25.3.0": "v25.3.0",
    "26.1.0": "v26.1.0",
}


# ==========================================================
# Helpers
# ==========================================================

def run(cmd, cwd=None, timeout=900, env=None):
    start = time.time()

    try:
        p = subprocess.run(
            [str(x) for x in cmd],
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout,
        )

        return {
            "returncode": p.returncode,
            "output": p.stdout,
            "seconds": round(time.time() - start, 3),
            "timeout": False,
        }

    except subprocess.TimeoutExpired as exc:

        text = exc.stdout or ""

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        return {
            "returncode": -999,
            "output": text,
            "seconds": round(time.time() - start, 3),
            "timeout": True,
        }


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def source_hashes(root):
    """
    Hash production Python source only.
    """
    src = root / "src" / "cattrs"

    result = {}

    if not src.exists():
        return result

    for p in sorted(src.rglob("*.py")):

        rel = p.relative_to(src)

        result[
            str(rel).replace("\\", "/")
        ] = sha256(p)

    return result


def compare_sources(frozen, gitcopy):

    a = source_hashes(frozen)
    b = source_hashes(gitcopy)

    a_names = set(a)
    b_names = set(b)

    added = sorted(b_names - a_names)
    missing = sorted(a_names - b_names)

    modified = sorted(
        name
        for name in (a_names & b_names)
        if a[name] != b[name]
    )

    identical = (
        not added
        and not missing
        and not modified
    )

    return {
        "identical": identical,
        "frozen_files": len(a),
        "git_files": len(b),
        "added": added,
        "missing": missing,
        "modified": modified,
    }


def python_exe(venv_dir):
    return venv_dir / "Scripts" / "python.exe"


def get_dependency_groups(project_dir):

    pyproject = project_dir / "pyproject.toml"

    if not pyproject.exists():
        return {}

    data = tomllib.loads(
        pyproject.read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

    groups = data.get(
        "dependency-groups",
        {}
    )

    if not isinstance(groups, dict):
        return {}

    return groups


def resolve_group(groups, name, seen=None):

    if seen is None:
        seen = set()

    if name in seen:
        return []

    seen.add(name)

    deps = []

    for item in groups.get(name, []):

        if isinstance(item, str):
            deps.append(item)

        elif isinstance(item, dict):

            include = item.get(
                "include-group"
            )

            if include:
                deps.extend(
                    resolve_group(
                        groups,
                        include,
                        seen
                    )
                )

    return deps


def detect_test_dependencies(project_dir):

    groups = get_dependency_groups(
        project_dir
    )

    preferred = [
        "test",
        "tests",
        "testing",
        "dev",
    ]

    for name in preferred:

        if name in groups:
            return name, resolve_group(
                groups,
                name
            )

    return None, []


def count_collected(text):

    patterns = [
        r"(\d+)\s+tests?\s+collected",
        r"collected\s+(\d+)\s+items?",
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.I
        )

        if matches:
            return int(matches[-1])

    return None


# ==========================================================
# Step 0: Verify Git
# ==========================================================

print("=" * 80)
print("CATTRS TAGGED-GIT EXECUTION VALIDATION")
print("=" * 80)

git_check = run(
    ["git", "--version"],
    timeout=60
)

if git_check["returncode"] != 0:

    raise RuntimeError(
        "\nGit is not available.\n\n"
        "In Anaconda Prompt run:\n\n"
        "conda install git -y\n\n"
        "Then restart this Jupyter kernel and rerun the cell."
    )

print(git_check["output"].strip())


# ==========================================================
# Process releases
# ==========================================================

records = []

for version, tag in VERSIONS.items():

    print("\n" + "=" * 80)
    print(f"CATTRS {version}")
    print("=" * 80)

    frozen_dir = (
        FROZEN_ROOT
        / version
    )

    git_dir = (
        EXEC_ROOT
        / version
    )

    venv_dir = (
        VENV_ROOT
        / f"cattrs_{version}"
    )

    log_dir = (
        RESULT_ROOT
        / version
    )

    log_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------------
    # 1. Fresh exact tagged checkout
    # ------------------------------------------------------

    if git_dir.exists():
        shutil.rmtree(
            git_dir,
            ignore_errors=True
        )

    clone = run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            tag,
            REPO,
            str(git_dir),
        ],
        timeout=600
    )

    (
        log_dir / "01_clone.txt"
    ).write_text(
        clone["output"],
        encoding="utf-8"
    )

    if clone["returncode"] != 0:

        print("Clone: FAIL")

        records.append({
            "project": "cattrs",
            "version": version,
            "tag": tag,
            "clone": "FAIL",
        })

        continue

    print("Clone: PASS")

    # ------------------------------------------------------
    # 2. Record exact commit
    # ------------------------------------------------------

    commit_result = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ],
        cwd=git_dir,
        timeout=60
    )

    commit = (
        commit_result["output"]
        .strip()
    )

    print("Commit:", commit)

    # ------------------------------------------------------
    # 3. Compare production code against frozen ZIP
    # ------------------------------------------------------

    comparison = compare_sources(
        frozen_dir,
        git_dir
    )

    print(
        "Frozen production files:",
        comparison["frozen_files"]
    )

    print(
        "Git production files   :",
        comparison["git_files"]
    )

    print(
        "Source equivalence      :",
        "PASS"
        if comparison["identical"]
        else "FAIL"
    )

    (
        log_dir
        / "02_source_equivalence.json"
    ).write_text(
        json.dumps(
            comparison,
            indent=2
        ),
        encoding="utf-8"
    )

    # Scientific safeguard:
    # Do not continue if subject code differs.
    if not comparison["identical"]:

        print(
            "\nSTOPPED: Git checkout differs "
            "from frozen ZIP production source."
        )

        records.append({
            "project": "cattrs",
            "version": version,
            "tag": tag,
            "commit": commit,
            "clone": "PASS",
            "source_equivalence": "FAIL",
            "install": "NOT_RUN",
            "collect": "NOT_RUN",
        })

        continue

    # ------------------------------------------------------
    # 4. Fresh isolated environment
    # ------------------------------------------------------

    if venv_dir.exists():
        shutil.rmtree(
            venv_dir,
            ignore_errors=True
        )

    venv.EnvBuilder(
        with_pip=True
    ).create(
        venv_dir
    )

    py = python_exe(
        venv_dir
    )

    # ------------------------------------------------------
    # 5. Packaging tools
    # ------------------------------------------------------

    bootstrap = run(
        [
            py,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "pip",
            "setuptools",
            "wheel",
        ],
        timeout=600
    )

    (
        log_dir
        / "03_bootstrap.txt"
    ).write_text(
        bootstrap["output"],
        encoding="utf-8"
    )

    # ------------------------------------------------------
    # 6. Install exact Git subject
    # ------------------------------------------------------

    install = run(
        [
            py,
            "-m",
            "pip",
            "install",
            "-e",
            ".",
        ],
        cwd=git_dir,
        timeout=900
    )

    (
        log_dir
        / "04_install.txt"
    ).write_text(
        install["output"],
        encoding="utf-8"
    )

    print(
        "Project install:",
        "PASS"
        if install["returncode"] == 0
        else "FAIL"
    )

    if install["returncode"] != 0:

        records.append({
            "project": "cattrs",
            "version": version,
            "tag": tag,
            "commit": commit,
            "clone": "PASS",
            "source_equivalence": "PASS",
            "install": "FAIL",
            "collect": "NOT_RUN",
        })

        continue

    # ------------------------------------------------------
    # 7. Install common testing infrastructure
    # ------------------------------------------------------

    common = run(
        [
            py,
            "-m",
            "pip",
            "install",
            "pytest",
            "pytest-cov",
            "coverage",
            "hypothesis",
            "typing_extensions",
        ],
        cwd=git_dir,
        timeout=900
    )

    (
        log_dir
        / "05_common_testing.txt"
    ).write_text(
        common["output"],
        encoding="utf-8"
    )

    # ------------------------------------------------------
    # 8. Install declared test dependency group if present
    # ------------------------------------------------------

    group_name, dependencies = (
        detect_test_dependencies(
            git_dir
        )
    )

    group_rc = 0

    if dependencies:

        dep_result = run(
            [
                py,
                "-m",
                "pip",
                "install",
                *dependencies,
            ],
            cwd=git_dir,
            timeout=1200
        )

        group_rc = (
            dep_result["returncode"]
        )

        (
            log_dir
            / "06_declared_test_dependencies.txt"
        ).write_text(
            dep_result["output"],
            encoding="utf-8"
        )

    print(
        "Declared test group:",
        group_name
    )

    print(
        "Declared test deps:",
        len(dependencies)
    )

    # ------------------------------------------------------
    # 9. pytest collection only
    # ------------------------------------------------------

    collect = run(
        [
            py,
            "-m",
            "pytest",
            "--collect-only",
            "-q",
        ],
        cwd=git_dir,
        timeout=1200
    )

    (
        log_dir
        / "07_collect.txt"
    ).write_text(
        collect["output"],
        encoding="utf-8"
    )

    test_count = count_collected(
        collect["output"]
    )

    print(
        "Collection:",
        "PASS"
        if collect["returncode"] == 0
        else "FAIL"
    )

    print(
        "Collected tests:",
        test_count
    )

    # ------------------------------------------------------
    # 10. pip dependency consistency
    # ------------------------------------------------------

    check = run(
        [
            py,
            "-m",
            "pip",
            "check",
        ],
        timeout=180
    )

    (
        log_dir
        / "08_pip_check.txt"
    ).write_text(
        check["output"],
        encoding="utf-8"
    )

    records.append({
        "project": "cattrs",
        "version": version,
        "tag": tag,
        "commit": commit,
        "clone": "PASS",
        "source_equivalence": (
            "PASS"
            if comparison["identical"]
            else "FAIL"
        ),
        "production_python_files":
            comparison["git_files"],
        "install": (
            "PASS"
            if install["returncode"] == 0
            else "FAIL"
        ),
        "declared_test_group":
            group_name,
        "declared_test_dependencies":
            len(dependencies),
        "dependency_install_returncode":
            group_rc,
        "collect": (
            "PASS"
            if collect["returncode"] == 0
            else "FAIL"
        ),
        "collected_tests":
            test_count,
        "collection_seconds":
            collect["seconds"],
        "pip_check_returncode":
            check["returncode"],
    })


# ==========================================================
# Save reproducibility result
# ==========================================================

output = (
    RESULT_ROOT
    / "cattrs_git_execution_validation.json"
)

output.write_text(
    json.dumps(
        records,
        indent=2
    ),
    encoding="utf-8"
)


# ==========================================================
# Final summary
# ==========================================================

print("\n")
print("=" * 80)
print("CATTRS GIT EXECUTION SUMMARY")
print("=" * 80)

for r in records:

    print(
        f"{r['version']:10s} "
        f"source={r.get('source_equivalence','?'):4s} "
        f"install={r.get('install','?'):7s} "
        f"collect={r.get('collect','?'):7s} "
        f"tests={str(r.get('collected_tests')):>6s} "
        f"commit={r.get('commit','')[:12]}"
    )

print("\nSaved:")
print(output)

print(
    "\nTagged-Git execution validation completed."
)

In [ ]:
import sys
import shutil
import subprocess

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Git:", shutil.which("git"))

if shutil.which("git"):
    print(
        subprocess.check_output(
            ["git", "--version"],
            text=True
        ).strip()
    )

In [ ]:
print("CATTRS TAGGED-GIT EXECUTION VALIDATION")

In [ ]:
import sys
import shutil
import subprocess

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Git path:", shutil.which("git"))

print("\nGit version:")
r = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
)
print("Return code:", r.returncode)
print(r.stdout)
print(r.stderr)

print("\nTesting access to cattrs repository:")
r = subprocess.run(
    ["git", "ls-remote", "--tags",
     "https://github.com/python-attrs/cattrs.git"],
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", r.returncode)

if r.returncode == 0:
    lines = r.stdout.splitlines()
    print("Repository access: PASS")
    print("Number of tag references:", len(lines))
    print("\nLast few tag references:")
    for line in lines[-10:]:
        print(line)
else:
    print("Repository access: FAIL")
    print(r.stderr)

In [ ]:
import sys
import shutil

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Git:", shutil.which("git"))

In [ ]:
import sys
import shutil

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Git:", shutil.which("git"))

In [ ]:
from pathlib import Path
import subprocess
import json
import re
import shutil
import time

print("CATTRS 24.1.0 — HISTORICAL LOCKED ENVIRONMENT VALIDATION")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

BASE_PY = Path(
    r"<LOCAL_USER_HOME>\conda_envs\emse-tier3\python.exe"
)

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

ENV = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
)

PY = ENV / "Scripts" / "python.exe"

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

LOCKED = [
    "attrs==23.1.0",
    "hypothesis==6.90.0",
    "pytest==8.0.0",
    "pytest-benchmark==4.0.0",
    "coverage==7.4.0",
    "immutables==0.20",
    "typing-extensions==4.8.0",
]

OPTIONAL_BACKENDS = [
    "PyYAML",
    "pymongo",
    "msgspec",
    "cbor2",
    "msgpack",
    "tomlkit",
]

def run(cmd, cwd=None, timeout=3600):
    return subprocess.run(
        cmd,
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout
    )

def parse_summary(output):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        m = re.findall(pattern, output, flags=re.I)
        if m:
            result[key] = int(m[-1])

    return result


report = {
    "source": str(SRC),
    "environment": str(ENV),
    "locked_versions": LOCKED,
    "status": None,
    "metrics": {},
    "python_version": None,
    "pytest_version": None,
    "pip_check": None,
}

# ==============================================================
# 1. Preconditions
# ==============================================================

if not BASE_PY.exists():
    raise FileNotFoundError(f"Base Python not found:\n{BASE_PY}")

if not SRC.exists():
    raise FileNotFoundError(f"Source not found:\n{SRC}")


# ==============================================================
# 2. Fresh environment
# ==============================================================

if ENV.exists():
    print("Removing old historical environment...")
    shutil.rmtree(ENV, ignore_errors=True)

ENV.parent.mkdir(parents=True, exist_ok=True)

print("\nCreating fresh Python 3.12 environment...")

p = run([
    str(BASE_PY),
    "-m",
    "venv",
    str(ENV)
])

if p.returncode != 0:
    raise RuntimeError(
        "Environment creation failed:\n" +
        p.stdout + "\n" + p.stderr
    )


# ==============================================================
# 3. Packaging tools
# ==============================================================

print("Installing packaging tools...")

p = run([
    str(PY),
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel"
])

if p.returncode != 0:
    raise RuntimeError(p.stdout + "\n" + p.stderr)


# ==============================================================
# 4. Install exact historical locked versions
# ==============================================================

print("\nInstalling exact historical locked dependencies...")

p = run([
    str(PY),
    "-m",
    "pip",
    "install",
    *LOCKED
], cwd=SRC)

if p.returncode != 0:
    print(p.stdout)
    print(p.stderr)
    raise RuntimeError("Historical dependency installation failed.")


# ==============================================================
# 5. Install required optional backends
# ==============================================================

print("\nInstalling optional serialization backends...")

p = run([
    str(PY),
    "-m",
    "pip",
    "install",
    *OPTIONAL_BACKENDS
], cwd=SRC)

if p.returncode != 0:
    print(p.stdout)
    print(p.stderr)
    raise RuntimeError("Optional backend installation failed.")


# ==============================================================
# 6. Install cattrs source WITHOUT dependency upgrading
# ==============================================================

print("\nInstalling cattrs 24.1.0 source without dependency resolution...")

p = run([
    str(PY),
    "-m",
    "pip",
    "install",
    "--no-deps",
    "-e",
    "."
], cwd=SRC)

if p.returncode != 0:
    print(p.stdout)
    print(p.stderr)
    raise RuntimeError("cattrs editable installation failed.")


# ==============================================================
# 7. Record exact environment
# ==============================================================

print("\nExact environment:")
print("-" * 100)

packages = [
    "cattrs",
    "attrs",
    "hypothesis",
    "pytest",
    "pytest-benchmark",
    "coverage",
    "immutables",
    "typing-extensions",
]

versions = {}

for pkg in packages:
    p = run([
        str(PY),
        "-c",
        (
            "import importlib.metadata as m; "
            f"print(m.version('{pkg}'))"
        )
    ])

    version = (
        p.stdout.strip()
        if p.returncode == 0
        else "NOT_INSTALLED"
    )

    versions[pkg] = version
    print(f"{pkg:20s}: {version}")

report["resolved_versions"] = versions


# ==============================================================
# 8. Python / pytest / pip check
# ==============================================================

p = run([
    str(PY),
    "-c",
    "import sys; print(sys.version)"
])

report["python_version"] = p.stdout.strip()
print("\nPython:")
print(report["python_version"])

p = run([
    str(PY),
    "-m",
    "pytest",
    "--version"
], cwd=SRC)

report["pytest_version"] = p.stdout.strip()
print("\npytest:")
print(report["pytest_version"])

p = run([
    str(PY),
    "-m",
    "pip",
    "check"
])

report["pip_check"] = (
    "PASS" if p.returncode == 0 else "FAIL"
)

print("\npip check:", report["pip_check"])

if p.stdout.strip():
    print(p.stdout.strip())

if p.stderr.strip():
    print(p.stderr.strip())


# ==============================================================
# 9. Functional suite — exclude benchmarks
# ==============================================================

print("\nRunning cattrs 24.1.0 functional suite...")
print("(bench directory excluded)")
print("-" * 100)

start = time.perf_counter()

p = run([
    str(PY),
    "-m",
    "pytest",
    "-q",
    "--ignore=bench"
], cwd=SRC, timeout=3600)

elapsed = time.perf_counter() - start

output = p.stdout + "\n" + p.stderr

metrics = parse_summary(output)

report["status"] = (
    "PASS" if p.returncode == 0 else "FAIL"
)

report["metrics"] = metrics
report["runtime_seconds"] = round(elapsed, 3)
report["return_code"] = p.returncode

print(output[-6000:])


# ==============================================================
# 10. Save raw log
# ==============================================================

log_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_baseline.log"
)

with open(
    log_file,
    "w",
    encoding="utf-8",
    errors="replace"
) as f:
    f.write(output)


# ==============================================================
# 11. Save JSON
# ==============================================================

json_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_validation.json"
)

with open(
    json_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==============================================================
# 12. Final summary
# ==============================================================

print("\n" + "=" * 100)
print("CATTRS 24.1.0 HISTORICAL ENVIRONMENT SUMMARY")
print("=" * 100)

print("Status :", report["status"])
print("Passed :", metrics["passed"])
print("Failed :", metrics["failed"])
print("Errors :", metrics["errors"])
print("Skipped:", metrics["skipped"])
print("XFailed:", metrics["xfailed"])
print("XPassed:", metrics["xpassed"])
print("Runtime:", report["runtime_seconds"], "s")
print("pip check:", report["pip_check"])

print("\nResolved versions:")
for pkg, version in versions.items():
    print(f"  {pkg:20s} {version}")

print("\nSaved:")
print(json_file)
print(log_file)

In [ ]:
from pathlib import Path
import subprocess
import json
import re
import time

print("CATTRS 24.1.0 — HISTORICAL ENVIRONMENT RECOVERY")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

ENV = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
)

PY = ENV / "Scripts" / "python.exe"

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, timeout=3600):
    return subprocess.run(
        cmd,
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout
    )


def parse_summary(output):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        m = re.findall(pattern, output, flags=re.I)
        if m:
            result[key] = int(m[-1])

    return result


if not SRC.exists():
    raise FileNotFoundError(f"Source not found:\n{SRC}")

if not PY.exists():
    raise FileNotFoundError(
        "Historical environment not found.\n"
        f"{PY}\n\n"
        "Re-run the previous cell through the dependency-installation stage."
    )


# ==============================================================
# 1. Show current environment
# ==============================================================

print("\nCurrent environment versions")
print("-" * 100)

packages = [
    "attrs",
    "hypothesis",
    "pytest",
    "pytest-benchmark",
    "coverage",
    "immutables",
    "typing-extensions",
]

versions = {}

for pkg in packages:
    p = run([
        str(PY),
        "-c",
        (
            "import importlib.metadata as m; "
            f"print(m.version('{pkg}'))"
        )
    ])

    value = (
        p.stdout.strip()
        if p.returncode == 0
        else "NOT_INSTALLED"
    )

    versions[pkg] = value
    print(f"{pkg:20s}: {value}")


# ==============================================================
# 2. Install cattrs NON-EDITABLY, without dependency resolution
# ==============================================================

print("\nInstalling frozen cattrs 24.1.0 source non-editably...")
print("-" * 100)

p = run([
    str(PY),
    "-m",
    "pip",
    "install",
    "--no-deps",
    "."
], cwd=SRC)

print(p.stdout[-5000:])
print(p.stderr[-5000:])

if p.returncode != 0:
    print("\nSTANDARD INSTALL FAILED.")
    print("Trying legacy-compatible no-build-isolation installation...")

    p = run([
        str(PY),
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        "."
    ], cwd=SRC)

    print(p.stdout[-5000:])
    print(p.stderr[-5000:])


if p.returncode != 0:
    raise RuntimeError(
        "cattrs non-editable installation also failed. "
        "Please paste the last pip error block."
    )


# ==============================================================
# 3. Confirm installed cattrs version
# ==============================================================

p = run([
    str(PY),
    "-c",
    (
        "import importlib.metadata as m; "
        "print(m.version('cattrs'))"
    )
])

cattrs_version = (
    p.stdout.strip()
    if p.returncode == 0
    else "NOT_INSTALLED"
)

print("\nInstalled cattrs:", cattrs_version)


# ==============================================================
# 4. pip check
# ==============================================================

p = run([
    str(PY),
    "-m",
    "pip",
    "check"
])

pip_check = (
    "PASS" if p.returncode == 0 else "FAIL"
)

print("pip check:", pip_check)

if p.stdout.strip():
    print(p.stdout.strip())

if p.stderr.strip():
    print(p.stderr.strip())


# ==============================================================
# 5. Run functional suite from source tree
# ==============================================================

print("\nRunning functional suite...")
print("(bench directory excluded)")
print("-" * 100)

start = time.perf_counter()

p = run([
    str(PY),
    "-m",
    "pytest",
    "-q",
    "--ignore=bench"
], cwd=SRC, timeout=3600)

elapsed = time.perf_counter() - start

output = p.stdout + "\n" + p.stderr

metrics = parse_summary(output)

status = (
    "PASS" if p.returncode == 0 else "FAIL"
)

print(output[-6000:])


# ==============================================================
# 6. Save evidence
# ==============================================================

report = {
    "status": status,
    "cattrs_version": cattrs_version,
    "resolved_versions": versions,
    "pip_check": pip_check,
    "metrics": metrics,
    "runtime_seconds": round(elapsed, 3),
    "return_code": p.returncode,
    "install_mode": "non_editable_no_deps",
}

json_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_recovery.json"
)

with open(
    json_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

log_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_recovery.log"
)

with open(
    log_file,
    "w",
    encoding="utf-8",
    errors="replace"
) as f:
    f.write(output)


# ==============================================================
# 7. Compact summary
# ==============================================================

print("\n" + "=" * 100)
print("CATTRS 24.1.0 RECOVERY SUMMARY")
print("=" * 100)

print("Status :", status)
print("Passed :", metrics["passed"])
print("Failed :", metrics["failed"])
print("Errors :", metrics["errors"])
print("Skipped:", metrics["skipped"])
print("XFailed:", metrics["xfailed"])
print("XPassed:", metrics["xpassed"])
print("Runtime:", round(elapsed, 3), "s")
print("pip check:", pip_check)
print("cattrs:", cattrs_version)

print("\nSaved:")
print(json_file)
print(log_file)

In [ ]:
from pathlib import Path
import subprocess

print("CATTRS 24.1.0 — INSTALL FAILURE DIAGNOSTIC")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

ENV = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
)

PY = ENV / "Scripts" / "python.exe"

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

if not SRC.exists():
    raise FileNotFoundError(f"Source not found: {SRC}")

if not PY.exists():
    raise FileNotFoundError(f"Python not found: {PY}")


def run(cmd, cwd=None):
    return subprocess.run(
        cmd,
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=1800
    )


# --------------------------------------------------------------
# 1. Python / pip
# --------------------------------------------------------------

print("\n1. PYTHON / PIP")
print("-" * 100)

p = run([str(PY), "--version"])
print(p.stdout.strip() or p.stderr.strip())

p = run([str(PY), "-m", "pip", "--version"])
print(p.stdout.strip() or p.stderr.strip())


# --------------------------------------------------------------
# 2. Git provenance
# --------------------------------------------------------------

print("\n2. GIT STATUS / TAG / COMMIT")
print("-" * 100)

for cmd in [
    ["git", "status", "--short"],
    ["git", "rev-parse", "HEAD"],
    ["git", "describe", "--tags", "--always"],
]:
    p = run(cmd, cwd=SRC)

    print("$", " ".join(cmd))
    print(p.stdout.strip() or p.stderr.strip() or "(no output)")


# --------------------------------------------------------------
# 3. Build backend from pyproject
# --------------------------------------------------------------

print("\n3. PYPROJECT BUILD SYSTEM")
print("-" * 100)

import tomllib

with open(SRC / "pyproject.toml", "rb") as f:
    data = tomllib.load(f)

build = data.get("build-system", {})

print("requires:")
for item in build.get("requires", []):
    print("  -", item)

print("build-backend:", build.get("build-backend"))


# --------------------------------------------------------------
# 4. Installed packaging/build tools
# --------------------------------------------------------------

print("\n4. BUILD TOOL VERSIONS")
print("-" * 100)

packages = [
    "pip",
    "setuptools",
    "wheel",
    "hatchling",
    "hatch-vcs",
    "setuptools-scm",
]

for package in packages:
    code = (
        "import importlib.metadata as m; "
        f"\ntry:\n print(m.version('{package}'))"
        "\nexcept m.PackageNotFoundError:\n print('NOT_INSTALLED')"
    )

    p = run([str(PY), "-c", code])

    print(
        f"{package:20s}: "
        f"{p.stdout.strip() or p.stderr.strip()}"
    )


# --------------------------------------------------------------
# 5. Standard isolated installation — verbose
# --------------------------------------------------------------

print("\n5. STANDARD ISOLATED BUILD")
print("-" * 100)

p1 = run([
    str(PY),
    "-m",
    "pip",
    "install",
    "-vvv",
    "--no-deps",
    "."
], cwd=SRC)

isolated_output = p1.stdout + "\n" + p1.stderr

print("Return code:", p1.returncode)
print("\n--- LAST 12000 CHARACTERS ---\n")
print(isolated_output[-12000:])


# --------------------------------------------------------------
# 6. No-build-isolation installation — verbose
# --------------------------------------------------------------

print("\n6. NO-BUILD-ISOLATION BUILD")
print("-" * 100)

p2 = run([
    str(PY),
    "-m",
    "pip",
    "install",
    "-vvv",
    "--no-deps",
    "--no-build-isolation",
    "."
], cwd=SRC)

nonisolated_output = p2.stdout + "\n" + p2.stderr

print("Return code:", p2.returncode)
print("\n--- LAST 12000 CHARACTERS ---\n")
print(nonisolated_output[-12000:])


# --------------------------------------------------------------
# 7. Save complete diagnostic logs
# --------------------------------------------------------------

isolated_log = (
    RESULT_DIR
    / "cattrs_24.1_install_isolated_verbose.log"
)

nonisolated_log = (
    RESULT_DIR
    / "cattrs_24.1_install_no_build_isolation_verbose.log"
)

isolated_log.write_text(
    isolated_output,
    encoding="utf-8",
    errors="replace"
)

nonisolated_log.write_text(
    nonisolated_output,
    encoding="utf-8",
    errors="replace"
)


# --------------------------------------------------------------
# 8. Compact final result
# --------------------------------------------------------------

print("\n" + "=" * 100)
print("INSTALL DIAGNOSTIC SUMMARY")
print("=" * 100)

print("Standard isolated install return code     :", p1.returncode)
print("No-build-isolation install return code    :", p2.returncode)

print("\nLogs:")
print(isolated_log)
print(nonisolated_log)

print("\nPlease send:")
print("1. BUILD TOOL VERSIONS")
print("2. The last error block from STANDARD ISOLATED BUILD")
print("3. The last error block from NO-BUILD-ISOLATION BUILD")

In [ ]:
from pathlib import Path
import subprocess
import tomllib
import importlib.metadata as metadata

print("CATTRS 24.1.0 — INSTALL FAILURE DIAGNOSTIC (CORRECTED)")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

ENV = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
)

PY = ENV / "Scripts" / "python.exe"

GIT = Path(
    r"<LOCAL_USER_HOME>\conda_envs\emse-tier3\Library\mingw64\bin\git.EXE"
)

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, timeout=1800):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout
    )


# ==============================================================
# 1. Preconditions
# ==============================================================

print("\n1. PRECONDITIONS")
print("-" * 100)

for label, path in [
    ("Source", SRC),
    ("Historical Python", PY),
    ("Git executable", GIT),
]:
    print(f"{label:20s}: {path}")
    print(f"{'':20s}  exists={path.exists()}")

if not SRC.exists():
    raise FileNotFoundError(f"Source not found: {SRC}")

if not PY.exists():
    raise FileNotFoundError(f"Historical Python not found: {PY}")

if not GIT.exists():
    raise FileNotFoundError(f"Git executable not found: {GIT}")


# ==============================================================
# 2. Python / pip
# ==============================================================

print("\n2. PYTHON / PIP")
print("-" * 100)

for cmd in [
    [PY, "--version"],
    [PY, "-m", "pip", "--version"],
]:
    p = run(cmd)
    print("$", " ".join(str(x) for x in cmd))
    print(p.stdout.strip() or p.stderr.strip())


# ==============================================================
# 3. Git provenance
# ==============================================================

print("\n3. GIT STATUS / COMMIT / TAG")
print("-" * 100)

git_results = {}

git_commands = {
    "status": [GIT, "status", "--short"],
    "commit": [GIT, "rev-parse", "HEAD"],
    "describe": [GIT, "describe", "--tags", "--always"],
}

for name, cmd in git_commands.items():
    p = run(cmd, cwd=SRC)

    git_results[name] = {
        "returncode": p.returncode,
        "stdout": p.stdout.strip(),
        "stderr": p.stderr.strip(),
    }

    print(f"\n{name}:")
    print(p.stdout.strip() or p.stderr.strip() or "(no output)")


# ==============================================================
# 4. Build backend
# ==============================================================

print("\n4. PYPROJECT BUILD SYSTEM")
print("-" * 100)

with open(SRC / "pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

build = pyproject.get("build-system", {})

print("requires:")
for item in build.get("requires", []):
    print("  -", item)

print("build-backend:", build.get("build-backend"))


# ==============================================================
# 5. Build tool versions INSIDE historical environment
# ==============================================================

print("\n5. BUILD TOOL VERSIONS")
print("-" * 100)

build_tools = [
    "pip",
    "setuptools",
    "wheel",
    "hatchling",
    "hatch-vcs",
    "setuptools-scm",
]

for package in build_tools:
    code = f"""
import importlib.metadata as m
try:
    print(m.version({package!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    p = run([PY, "-c", code])

    value = p.stdout.strip() or p.stderr.strip()
    print(f"{package:20s}: {value}")


# ==============================================================
# 6. Verify Git works from the historical environment context
# ==============================================================

print("\n6. GIT ACCESS CHECK")
print("-" * 100)

p = run(
    [GIT, "rev-parse", "--is-inside-work-tree"],
    cwd=SRC
)

print("Return code:", p.returncode)
print(p.stdout.strip() or p.stderr.strip())


# ==============================================================
# 7. Standard isolated install
# ==============================================================

print("\n7. STANDARD ISOLATED INSTALL")
print("-" * 100)

p1 = run([
    PY,
    "-m",
    "pip",
    "install",
    "-vvv",
    "--no-deps",
    "."
], cwd=SRC)

isolated_output = p1.stdout + "\n" + p1.stderr

print("Return code:", p1.returncode)
print("\n--- LAST 10000 CHARACTERS ---\n")
print(isolated_output[-10000:])


# ==============================================================
# 8. No-build-isolation install
# ==============================================================

print("\n8. NO-BUILD-ISOLATION INSTALL")
print("-" * 100)

p2 = run([
    PY,
    "-m",
    "pip",
    "install",
    "-vvv",
    "--no-deps",
    "--no-build-isolation",
    "."
], cwd=SRC)

nonisolated_output = p2.stdout + "\n" + p2.stderr

print("Return code:", p2.returncode)
print("\n--- LAST 10000 CHARACTERS ---\n")
print(nonisolated_output[-10000:])


# ==============================================================
# 9. Save logs
# ==============================================================

isolated_log = (
    RESULT_DIR
    / "cattrs_24.1_install_isolated_verbose.log"
)

nonisolated_log = (
    RESULT_DIR
    / "cattrs_24.1_install_no_build_isolation_verbose.log"
)

isolated_log.write_text(
    isolated_output,
    encoding="utf-8",
    errors="replace"
)

nonisolated_log.write_text(
    nonisolated_output,
    encoding="utf-8",
    errors="replace"
)


# ==============================================================
# 10. Final summary
# ==============================================================

print("\n" + "=" * 100)
print("INSTALL DIAGNOSTIC SUMMARY")
print("=" * 100)

print("Git commit     :", git_results["commit"]["stdout"])
print("Git describe   :", git_results["describe"]["stdout"])
print("Git worktree   :", "CLEAN" if not git_results["status"]["stdout"] else "MODIFIED")

print("Standard isolated install return code  :", p1.returncode)
print("No-build-isolation return code         :", p2.returncode)

print("\nSaved:")
print(isolated_log)
print(nonisolated_log)

print("\nPlease send:")
print("1. BUILD TOOL VERSIONS")
print("2. PYPROJECT BUILD SYSTEM")
print("3. Last error block from STANDARD ISOLATED INSTALL")
print("4. Last error block from NO-BUILD-ISOLATION INSTALL")

In [ ]:
from pathlib import Path
import subprocess
import tomllib
import json
import re
import time

print("CATTRS 24.1.0 — BUILD-BACKEND RECOVERY AND BASELINE VALIDATION")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

ENV = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
)

PY = ENV / "Scripts" / "python.exe"

GIT = Path(
    r"<LOCAL_USER_HOME>\conda_envs\emse-tier3\Library\mingw64\bin\git.EXE"
)

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, timeout=3600):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout
    )

def version(pkg):
    code = f"""
import importlib.metadata as m
try:
    print(m.version({pkg!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    p = run([PY, "-c", code])
    return p.stdout.strip() or "UNKNOWN"

def parse_summary(output):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        hits = re.findall(pattern, output, flags=re.I)
        if hits:
            result[key] = int(hits[-1])

    return result


# ==============================================================
# 1. Preconditions and provenance
# ==============================================================

for path in [SRC, PY, GIT]:
    if not path.exists():
        raise FileNotFoundError(path)

commit = run(
    [GIT, "rev-parse", "HEAD"],
    cwd=SRC
).stdout.strip()

tag = run(
    [GIT, "describe", "--tags", "--exact-match"],
    cwd=SRC
).stdout.strip()

git_status = run(
    [GIT, "status", "--short"],
    cwd=SRC
).stdout.strip()

print("\nProvenance")
print("-" * 100)
print("Commit   :", commit)
print("Tag      :", tag)
print("Worktree :", "CLEAN" if not git_status else "MODIFIED")

if git_status:
    raise RuntimeError(
        "Worktree is not clean. Stop before proceeding."
    )


# ==============================================================
# 2. Read EXACT build requirements from frozen pyproject
# ==============================================================

with open(SRC / "pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

build_system = pyproject.get("build-system", {})
build_requires = build_system.get("requires", [])
build_backend = build_system.get("build-backend")

print("\nFrozen pyproject build configuration")
print("-" * 100)

print("Build backend:", build_backend)
print("Build requirements:")

for req in build_requires:
    print("  -", req)

if not build_requires:
    raise RuntimeError(
        "No build-system requirements found in pyproject.toml."
    )


# ==============================================================
# 3. Verify locked TEST/RUNTIME dependencies BEFORE build fix
# ==============================================================

locked_expected = {
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
    "pytest": "8.0.0",
    "pytest-benchmark": "4.0.0",
    "coverage": "7.4.0",
    "immutables": "0.20",
    "typing-extensions": "4.8.0",
}

print("\nLocked runtime/test environment BEFORE build-backend installation")
print("-" * 100)

before = {}

for pkg, expected in locked_expected.items():
    actual = version(pkg)
    before[pkg] = actual

    marker = "OK" if actual == expected else "MISMATCH"

    print(
        f"{pkg:20s} "
        f"expected={expected:10s} "
        f"actual={actual:10s} "
        f"{marker}"
    )

mismatches = {
    pkg: (expected, before[pkg])
    for pkg, expected in locked_expected.items()
    if before[pkg] != expected
}

if mismatches:
    raise RuntimeError(
        "Locked dependency environment has changed. "
        f"Mismatches: {mismatches}"
    )


# ==============================================================
# 4. Install ONLY build-system requirements declared upstream
# ==============================================================

print("\nInstalling upstream-declared build-system requirements...")
print("-" * 100)

p = run([
    PY,
    "-m",
    "pip",
    "install",
    *build_requires
], cwd=SRC)

print(p.stdout[-4000:])
print(p.stderr[-4000:])

if p.returncode != 0:
    raise RuntimeError(
        "Upstream-declared build requirements could not be installed."
    )


# ==============================================================
# 5. Show resulting build tools
# ==============================================================

print("\nBuild tools now available")
print("-" * 100)

for pkg in [
    "hatchling",
    "hatch-vcs",
    "setuptools-scm",
]:
    print(f"{pkg:20s}: {version(pkg)}")


# ==============================================================
# 6. Verify locked dependencies did NOT drift
# ==============================================================

print("\nLocked environment AFTER build-backend installation")
print("-" * 100)

after = {}

for pkg, expected in locked_expected.items():
    actual = version(pkg)
    after[pkg] = actual

    marker = "OK" if actual == expected else "MISMATCH"

    print(
        f"{pkg:20s} "
        f"expected={expected:10s} "
        f"actual={actual:10s} "
        f"{marker}"
    )

post_mismatches = {
    pkg: (expected, after[pkg])
    for pkg, expected in locked_expected.items()
    if after[pkg] != expected
}

if post_mismatches:
    raise RuntimeError(
        "Build-tool installation changed locked runtime/test "
        f"dependencies: {post_mismatches}"
    )


# ==============================================================
# 7. Install cattrs from frozen source, NO dependency resolution
# ==============================================================

print("\nInstalling frozen cattrs 24.1.0 source...")
print("-" * 100)

p = run([
    PY,
    "-m",
    "pip",
    "install",
    "--no-deps",
    "--no-build-isolation",
    "."
], cwd=SRC)

install_output = p.stdout + "\n" + p.stderr

print(install_output[-6000:])

install_log = (
    RESULT_DIR
    / "cattrs_24.1_build_backend_recovery_install.log"
)

install_log.write_text(
    install_output,
    encoding="utf-8",
    errors="replace"
)

if p.returncode != 0:
    print("\nINSTALL STILL FAILED.")
    print("Saved:", install_log)
    print("\nSend me the final error block above.")
else:
    print("\nInstallation PASS.")


# ==============================================================
# 8. Continue only if installation succeeded
# ==============================================================

if p.returncode == 0:

    installed_cattrs = version("cattrs")

    print("Installed cattrs:", installed_cattrs)

    # ----------------------------------------------------------
    # pip check
    # ----------------------------------------------------------

    pc = run([
        PY,
        "-m",
        "pip",
        "check"
    ])

    pip_check = "PASS" if pc.returncode == 0 else "FAIL"

    print("\npip check:", pip_check)

    if pc.stdout.strip():
        print(pc.stdout.strip())

    if pc.stderr.strip():
        print(pc.stderr.strip())


    # ----------------------------------------------------------
    # 9. Functional native suite
    # ----------------------------------------------------------

    print("\nRunning functional native suite...")
    print("Benchmark directory excluded.")
    print("-" * 100)

    start = time.perf_counter()

    test = run([
        PY,
        "-m",
        "pytest",
        "-q",
        "--ignore=bench"
    ], cwd=SRC, timeout=3600)

    elapsed = time.perf_counter() - start

    test_output = test.stdout + "\n" + test.stderr
    metrics = parse_summary(test_output)

    status = "PASS" if test.returncode == 0 else "FAIL"

    print(test_output[-7000:])


    # ----------------------------------------------------------
    # 10. Verify source remained untouched
    # ----------------------------------------------------------

    final_git_status = run(
        [GIT, "status", "--short"],
        cwd=SRC
    ).stdout.strip()

    source_integrity = (
        "CLEAN"
        if not final_git_status
        else "MODIFIED"
    )


    # ----------------------------------------------------------
    # 11. Save evidence
    # ----------------------------------------------------------

    test_log = (
        RESULT_DIR
        / "cattrs_24.1_historical_functional_baseline.log"
    )

    test_log.write_text(
        test_output,
        encoding="utf-8",
        errors="replace"
    )

    report = {
        "subject": "cattrs",
        "release": "24.1.0",
        "commit": commit,
        "tag": tag,
        "python": "3.12.14",
        "build_backend": build_backend,
        "build_requirements": build_requires,
        "locked_dependencies": after,
        "installed_cattrs": installed_cattrs,
        "pip_check": pip_check,
        "functional_suite": {
            "bench_excluded": True,
            "status": status,
            **metrics,
            "runtime_seconds": round(elapsed, 3),
        },
        "source_integrity": source_integrity,
    }

    json_file = (
        RESULT_DIR
        / "cattrs_24.1_historical_environment_final.json"
    )

    json_file.write_text(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


    # ----------------------------------------------------------
    # 12. Final summary
    # ----------------------------------------------------------

    print("\n" + "=" * 100)
    print("CATTRS 24.1.0 FINAL HISTORICAL BASELINE SUMMARY")
    print("=" * 100)

    print("Status          :", status)
    print("Passed          :", metrics["passed"])
    print("Failed          :", metrics["failed"])
    print("Errors          :", metrics["errors"])
    print("Skipped         :", metrics["skipped"])
    print("XFailed         :", metrics["xfailed"])
    print("XPassed         :", metrics["xpassed"])
    print("Runtime         :", round(elapsed, 3), "s")
    print("pip check       :", pip_check)
    print("Source integrity:", source_integrity)
    print("cattrs          :", installed_cattrs)

    print("\nLocked dependencies:")
    for pkg, ver in after.items():
        print(f"  {pkg:20s} {ver}")

    print("\nSaved:")
    print(json_file)
    print(test_log)
    print(install_log)

In [ ]:
from pathlib import Path
import subprocess
import tomllib
import json
import re
import time
import os

print("CATTRS 24.1.0 — VCS-AWARE BUILD RECOVERY")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

ENV = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
)

PY = ENV / "Scripts" / "python.exe"

GIT = Path(
    r"<LOCAL_USER_HOME>\conda_envs\emse-tier3\Library\mingw64\bin\git.EXE"
)

GIT_DIR = GIT.parent

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


def make_env():
    env = os.environ.copy()

    # Put the validated Git directory first so build backends can call `git`.
    current_path = env.get("PATH", "")
    env["PATH"] = str(GIT_DIR) + os.pathsep + current_path

    return env


BUILD_ENV = make_env()


def run(cmd, cwd=None, timeout=3600, env=BUILD_ENV):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env=env
    )


def version(pkg):
    code = f"""
import importlib.metadata as m
try:
    print(m.version({pkg!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    p = run([PY, "-c", code])
    return p.stdout.strip() or "UNKNOWN"


def parse_summary(output):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        hits = re.findall(pattern, output, flags=re.I)
        if hits:
            result[key] = int(hits[-1])

    return result


# ==============================================================
# 1. Preconditions
# ==============================================================

print("\n1. PRECONDITIONS")
print("-" * 100)

for label, path in [
    ("Source", SRC),
    ("Python", PY),
    ("Git", GIT),
]:
    print(f"{label:15s}: {path}")
    print(f"{'':15s}  exists={path.exists()}")

if not SRC.exists():
    raise FileNotFoundError(SRC)

if not PY.exists():
    raise FileNotFoundError(PY)

if not GIT.exists():
    raise FileNotFoundError(GIT)


# ==============================================================
# 2. Confirm Git is now visible BY NAME
# ==============================================================

print("\n2. GIT VISIBILITY INSIDE BUILD ENVIRONMENT")
print("-" * 100)

p = run(
    ["git", "--version"],
    cwd=SRC
)

print("git --version:")
print(p.stdout.strip() or p.stderr.strip())

if p.returncode != 0:
    raise RuntimeError(
        "Git is still not visible by name inside the build environment."
    )

p = run(
    ["git", "rev-parse", "HEAD"],
    cwd=SRC
)

commit = p.stdout.strip()

print("\nCommit:")
print(commit)

p = run(
    ["git", "describe", "--tags", "--exact-match"],
    cwd=SRC
)

tag = p.stdout.strip()

print("\nExact tag:")
print(tag)

p = run(
    ["git", "status", "--short"],
    cwd=SRC
)

git_status = p.stdout.strip()

print("\nWorktree:")
print("CLEAN" if not git_status else git_status)

if git_status:
    raise RuntimeError(
        "Source tree is modified. Stop before continuing."
    )


# ==============================================================
# 3. Verify setuptools-scm directly
# ==============================================================

print("\n3. SETUPTOOLS-SCM VERSION DISCOVERY")
print("-" * 100)

code = r"""
from setuptools_scm import get_version
print(get_version(root=".", relative_to=None))
"""

p = run(
    [PY, "-c", code],
    cwd=SRC
)

scm_output = p.stdout.strip()
scm_error = p.stderr.strip()

print("Return code:", p.returncode)
print("Detected version:", scm_output or "(none)")

if scm_error:
    print("stderr:")
    print(scm_error)

if p.returncode != 0:
    raise RuntimeError(
        "setuptools-scm still cannot detect the version even with Git on PATH."
    )


# ==============================================================
# 4. Verify locked dependency versions remain unchanged
# ==============================================================

locked_expected = {
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
    "pytest": "8.0.0",
    "pytest-benchmark": "4.0.0",
    "coverage": "7.4.0",
    "immutables": "0.20",
    "typing-extensions": "4.8.0",
}

print("\n4. LOCKED DEPENDENCY CHECK")
print("-" * 100)

locked_actual = {}

for pkg, expected in locked_expected.items():
    actual = version(pkg)
    locked_actual[pkg] = actual

    marker = "OK" if actual == expected else "MISMATCH"

    print(
        f"{pkg:20s}"
        f" expected={expected:10s}"
        f" actual={actual:10s}"
        f" {marker}"
    )

mismatches = {
    pkg: {
        "expected": expected,
        "actual": locked_actual[pkg],
    }
    for pkg, expected in locked_expected.items()
    if locked_actual[pkg] != expected
}

if mismatches:
    raise RuntimeError(
        f"Locked environment drift detected: {mismatches}"
    )


# ==============================================================
# 5. Install frozen source using visible Git
# ==============================================================

print("\n5. INSTALL FROZEN CAT​​TRS 24.1.0")
print("-" * 100)

p = run([
    PY,
    "-m",
    "pip",
    "install",
    "--no-deps",
    "--no-build-isolation",
    "."
], cwd=SRC)

install_output = p.stdout + "\n" + p.stderr

print(install_output[-7000:])

install_log = (
    RESULT_DIR
    / "cattrs_24.1_vcs_aware_install.log"
)

install_log.write_text(
    install_output,
    encoding="utf-8",
    errors="replace"
)

if p.returncode != 0:
    print("\nINSTALL FAILED.")
    print("Saved:", install_log)
    raise RuntimeError(
        "VCS-aware installation failed. "
        "Paste the final pip error block."
    )

print("\nInstallation PASS.")


# ==============================================================
# 6. Confirm installed release
# ==============================================================

installed_cattrs = version("cattrs")

print("\nInstalled cattrs:", installed_cattrs)

if installed_cattrs != "24.1.0":
    print(
        "WARNING: installed metadata version differs from 24.1.0."
    )


# ==============================================================
# 7. pip check
# ==============================================================

print("\n6. PIP CHECK")
print("-" * 100)

p = run([
    PY,
    "-m",
    "pip",
    "check"
])

pip_check = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)

print("pip check:", pip_check)

if p.stdout.strip():
    print(p.stdout.strip())

if p.stderr.strip():
    print(p.stderr.strip())


# ==============================================================
# 8. Functional suite
# ==============================================================

print("\n7. FUNCTIONAL SUITE")
print("(bench directory excluded)")
print("-" * 100)

start = time.perf_counter()

p = run([
    PY,
    "-m",
    "pytest",
    "-q",
    "--ignore=bench"
], cwd=SRC, timeout=3600)

elapsed = time.perf_counter() - start

test_output = p.stdout + "\n" + p.stderr

metrics = parse_summary(test_output)

status = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)

print(test_output[-8000:])


# ==============================================================
# 9. Final source integrity
# ==============================================================

p = run(
    ["git", "status", "--short"],
    cwd=SRC
)

final_status = p.stdout.strip()

source_integrity = (
    "CLEAN"
    if not final_status
    else "MODIFIED"
)


# ==============================================================
# 10. Save evidence
# ==============================================================

test_log = (
    RESULT_DIR
    / "cattrs_24.1_vcs_historical_baseline.log"
)

test_log.write_text(
    test_output,
    encoding="utf-8",
    errors="replace"
)

report = {
    "subject": "cattrs",
    "release": "24.1.0",
    "commit": commit,
    "tag": tag,
    "scm_detected_version": scm_output,
    "python": "3.12.14",
    "git_executable": str(GIT),
    "git_added_to_path": True,
    "locked_dependencies": locked_actual,
    "installed_cattrs": installed_cattrs,
    "pip_check": pip_check,
    "functional_suite": {
        "bench_excluded": True,
        "status": status,
        **metrics,
        "runtime_seconds": round(elapsed, 3),
    },
    "source_integrity": source_integrity,
}

json_file = (
    RESULT_DIR
    / "cattrs_24.1_vcs_historical_baseline.json"
)

json_file.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ==============================================================
# 11. Final summary
# ==============================================================

print("\n" + "=" * 100)
print("CATTRS 24.1.0 VCS-AWARE BASELINE SUMMARY")
print("=" * 100)

print("Commit          :", commit)
print("Tag             :", tag)
print("SCM version     :", scm_output)
print("Installed cattrs:", installed_cattrs)
print("Status          :", status)
print("Passed          :", metrics["passed"])
print("Failed          :", metrics["failed"])
print("Errors          :", metrics["errors"])
print("Skipped         :", metrics["skipped"])
print("XFailed         :", metrics["xfailed"])
print("XPassed         :", metrics["xpassed"])
print("Runtime         :", round(elapsed, 3), "s")
print("pip check       :", pip_check)
print("Source integrity:", source_integrity)

print("\nSaved:")
print(json_file)
print(test_log)
print(install_log)

In [ ]:
from pathlib import Path
import subprocess
import os
import shutil
import json
import re
import time

print("CATTRS 24.1.0 — WINDOWS GIT DISCOVERY + VCS BUILD RECOVERY")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

PY = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
    / "Scripts"
    / "python.exe"
)

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


# ==============================================================
# 1. Discover the REAL Git executable
# ==============================================================

print("\n1. DISCOVERING GIT")
print("-" * 100)

candidates = [
    Path(r"<LOCAL_USER_HOME>\conda_envs\emse-tier3\Library\mingw64\bin\git.exe"),
    Path(r"<LOCAL_USER_HOME>\conda_envs\emse-tier3\Library\bin\git.exe"),
    Path(r"C:\Program Files\Git\cmd\git.exe"),
    Path(r"C:\Program Files\Git\bin\git.exe"),
    Path(r"C:\Program Files (x86)\Git\cmd\git.exe"),
]

# Try Python's normal lookup too.
normal_git = shutil.which("git")
if normal_git:
    candidates.insert(0, Path(normal_git))

# Ask Windows directly.
try:
    w = subprocess.run(
        ["where.exe", "git"],
        capture_output=True,
        text=True
    )

    if w.returncode == 0:
        for line in w.stdout.splitlines():
            line = line.strip()
            if line:
                candidates.insert(0, Path(line))
except Exception as e:
    print("where.exe diagnostic:", repr(e))


# Remove duplicate candidates.
unique = []

for candidate in candidates:
    s = str(candidate).lower()

    if all(str(x).lower() != s for x in unique):
        unique.append(candidate)


GIT = None

for candidate in unique:
    print(f"{candidate}")
    print("   exists =", candidate.exists())

    if candidate.exists():
        try:
            p = subprocess.run(
                [str(candidate), "--version"],
                capture_output=True,
                text=True
            )

            print(
                "   execution =",
                p.stdout.strip() or p.stderr.strip()
            )

            if p.returncode == 0:
                GIT = candidate
                break

        except Exception as e:
            print("   execution error =", repr(e))


if GIT is None:
    raise RuntimeError(
        "\nNo executable Git installation could be found.\n"
        "Send me the output printed under DISCOVERING GIT."
    )


print("\nSELECTED GIT:")
print(GIT)


# ==============================================================
# 2. Build subprocess environment
# ==============================================================

BUILD_ENV = os.environ.copy()

git_dir = str(GIT.parent)

BUILD_ENV["PATH"] = (
    git_dir
    + os.pathsep
    + BUILD_ENV.get("PATH", "")
)


def run(cmd, cwd=None, timeout=3600):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env=BUILD_ENV
    )


# ==============================================================
# 3. Verify provenance using absolute Git
# ==============================================================

print("\n2. SOURCE PROVENANCE")
print("-" * 100)

p = run([GIT, "rev-parse", "HEAD"], cwd=SRC)
commit = p.stdout.strip()

print("Commit:", commit)

p = run(
    [GIT, "describe", "--tags", "--exact-match"],
    cwd=SRC
)
tag = p.stdout.strip()

print("Tag   :", tag)

p = run([GIT, "status", "--short"], cwd=SRC)
git_status = p.stdout.strip()

print(
    "Status:",
    "CLEAN" if not git_status else git_status
)

if git_status:
    raise RuntimeError(
        "Worktree is not clean. Stop before proceeding."
    )


# ==============================================================
# 4. Check whether Git is callable BY NAME
# ==============================================================

print("\n3. GIT CALLABLE BY NAME")
print("-" * 100)

try:
    p = run(["git.exe", "--version"], cwd=SRC)

    print("Return code:", p.returncode)
    print(p.stdout.strip() or p.stderr.strip())

    git_by_name = p.returncode == 0

except FileNotFoundError:
    git_by_name = False
    print("git.exe still not callable by name.")


# ==============================================================
# 5. If necessary, create a controlled Git shim directory
# ==============================================================

SHIM_DIR = RESULT_DIR / "_git_shim"
SHIM_DIR.mkdir(parents=True, exist_ok=True)

shim_git = SHIM_DIR / "git.exe"

if not git_by_name:

    print("\nCreating controlled git.exe shim...")

    # Copy the actual executable rather than editing system PATH.
    shutil.copy2(GIT, shim_git)

    BUILD_ENV["PATH"] = (
        str(SHIM_DIR)
        + os.pathsep
        + git_dir
        + os.pathsep
        + os.environ.get("PATH", "")
    )

    try:
        p = run(["git.exe", "--version"], cwd=SRC)

        print("Shim return code:", p.returncode)
        print(p.stdout.strip() or p.stderr.strip())

        git_by_name = p.returncode == 0

    except Exception as e:
        print("Shim execution error:", repr(e))
        git_by_name = False


if not git_by_name:
    raise RuntimeError(
        "Git was discovered but cannot be invoked by name "
        "inside the build subprocess. Send the output above."
    )


# ==============================================================
# 6. Verify setuptools-scm directly
# ==============================================================

print("\n4. SETUPTOOLS-SCM DIRECT CHECK")
print("-" * 100)

code = """
from setuptools_scm import get_version
print(get_version(root="."))
"""

p = run(
    [PY, "-c", code],
    cwd=SRC
)

print("Return code:", p.returncode)
print("stdout:", p.stdout.strip())

if p.stderr.strip():
    print("stderr:")
    print(p.stderr.strip())

if p.returncode != 0:
    raise RuntimeError(
        "Git now works, but setuptools-scm version discovery "
        "still fails. Send this SETUPTOOLS-SCM DIRECT CHECK."
    )

scm_version = p.stdout.strip()


# ==============================================================
# 7. Verify historical dependencies
# ==============================================================

print("\n5. HISTORICAL DEPENDENCY CHECK")
print("-" * 100)

expected = {
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
    "pytest": "8.0.0",
    "pytest-benchmark": "4.0.0",
    "coverage": "7.4.0",
    "immutables": "0.20",
    "typing-extensions": "4.8.0",
}


def package_version(pkg):
    code = f"""
import importlib.metadata as m
try:
    print(m.version({pkg!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    q = run([PY, "-c", code])
    return q.stdout.strip()


actual = {}

for pkg, wanted in expected.items():
    got = package_version(pkg)
    actual[pkg] = got

    print(
        f"{pkg:20s} "
        f"expected={wanted:10s} "
        f"actual={got:10s} "
        f"{'OK' if got == wanted else 'MISMATCH'}"
    )


mismatches = {
    k: (expected[k], actual[k])
    for k in expected
    if expected[k] != actual[k]
}

if mismatches:
    raise RuntimeError(
        f"Historical dependency drift detected: {mismatches}"
    )


# ==============================================================
# 8. Install cattrs from unchanged tagged source
# ==============================================================

print("\n6. INSTALL CAT​​TRS 24.1.0")
print("-" * 100)

p = run(
    [
        PY,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        ".",
    ],
    cwd=SRC
)

install_output = p.stdout + "\n" + p.stderr

print(install_output[-7000:])

install_log = (
    RESULT_DIR
    / "cattrs_24.1_windows_git_recovery_install.log"
)

install_log.write_text(
    install_output,
    encoding="utf-8",
    errors="replace"
)

if p.returncode != 0:
    raise RuntimeError(
        "Installation still failed. "
        "Send the final error block printed above."
    )


# ==============================================================
# 9. Verify installed package
# ==============================================================

installed_cattrs = package_version("cattrs")

print("\nInstalled cattrs:", installed_cattrs)


# ==============================================================
# 10. pip check
# ==============================================================

p = run(
    [PY, "-m", "pip", "check"]
)

pip_check = "PASS" if p.returncode == 0 else "FAIL"

print("pip check:", pip_check)

if p.stdout.strip():
    print(p.stdout.strip())


# ==============================================================
# 11. Run functional suite
# ==============================================================

print("\n7. FUNCTIONAL NATIVE SUITE")
print("-" * 100)

start = time.perf_counter()

p = run(
    [
        PY,
        "-m",
        "pytest",
        "-q",
        "--ignore=bench",
    ],
    cwd=SRC,
    timeout=3600
)

elapsed = time.perf_counter() - start

output = p.stdout + "\n" + p.stderr

print(output[-8000:])


def count(pattern):
    hits = re.findall(
        pattern,
        output,
        flags=re.I
    )

    return int(hits[-1]) if hits else 0


metrics = {
    "passed": count(r"(\d+)\s+passed"),
    "failed": count(r"(\d+)\s+failed"),
    "errors": count(r"(\d+)\s+errors?"),
    "skipped": count(r"(\d+)\s+skipped"),
    "xfailed": count(r"(\d+)\s+xfailed"),
    "xpassed": count(r"(\d+)\s+xpassed"),
}

status = "PASS" if p.returncode == 0 else "FAIL"


# ==============================================================
# 12. Final integrity
# ==============================================================

q = run(
    [GIT, "status", "--short"],
    cwd=SRC
)

final_git_status = q.stdout.strip()

source_integrity = (
    "CLEAN"
    if not final_git_status
    else "MODIFIED"
)


# ==============================================================
# 13. Save evidence
# ==============================================================

test_log = (
    RESULT_DIR
    / "cattrs_24.1_historical_final_baseline.log"
)

test_log.write_text(
    output,
    encoding="utf-8",
    errors="replace"
)

report = {
    "project": "cattrs",
    "release": "24.1.0",
    "commit": commit,
    "tag": tag,
    "git_executable": str(GIT),
    "scm_detected_version": scm_version,
    "installed_cattrs": installed_cattrs,
    "python": "3.12.14",
    "historical_dependencies": actual,
    "pip_check": pip_check,
    "bench_excluded": True,
    "status": status,
    "metrics": metrics,
    "runtime_seconds": round(elapsed, 3),
    "source_integrity": source_integrity,
}

json_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_final_baseline.json"
)

json_file.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ==============================================================
# 14. Final summary
# ==============================================================

print("\n" + "=" * 100)
print("CATTRS 24.1.0 FINAL BASELINE SUMMARY")
print("=" * 100)

print("Git executable  :", GIT)
print("Commit          :", commit)
print("Tag             :", tag)
print("SCM version     :", scm_version)
print("Installed cattrs:", installed_cattrs)

print("\nStatus          :", status)
print("Passed          :", metrics["passed"])
print("Failed          :", metrics["failed"])
print("Errors          :", metrics["errors"])
print("Skipped         :", metrics["skipped"])
print("XFailed         :", metrics["xfailed"])
print("XPassed         :", metrics["xpassed"])
print("Runtime         :", round(elapsed, 3), "s")
print("pip check       :", pip_check)
print("Source integrity:", source_integrity)

print("\nSaved:")
print(json_file)
print(test_log)
print(install_log)

In [ ]:
from pathlib import Path
import subprocess
import os
import json
import re
import time

print("CATTRS 24.1.0 — VERIFIED-VERSION BUILD RECOVERY")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

PY = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
    / "Scripts"
    / "python.exe"
)

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, timeout=3600, env=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env=env,
    )


def package_version(pkg, env=None):
    code = f"""
import importlib.metadata as m
try:
    print(m.version({pkg!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    p = run(
        [PY, "-c", code],
        env=env
    )
    return p.stdout.strip() or "UNKNOWN"


def parse_summary(output):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        hits = re.findall(
            pattern,
            output,
            flags=re.I
        )

        if hits:
            result[key] = int(hits[-1])

    return result


# ==============================================================
# 1. Preconditions
# ==============================================================

print("\n1. PRECONDITIONS")
print("-" * 100)

print("Source :", SRC)
print("Python :", PY)

if not SRC.exists():
    raise FileNotFoundError(SRC)

if not PY.exists():
    raise FileNotFoundError(PY)


# ==============================================================
# 2. Build environment
#
# Version is NOT guessed:
# it was independently verified from exact tag v24.1.0.
# ==============================================================

BUILD_ENV = os.environ.copy()

BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION"] = "24.1.0"
BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS"] = "24.1.0"

print("\n2. VERIFIED VERSION OVERRIDE")
print("-" * 100)

print(
    "SETUPTOOLS_SCM_PRETEND_VERSION =",
    BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION"]
)

print(
    "SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS =",
    BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS"]
)


# ==============================================================
# 3. Verify exact historical dependencies BEFORE installation
# ==============================================================

expected = {
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
    "pytest": "8.0.0",
    "pytest-benchmark": "4.0.0",
    "coverage": "7.4.0",
    "immutables": "0.20",
    "typing-extensions": "4.8.0",
}

print("\n3. HISTORICAL DEPENDENCY CHECK")
print("-" * 100)

actual = {}

for pkg, wanted in expected.items():

    got = package_version(
        pkg,
        env=BUILD_ENV
    )

    actual[pkg] = got

    marker = (
        "OK"
        if got == wanted
        else "MISMATCH"
    )

    print(
        f"{pkg:20s} "
        f"expected={wanted:10s} "
        f"actual={got:10s} "
        f"{marker}"
    )


mismatches = {
    pkg: {
        "expected": expected[pkg],
        "actual": actual[pkg],
    }
    for pkg in expected
    if actual[pkg] != expected[pkg]
}

if mismatches:
    raise RuntimeError(
        f"Historical dependency drift detected: {mismatches}"
    )


# ==============================================================
# 4. Check required build backend availability
# ==============================================================

print("\n4. BUILD BACKEND CHECK")
print("-" * 100)

for pkg in [
    "hatchling",
    "hatch-vcs",
    "setuptools-scm",
]:
    print(
        f"{pkg:20s}: "
        f"{package_version(pkg, env=BUILD_ENV)}"
    )


# ==============================================================
# 5. Install exact frozen cattrs source
#    - no dependency resolution
#    - no build isolation
#    - verified version supplied to setuptools-scm
# ==============================================================

print("\n5. INSTALLING FROZEN CAT​​TRS 24.1.0")
print("-" * 100)

p = run(
    [
        PY,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        ".",
    ],
    cwd=SRC,
    env=BUILD_ENV,
)

install_output = (
    p.stdout
    + "\n"
    + p.stderr
)

print(install_output[-7000:])


install_log = (
    RESULT_DIR
    / "cattrs_24.1_verified_version_install.log"
)

install_log.write_text(
    install_output,
    encoding="utf-8",
    errors="replace",
)


if p.returncode != 0:
    print("\nINSTALL STATUS: FAIL")
    print("Saved:", install_log)

    raise RuntimeError(
        "Installation still failed. "
        "Send only the final pip error block."
    )


print("\nINSTALL STATUS: PASS")


# ==============================================================
# 6. Confirm installed cattrs metadata
# ==============================================================

installed_cattrs = package_version(
    "cattrs",
    env=BUILD_ENV
)

print("\nInstalled cattrs:", installed_cattrs)

if installed_cattrs != "24.1.0":
    raise RuntimeError(
        f"Unexpected installed cattrs version: {installed_cattrs}"
    )


# ==============================================================
# 7. Verify locked dependencies again
# ==============================================================

print("\n6. POST-INSTALL DEPENDENCY CHECK")
print("-" * 100)

post_actual = {}

for pkg, wanted in expected.items():

    got = package_version(
        pkg,
        env=BUILD_ENV
    )

    post_actual[pkg] = got

    marker = (
        "OK"
        if got == wanted
        else "MISMATCH"
    )

    print(
        f"{pkg:20s} "
        f"expected={wanted:10s} "
        f"actual={got:10s} "
        f"{marker}"
    )


post_mismatches = {
    pkg: {
        "expected": expected[pkg],
        "actual": post_actual[pkg],
    }
    for pkg in expected
    if post_actual[pkg] != expected[pkg]
}

if post_mismatches:
    raise RuntimeError(
        "Installation changed historical dependencies: "
        f"{post_mismatches}"
    )


# ==============================================================
# 8. pip check
# ==============================================================

print("\n7. PIP CHECK")
print("-" * 100)

p = run(
    [
        PY,
        "-m",
        "pip",
        "check",
    ],
    env=BUILD_ENV,
)

pip_check = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)

print("pip check:", pip_check)

if p.stdout.strip():
    print(p.stdout.strip())

if p.stderr.strip():
    print(p.stderr.strip())


# ==============================================================
# 9. Run FUNCTIONAL suite
# ==============================================================

print("\n8. FUNCTIONAL NATIVE SUITE")
print("bench directory excluded")
print("-" * 100)

start = time.perf_counter()

p = run(
    [
        PY,
        "-m",
        "pytest",
        "-q",
        "--ignore=bench",
    ],
    cwd=SRC,
    timeout=3600,
    env=BUILD_ENV,
)

elapsed = (
    time.perf_counter()
    - start
)

test_output = (
    p.stdout
    + "\n"
    + p.stderr
)

print(test_output[-9000:])

metrics = parse_summary(
    test_output
)

status = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)


# ==============================================================
# 10. Save test log
# ==============================================================

test_log = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_final_test.log"
)

test_log.write_text(
    test_output,
    encoding="utf-8",
    errors="replace",
)


# ==============================================================
# 11. Evidence record
# ==============================================================

report = {
    "project": "cattrs",
    "release": "24.1.0",

    "validated_provenance": {
        "tag": "v24.1.0",
        "commit":
            "3cb670705b810926d12ecda5315d9e5b61a04e5c",
        "source_equivalence": "PASS",
        "worktree_previously_verified_clean": True,
    },

    "python": "3.12.14",

    "version_resolution": {
        "mechanism":
            "SETUPTOOLS_SCM_PRETEND_VERSION",
        "value":
            "24.1.0",
        "reason":
            "Windows build subprocess could not invoke Git; "
            "version value derived from independently verified "
            "exact upstream tag v24.1.0.",
    },

    "historical_dependencies":
        post_actual,

    "installed_cattrs":
        installed_cattrs,

    "pip_check":
        pip_check,

    "functional_suite": {
        "bench_excluded": True,
        "status": status,
        **metrics,
        "runtime_seconds":
            round(elapsed, 3),
    },
}


json_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_final.json"
)

json_file.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ==============================================================
# 12. FINAL SUMMARY
# ==============================================================

print("\n" + "=" * 100)
print("CATTRS 24.1.0 FINAL HISTORICAL BASELINE SUMMARY")
print("=" * 100)

print("Provenance tag   : v24.1.0")
print(
    "Commit           : "
    "3cb670705b810926d12ecda5315d9e5b61a04e5c"
)

print(
    "Version mechanism: "
    "SETUPTOOLS_SCM_PRETEND_VERSION=24.1.0"
)

print("Installed cattrs :", installed_cattrs)

print("\nStatus           :", status)
print("Passed           :", metrics["passed"])
print("Failed           :", metrics["failed"])
print("Errors           :", metrics["errors"])
print("Skipped          :", metrics["skipped"])
print("XFailed          :", metrics["xfailed"])
print("XPassed          :", metrics["xpassed"])
print(
    "Runtime          :",
    round(elapsed, 3),
    "s"
)

print("pip check        :", pip_check)

print("\nHistorical dependencies:")

for pkg, ver in post_actual.items():
    print(
        f"  {pkg:20s} {ver}"
    )

print("\nSaved:")
print(json_file)
print(test_log)
print(install_log)

In [ ]:
from pathlib import Path
import subprocess
import os
import json
import re
import time

print("CATTRS 24.1.0 — VERIFIED-VERSION BUILD RECOVERY")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

PY = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
    / "Scripts"
    / "python.exe"
)

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, timeout=3600, env=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env=env,
    )


def package_version(pkg, env=None):
    code = f"""
import importlib.metadata as m
try:
    print(m.version({pkg!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    p = run(
        [PY, "-c", code],
        env=env
    )
    return p.stdout.strip() or "UNKNOWN"


def parse_summary(output):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        hits = re.findall(
            pattern,
            output,
            flags=re.I
        )

        if hits:
            result[key] = int(hits[-1])

    return result


# ==============================================================
# 1. Preconditions
# ==============================================================

print("\n1. PRECONDITIONS")
print("-" * 100)

print("Source :", SRC)
print("Python :", PY)

if not SRC.exists():
    raise FileNotFoundError(SRC)

if not PY.exists():
    raise FileNotFoundError(PY)


# ==============================================================
# 2. Build environment
#
# Version is NOT guessed:
# it was independently verified from exact tag v24.1.0.
# ==============================================================

BUILD_ENV = os.environ.copy()

BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION"] = "24.1.0"
BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS"] = "24.1.0"

print("\n2. VERIFIED VERSION OVERRIDE")
print("-" * 100)

print(
    "SETUPTOOLS_SCM_PRETEND_VERSION =",
    BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION"]
)

print(
    "SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS =",
    BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS"]
)


# ==============================================================
# 3. Verify exact historical dependencies BEFORE installation
# ==============================================================

expected = {
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
    "pytest": "8.0.0",
    "pytest-benchmark": "4.0.0",
    "coverage": "7.4.0",
    "immutables": "0.20",
    "typing-extensions": "4.8.0",
}

print("\n3. HISTORICAL DEPENDENCY CHECK")
print("-" * 100)

actual = {}

for pkg, wanted in expected.items():

    got = package_version(
        pkg,
        env=BUILD_ENV
    )

    actual[pkg] = got

    marker = (
        "OK"
        if got == wanted
        else "MISMATCH"
    )

    print(
        f"{pkg:20s} "
        f"expected={wanted:10s} "
        f"actual={got:10s} "
        f"{marker}"
    )


mismatches = {
    pkg: {
        "expected": expected[pkg],
        "actual": actual[pkg],
    }
    for pkg in expected
    if actual[pkg] != expected[pkg]
}

if mismatches:
    raise RuntimeError(
        f"Historical dependency drift detected: {mismatches}"
    )


# ==============================================================
# 4. Check required build backend availability
# ==============================================================

print("\n4. BUILD BACKEND CHECK")
print("-" * 100)

for pkg in [
    "hatchling",
    "hatch-vcs",
    "setuptools-scm",
]:
    print(
        f"{pkg:20s}: "
        f"{package_version(pkg, env=BUILD_ENV)}"
    )


# ==============================================================
# 5. Install exact frozen cattrs source
#    - no dependency resolution
#    - no build isolation
#    - verified version supplied to setuptools-scm
# ==============================================================

print("\n5. INSTALLING FROZEN CAT​​TRS 24.1.0")
print("-" * 100)

p = run(
    [
        PY,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        ".",
    ],
    cwd=SRC,
    env=BUILD_ENV,
)

install_output = (
    p.stdout
    + "\n"
    + p.stderr
)

print(install_output[-7000:])


install_log = (
    RESULT_DIR
    / "cattrs_24.1_verified_version_install.log"
)

install_log.write_text(
    install_output,
    encoding="utf-8",
    errors="replace",
)


if p.returncode != 0:
    print("\nINSTALL STATUS: FAIL")
    print("Saved:", install_log)

    raise RuntimeError(
        "Installation still failed. "
        "Send only the final pip error block."
    )


print("\nINSTALL STATUS: PASS")


# ==============================================================
# 6. Confirm installed cattrs metadata
# ==============================================================

installed_cattrs = package_version(
    "cattrs",
    env=BUILD_ENV
)

print("\nInstalled cattrs:", installed_cattrs)

if installed_cattrs != "24.1.0":
    raise RuntimeError(
        f"Unexpected installed cattrs version: {installed_cattrs}"
    )


# ==============================================================
# 7. Verify locked dependencies again
# ==============================================================

print("\n6. POST-INSTALL DEPENDENCY CHECK")
print("-" * 100)

post_actual = {}

for pkg, wanted in expected.items():

    got = package_version(
        pkg,
        env=BUILD_ENV
    )

    post_actual[pkg] = got

    marker = (
        "OK"
        if got == wanted
        else "MISMATCH"
    )

    print(
        f"{pkg:20s} "
        f"expected={wanted:10s} "
        f"actual={got:10s} "
        f"{marker}"
    )


post_mismatches = {
    pkg: {
        "expected": expected[pkg],
        "actual": post_actual[pkg],
    }
    for pkg in expected
    if post_actual[pkg] != expected[pkg]
}

if post_mismatches:
    raise RuntimeError(
        "Installation changed historical dependencies: "
        f"{post_mismatches}"
    )


# ==============================================================
# 8. pip check
# ==============================================================

print("\n7. PIP CHECK")
print("-" * 100)

p = run(
    [
        PY,
        "-m",
        "pip",
        "check",
    ],
    env=BUILD_ENV,
)

pip_check = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)

print("pip check:", pip_check)

if p.stdout.strip():
    print(p.stdout.strip())

if p.stderr.strip():
    print(p.stderr.strip())


# ==============================================================
# 9. Run FUNCTIONAL suite
# ==============================================================

print("\n8. FUNCTIONAL NATIVE SUITE")
print("bench directory excluded")
print("-" * 100)

start = time.perf_counter()

p = run(
    [
        PY,
        "-m",
        "pytest",
        "-q",
        "--ignore=bench",
    ],
    cwd=SRC,
    timeout=3600,
    env=BUILD_ENV,
)

elapsed = (
    time.perf_counter()
    - start
)

test_output = (
    p.stdout
    + "\n"
    + p.stderr
)

print(test_output[-9000:])

metrics = parse_summary(
    test_output
)

status = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)


# ==============================================================
# 10. Save test log
# ==============================================================

test_log = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_final_test.log"
)

test_log.write_text(
    test_output,
    encoding="utf-8",
    errors="replace",
)


# ==============================================================
# 11. Evidence record
# ==============================================================

report = {
    "project": "cattrs",
    "release": "24.1.0",

    "validated_provenance": {
        "tag": "v24.1.0",
        "commit":
            "3cb670705b810926d12ecda5315d9e5b61a04e5c",
        "source_equivalence": "PASS",
        "worktree_previously_verified_clean": True,
    },

    "python": "3.12.14",

    "version_resolution": {
        "mechanism":
            "SETUPTOOLS_SCM_PRETEND_VERSION",
        "value":
            "24.1.0",
        "reason":
            "Windows build subprocess could not invoke Git; "
            "version value derived from independently verified "
            "exact upstream tag v24.1.0.",
    },

    "historical_dependencies":
        post_actual,

    "installed_cattrs":
        installed_cattrs,

    "pip_check":
        pip_check,

    "functional_suite": {
        "bench_excluded": True,
        "status": status,
        **metrics,
        "runtime_seconds":
            round(elapsed, 3),
    },
}


json_file = (
    RESULT_DIR
    / "cattrs_24.1_historical_locked_final.json"
)

json_file.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ==============================================================
# 12. FINAL SUMMARY
# ==============================================================

print("\n" + "=" * 100)
print("CATTRS 24.1.0 FINAL HISTORICAL BASELINE SUMMARY")
print("=" * 100)

print("Provenance tag   : v24.1.0")
print(
    "Commit           : "
    "3cb670705b810926d12ecda5315d9e5b61a04e5c"
)

print(
    "Version mechanism: "
    "SETUPTOOLS_SCM_PRETEND_VERSION=24.1.0"
)

print("Installed cattrs :", installed_cattrs)

print("\nStatus           :", status)
print("Passed           :", metrics["passed"])
print("Failed           :", metrics["failed"])
print("Errors           :", metrics["errors"])
print("Skipped          :", metrics["skipped"])
print("XFailed          :", metrics["xfailed"])
print("XPassed          :", metrics["xpassed"])
print(
    "Runtime          :",
    round(elapsed, 3),
    "s"
)

print("pip check        :", pip_check)

print("\nHistorical dependencies:")

for pkg, ver in post_actual.items():
    print(
        f"  {pkg:20s} {ver}"
    )

print("\nSaved:")
print(json_file)
print(test_log)
print(install_log)

In [ ]:
from pathlib import Path
import json

p = Path(
    r"<LOCAL_WORKSPACE>\baseline_execution\cattrs_24_1_resolution"
    r"\cattrs_24.1_historical_locked_final.json"
)

with open(p, "r", encoding="utf-8") as f:
    d = json.load(f)

print("CATTRS 24.1.0 FINAL RESULT")
print("=" * 60)

fs = d["functional_suite"]

print("Status :", fs.get("status"))
print("Passed :", fs.get("passed"))
print("Failed :", fs.get("failed"))
print("Errors :", fs.get("errors"))
print("Skipped:", fs.get("skipped"))
print("XFailed:", fs.get("xfailed"))
print("XPassed:", fs.get("xpassed"))
print("Runtime:", fs.get("runtime_seconds"), "s")
print("pip check:", d.get("pip_check"))

In [ ]:
from pathlib import Path
import re

log_file = Path(
    r"<LOCAL_WORKSPACE>\baseline_execution\cattrs_24_1_resolution"
    r"\cattrs_24.1_historical_locked_final_test.log"
)

text = log_file.read_text(
    encoding="utf-8",
    errors="replace"
)

print("CATTRS 24.1.0 — COLLECTION ERROR DIAGNOSTIC")
print("=" * 100)

# Print the tail first because pytest usually places collection errors there.
print("\nLAST 12000 CHARACTERS OF TEST LOG")
print("-" * 100)
print(text[-12000:])

print("\n" + "=" * 100)
print("COLLECTION ERROR SECTIONS")
print("=" * 100)

# Capture pytest's ERROR collecting ... sections.
pattern = re.compile(
    r"_{5,}\s+ERROR collecting .*?(?="
    r"\n_{5,}\s+ERROR collecting|"
    r"\n={5,}\s+short test summary info|"
    r"\n={5,}\s+warnings summary|"
    r"\Z)",
    flags=re.S | re.I,
)

matches = pattern.findall(text)

if matches:
    for i, block in enumerate(matches, 1):
        print(f"\nERROR BLOCK {i}")
        print("-" * 100)
        print(block[-7000:])
else:
    print(
        "\nNo structured 'ERROR collecting' block was parsed. "
        "Use the log tail printed above."
    )

In [ ]:
from pathlib import Path
import subprocess
import tomllib
import json
import re
import time
import os

print("CATTRS 24.1.0 — HISTORICAL UJSON RECOVERY + BASELINE RERUN")
print("=" * 100)

ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = (
    ROOT
    / "_git_exec"
    / "cattrs_equivalence"
    / "20260913_115702"
    / "24.1.0"
)

PY = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
    / "Scripts"
    / "python.exe"
)

LOCK = SRC / "pdm.lock"
PYPROJECT = SRC / "pyproject.toml"

RESULT_DIR = (
    ROOT
    / "baseline_execution"
    / "cattrs_24_1_resolution"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

BUILD_ENV = os.environ.copy()

# Preserve the already validated release-version workaround.
BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION"] = "24.1.0"
BUILD_ENV["SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CATTRS"] = "24.1.0"


def run(cmd, cwd=None, timeout=3600):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env=BUILD_ENV,
    )


def pkg_version(name):
    code = f"""
import importlib.metadata as m
try:
    print(m.version({name!r}))
except m.PackageNotFoundError:
    print("NOT_INSTALLED")
"""
    p = run([PY, "-c", code])
    return p.stdout.strip() or "UNKNOWN"


def parse_summary(text):
    result = {
        "passed": 0,
        "failed": 0,
        "errors": 0,
        "skipped": 0,
        "xfailed": 0,
        "xpassed": 0,
    }

    patterns = {
        "passed": r"(\d+)\s+passed",
        "failed": r"(\d+)\s+failed",
        "errors": r"(\d+)\s+errors?",
        "skipped": r"(\d+)\s+skipped",
        "xfailed": r"(\d+)\s+xfailed",
        "xpassed": r"(\d+)\s+xpassed",
    }

    for key, pattern in patterns.items():
        hits = re.findall(pattern, text, flags=re.I)
        if hits:
            result[key] = int(hits[-1])

    return result


# ------------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------------

print("\n1. PRECONDITIONS")
print("-" * 100)

for label, path in [
    ("Source", SRC),
    ("Python", PY),
    ("pdm.lock", LOCK),
    ("pyproject.toml", PYPROJECT),
]:
    print(f"{label:16s}: {path}")
    print(f"{'':16s}  exists={path.exists()}")

if not all(p.exists() for p in [SRC, PY, LOCK, PYPROJECT]):
    raise RuntimeError("One or more required files are missing.")


# ------------------------------------------------------------------
# 2. Inspect pyproject for ujson declaration
# ------------------------------------------------------------------

print("\n2. PYPROJECT UJSON DECLARATION")
print("-" * 100)

with open(PYPROJECT, "rb") as f:
    pyproject = tomllib.load(f)

optional = (
    pyproject
    .get("project", {})
    .get("optional-dependencies", {})
)

ujson_declarations = []

for group, deps in optional.items():
    for dep in deps:
        if re.match(r"(?i)^ujson(?:[\s<>=!~;\[].*)?$", dep.strip()):
            ujson_declarations.append((group, dep))

if ujson_declarations:
    for group, dep in ujson_declarations:
        print(f"Group {group!r}: {dep}")
else:
    print("No ujson declaration found in [project.optional-dependencies].")


# ------------------------------------------------------------------
# 3. Obtain exact ujson version from historical pdm.lock
# ------------------------------------------------------------------

print("\n3. HISTORICAL LOCK LOOKUP")
print("-" * 100)

with open(LOCK, "rb") as f:
    lock_data = tomllib.load(f)

packages = lock_data.get("package", [])

ujson_entries = []

for package in packages:
    name = str(package.get("name", "")).lower()

    if name == "ujson":
        ujson_entries.append(package)

if not ujson_entries:
    print("ujson was NOT found in pdm.lock.")
    print()
    print("STOPPING WITHOUT INSTALLING ANYTHING.")
    print(
        "Send me this output together with the PYPROJECT UJSON "
        "DECLARATION section."
    )
    raise RuntimeError(
        "No historical ujson version available in pdm.lock; "
        "do not install an arbitrary version."
    )

versions = sorted(
    {
        str(entry.get("version"))
        for entry in ujson_entries
        if entry.get("version") is not None
    }
)

print("ujson entries found :", len(ujson_entries))
print("Locked versions     :", versions)

if len(versions) != 1:
    raise RuntimeError(
        f"Expected one locked ujson version, found: {versions}"
    )

UJSON_VERSION = versions[0]

print("Selected version    :", UJSON_VERSION)


# ------------------------------------------------------------------
# 4. Show current ujson state
# ------------------------------------------------------------------

print("\n4. CURRENT UJSON STATE")
print("-" * 100)

before_ujson = pkg_version("ujson")
print("Current ujson:", before_ujson)


# ------------------------------------------------------------------
# 5. Verify core historical dependencies before installation
# ------------------------------------------------------------------

expected_core = {
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
    "pytest": "8.0.0",
    "pytest-benchmark": "4.0.0",
    "coverage": "7.4.0",
    "immutables": "0.20",
    "typing-extensions": "4.8.0",
}

print("\n5. CORE HISTORICAL DEPENDENCY CHECK")
print("-" * 100)

core_before = {}

for pkg, expected in expected_core.items():
    actual = pkg_version(pkg)
    core_before[pkg] = actual

    print(
        f"{pkg:20s} "
        f"expected={expected:10s} "
        f"actual={actual:10s} "
        f"{'OK' if actual == expected else 'MISMATCH'}"
    )

bad = {
    pkg: (expected_core[pkg], core_before[pkg])
    for pkg in expected_core
    if core_before[pkg] != expected_core[pkg]
}

if bad:
    raise RuntimeError(
        f"Historical core dependency drift detected: {bad}"
    )


# ------------------------------------------------------------------
# 6. Install ONLY exact historically locked ujson
# ------------------------------------------------------------------

print("\n6. INSTALLING HISTORICALLY LOCKED UJSON")
print("-" * 100)

if before_ujson == UJSON_VERSION:
    print(
        f"ujson {UJSON_VERSION} is already installed; "
        "no installation required."
    )
else:
    p = run([
        PY,
        "-m",
        "pip",
        "install",
        "--no-deps",
        f"ujson=={UJSON_VERSION}",
    ])

    install_text = p.stdout + "\n" + p.stderr
    print(install_text[-5000:])

    install_log = (
        RESULT_DIR
        / "cattrs_24.1_historical_ujson_install.log"
    )

    install_log.write_text(
        install_text,
        encoding="utf-8",
        errors="replace",
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Historical ujson installation failed. "
            "Send the pip error block."
        )

after_ujson = pkg_version("ujson")

print("\nInstalled ujson:", after_ujson)

if after_ujson != UJSON_VERSION:
    raise RuntimeError(
        f"Expected ujson {UJSON_VERSION}, got {after_ujson}"
    )


# ------------------------------------------------------------------
# 7. Verify core dependencies did not drift
# ------------------------------------------------------------------

print("\n7. POST-INSTALL CORE DEPENDENCY CHECK")
print("-" * 100)

core_after = {}

for pkg, expected in expected_core.items():
    actual = pkg_version(pkg)
    core_after[pkg] = actual

    print(
        f"{pkg:20s} "
        f"expected={expected:10s} "
        f"actual={actual:10s} "
        f"{'OK' if actual == expected else 'MISMATCH'}"
    )

bad_after = {
    pkg: (expected_core[pkg], core_after[pkg])
    for pkg in expected_core
    if core_after[pkg] != expected_core[pkg]
}

if bad_after:
    raise RuntimeError(
        f"Installing ujson changed locked dependencies: {bad_after}"
    )


# ------------------------------------------------------------------
# 8. Collection-only validation
# ------------------------------------------------------------------

print("\n8. COLLECTION-ONLY VALIDATION")
print("-" * 100)

start = time.perf_counter()

p = run(
    [
        PY,
        "-m",
        "pytest",
        "--collect-only",
        "-q",
        "--ignore=bench",
    ],
    cwd=SRC,
    timeout=1800,
)

collection_elapsed = time.perf_counter() - start
collection_output = p.stdout + "\n" + p.stderr

collection_log = (
    RESULT_DIR
    / "cattrs_24.1_historical_collection_after_ujson.log"
)

collection_log.write_text(
    collection_output,
    encoding="utf-8",
    errors="replace",
)

print(collection_output[-7000:])

collection_status = (
    "PASS"
    if p.returncode == 0
    else "FAIL"
)

print("\nCollection status:", collection_status)
print("Collection runtime:", round(collection_elapsed, 3), "s")

if p.returncode != 0:
    print()
    print("=" * 100)
    print("STOP: COLLECTION STILL FAILS")
    print("=" * 100)
    print(
        "Do not install anything else yet. "
        "Send me the final collection error block above."
    )

else:

    # --------------------------------------------------------------
    # 9. Full functional native suite
    # --------------------------------------------------------------

    print("\n9. FULL FUNCTIONAL NATIVE SUITE")
    print("bench directory excluded")
    print("-" * 100)

    start = time.perf_counter()

    p = run(
        [
            PY,
            "-m",
            "pytest",
            "-q",
            "--ignore=bench",
        ],
        cwd=SRC,
        timeout=3600,
    )

    elapsed = time.perf_counter() - start

    test_output = p.stdout + "\n" + p.stderr

    test_log = (
        RESULT_DIR
        / "cattrs_24.1_historical_after_ujson_test.log"
    )

    test_log.write_text(
        test_output,
        encoding="utf-8",
        errors="replace",
    )

    print(test_output[-10000:])

    metrics = parse_summary(test_output)

    status = (
        "PASS"
        if p.returncode == 0
        else "FAIL"
    )


    # --------------------------------------------------------------
    # 10. pip check
    # --------------------------------------------------------------

    pc = run(
        [
            PY,
            "-m",
            "pip",
            "check",
        ]
    )

    pip_check = (
        "PASS"
        if pc.returncode == 0
        else "FAIL"
    )


    # --------------------------------------------------------------
    # 11. Save final evidence
    # --------------------------------------------------------------

    report = {
        "project": "cattrs",
        "release": "24.1.0",

        "provenance": {
            "tag": "v24.1.0",
            "commit":
                "3cb670705b810926d12ecda5315d9e5b61a04e5c",
            "source_equivalence": "PASS",
        },

        "python": "3.12.14",

        "historical_core_dependencies":
            core_after,

        "historical_optional_dependency_added": {
            "name": "ujson",
            "version": UJSON_VERSION,
            "source": "release pdm.lock",
        },

        "collection": {
            "status": collection_status,
            "runtime_seconds":
                round(collection_elapsed, 3),
        },

        "functional_suite": {
            "bench_excluded": True,
            "status": status,
            **metrics,
            "runtime_seconds":
                round(elapsed, 3),
        },

        "pip_check": pip_check,
    }

    json_file = (
        RESULT_DIR
        / "cattrs_24.1_historical_after_ujson.json"
    )

    json_file.write_text(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # --------------------------------------------------------------
    # 12. Final summary
    # --------------------------------------------------------------

    print("\n" + "=" * 100)
    print("CATTRS 24.1.0 HISTORICAL BASELINE — AFTER LOCKED UJSON")
    print("=" * 100)

    print(
        "ujson             :",
        UJSON_VERSION,
        "(from pdm.lock)"
    )

    print("Collection        :", collection_status)
    print("Status            :", status)
    print("Passed            :", metrics["passed"])
    print("Failed            :", metrics["failed"])
    print("Errors            :", metrics["errors"])
    print("Skipped           :", metrics["skipped"])
    print("XFailed           :", metrics["xfailed"])
    print("XPassed           :", metrics["xpassed"])
    print("Runtime           :", round(elapsed, 3), "s")
    print("pip check         :", pip_check)

    print("\nSaved:")
    print(json_file)
    print(test_log)
    print(collection_log)

In [ ]:
# ============================================================
# CATTRS 24.1.0 — HISTORICAL OPTIONAL BACKEND RECOVERY
# Goal:
#   1. Read frozen pdm.lock / pyproject.toml
#   2. Determine historically locked optional backends
#   3. Compare against current historical environment
#   4. Install only missing historically locked backends
#   5. Run collection
#   6. Run full native functional suite excluding bench
#   7. Save reproducibility logs + JSON
# ============================================================

from pathlib import Path
import subprocess
import sys
import json
import re
import time
import importlib.metadata as md
from datetime import datetime

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
ROOT = Path(r"<LOCAL_WORKSPACE>")

SRC = ROOT / "_git_exec" / "cattrs_equivalence" / "20260913_115702" / "24.1.0"
VENV = ROOT / "_historical_venvs" / "cattrs-24.1.0-locked"
PYTHON = VENV / "Scripts" / "python.exe"

OUT = ROOT / "baseline_execution" / "cattrs_24_1_resolution"
OUT.mkdir(parents=True, exist_ok=True)

LOCK = SRC / "pdm.lock"
PYPROJECT = SRC / "pyproject.toml"

if not PYTHON.exists():
    raise FileNotFoundError(f"Historical Python not found: {PYTHON}")

if not LOCK.exists():
    raise FileNotFoundError(f"pdm.lock not found: {LOCK}")

print("=" * 100)
print("CATTRS 24.1.0 — HISTORICAL OPTIONAL BACKEND RECOVERY")
print("=" * 100)
print("Source :", SRC)
print("Python :", PYTHON)
print("Lock   :", LOCK)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def run(cmd, cwd=None, timeout=None):
    print("\n>", " ".join(map(str, cmd)))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        timeout=timeout
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p

def get_installed_version(dist_name):
    code = (
        "import importlib.metadata as m\n"
        f"try:\n print(m.version({dist_name!r}))\n"
        "except Exception:\n print('NOT_INSTALLED')\n"
    )
    p = subprocess.run(
        [str(PYTHON), "-c", code],
        text=True,
        capture_output=True
    )
    return p.stdout.strip()

# ------------------------------------------------------------
# 1. READ LOCK
# ------------------------------------------------------------
lock_text = LOCK.read_text(encoding="utf-8", errors="replace")

# We already know ujson came from pdm.lock. Now recover all likely
# optional serialization/test backends using exact lock versions only.
candidate_packages = [
    "ujson",
    "orjson",
    "msgpack",
    "PyYAML",
    "tomlkit",
    "cbor2",
    "pymongo",
    "msgspec",
]

def extract_lock_version(package_name, text):
    # PDM lock commonly stores blocks such as:
    # [[package]]
    # name = "orjson"
    # version = "3.x.x"
    blocks = re.split(r"\n\[\[package\]\]\n", "\n" + text)

    wanted = package_name.lower()

    aliases = {
        "pyyaml": {"pyyaml"},
        "pymongo": {"pymongo"},
        "msgpack": {"msgpack"},
        "orjson": {"orjson"},
        "ujson": {"ujson"},
        "tomlkit": {"tomlkit"},
        "cbor2": {"cbor2"},
        "msgspec": {"msgspec"},
    }

    accepted = aliases.get(wanted, {wanted})

    for block in blocks:
        m_name = re.search(r'(?m)^name\s*=\s*"([^"]+)"', block)
        m_ver = re.search(r'(?m)^version\s*=\s*"([^"]+)"', block)

        if m_name and m_ver:
            n = m_name.group(1).lower()
            if n in accepted:
                return m_ver.group(1)

    return None

locked_versions = {}
for pkg in candidate_packages:
    locked_versions[pkg] = extract_lock_version(pkg, lock_text)

print("\n" + "-" * 100)
print("LOCKED OPTIONAL BACKENDS FOUND")
print("-" * 100)

for pkg, ver in locked_versions.items():
    print(f"{pkg:12s}: {ver if ver else 'NOT FOUND IN pdm.lock'}")

# ------------------------------------------------------------
# 2. CURRENT ENVIRONMENT STATE
# ------------------------------------------------------------
print("\n" + "-" * 100)
print("CURRENT HISTORICAL ENVIRONMENT")
print("-" * 100)

installed_before = {}
for pkg in candidate_packages:
    dist_name = pkg
    installed_before[pkg] = get_installed_version(dist_name)
    print(f"{pkg:12s}: {installed_before[pkg]}")

# ------------------------------------------------------------
# 3. BUILD CONTROLLED INSTALL LIST
# ------------------------------------------------------------
to_install = []

for pkg, locked_ver in locked_versions.items():
    if not locked_ver:
        continue

    current = installed_before[pkg]

    if current == "NOT_INSTALLED":
        to_install.append(f"{pkg}=={locked_ver}")

    elif current != locked_ver:
        # Do NOT silently overwrite without recording it.
        print(
            f"\nWARNING: {pkg} installed version {current} differs "
            f"from historical lock {locked_ver}."
        )
        print(f"Will normalize to historical lock: {pkg}=={locked_ver}")
        to_install.append(f"{pkg}=={locked_ver}")

print("\n" + "-" * 100)
print("CONTROLLED INSTALL PLAN")
print("-" * 100)

if to_install:
    for x in to_install:
        print(x)
else:
    print("No missing or mismatched locked optional backends found.")

# ------------------------------------------------------------
# 4. INSTALL ONLY HISTORICALLY LOCKED VERSIONS
# ------------------------------------------------------------
install_records = []

for spec in to_install:
    print("\nInstalling historical backend:", spec)

    p = run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-deps",
        spec
    ])

    install_records.append({
        "spec": spec,
        "returncode": p.returncode,
        "stdout": p.stdout,
        "stderr": p.stderr,
    })

    if p.returncode != 0:
        raise RuntimeError(
            f"Historical backend installation failed: {spec}\n"
            "Stop here. Do not substitute a newer version."
        )

# ------------------------------------------------------------
# 5. VERIFY FINAL BACKEND VERSIONS
# ------------------------------------------------------------
print("\n" + "-" * 100)
print("FINAL OPTIONAL BACKEND VERSIONS")
print("-" * 100)

installed_after = {}

for pkg in candidate_packages:
    installed_after[pkg] = get_installed_version(pkg)
    print(f"{pkg:12s}: {installed_after[pkg]}")

# ------------------------------------------------------------
# 6. PIP CHECK
# ------------------------------------------------------------
print("\n" + "-" * 100)
print("PIP CHECK")
print("-" * 100)

pip_check = run([PYTHON, "-m", "pip", "check"])
pip_check_status = "PASS" if pip_check.returncode == 0 else "FAIL"

# ------------------------------------------------------------
# 7. COLLECTION-ONLY RUN
# ------------------------------------------------------------
print("\n" + "-" * 100)
print("COLLECTION CHECK")
print("bench directory excluded")
print("-" * 100)

collect_log = OUT / "cattrs_24.1_historical_collection_all_backends.log"

t0 = time.perf_counter()

collect = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pytest",
        "--collect-only",
        "-q",
        "--ignore=bench",
    ],
    cwd=str(SRC),
    text=True,
    capture_output=True,
)

collect_runtime = time.perf_counter() - t0

collect_text = (collect.stdout or "") + "\n" + (collect.stderr or "")
collect_log.write_text(collect_text, encoding="utf-8")

print(collect_text)

collection_status = "PASS" if collect.returncode == 0 else "FAIL"

print("Collection status :", collection_status)
print("Collection runtime:", round(collect_runtime, 3), "s")

if collect.returncode != 0:
    result = {
        "project": "cattrs",
        "version": "24.1.0",
        "phase": "historical_optional_backend_recovery",
        "collection_status": collection_status,
        "collection_runtime_seconds": collect_runtime,
        "pip_check": pip_check_status,
        "locked_versions": locked_versions,
        "installed_before": installed_before,
        "installed_after": installed_after,
        "install_records": install_records,
        "timestamp": datetime.now().isoformat(),
    }

    json_path = OUT / "cattrs_24.1_historical_all_backends_collection_failure.json"
    json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")

    raise RuntimeError(
        "Collection still fails after restoring all historically locked "
        "optional backends found in pdm.lock. Inspect the saved log before "
        "installing anything else."
    )

# ------------------------------------------------------------
# 8. FULL FUNCTIONAL SUITE
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("FULL FUNCTIONAL NATIVE SUITE")
print("bench directory excluded")
print("=" * 100)

test_log = OUT / "cattrs_24.1_historical_all_backends_test.log"

t0 = time.perf_counter()

test = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pytest",
        "-q",
        "--ignore=bench",
    ],
    cwd=str(SRC),
    text=True,
    capture_output=True,
)

runtime = time.perf_counter() - t0

test_text = (test.stdout or "") + "\n" + (test.stderr or "")
test_log.write_text(test_text, encoding="utf-8")

print(test_text)

# ------------------------------------------------------------
# 9. PARSE PYTEST SUMMARY
# ------------------------------------------------------------
summary_patterns = {
    "passed": r"(\d+)\s+passed",
    "failed": r"(\d+)\s+failed",
    "errors": r"(\d+)\s+errors?",
    "skipped": r"(\d+)\s+skipped",
    "xfailed": r"(\d+)\s+xfailed",
    "xpassed": r"(\d+)\s+xpassed",
}

counts = {}

for key, pattern in summary_patterns.items():
    m = re.search(pattern, test_text)
    counts[key] = int(m.group(1)) if m else 0

status = "PASS" if test.returncode == 0 else "FAIL"

# ------------------------------------------------------------
# 10. SAVE FINAL RESULT
# ------------------------------------------------------------
result = {
    "project": "cattrs",
    "version": "24.1.0",
    "python": "3.12.14",
    "source_tag": "v24.1.0",
    "source_commit": "3cb670705b810926d12ecda5315d9e5b61a04e5c",
    "source_equivalence": "PASS",
    "bench_excluded": True,
    "historical_environment": True,
    "locked_versions": locked_versions,
    "installed_before": installed_before,
    "installed_after": installed_after,
    "install_records": install_records,
    "collection_status": collection_status,
    "collection_runtime_seconds": collect_runtime,
    "status": status,
    "passed": counts["passed"],
    "failed": counts["failed"],
    "errors": counts["errors"],
    "skipped": counts["skipped"],
    "xfailed": counts["xfailed"],
    "xpassed": counts["xpassed"],
    "runtime_seconds": runtime,
    "pip_check": pip_check_status,
    "pytest_returncode": test.returncode,
    "timestamp": datetime.now().isoformat(),
}

json_path = OUT / "cattrs_24.1_historical_all_backends.json"
json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")

# ------------------------------------------------------------
# 11. FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("CATTRS 24.1.0 HISTORICAL BASELINE — ALL LOCKED OPTIONAL BACKENDS")
print("=" * 100)

print("Collection        :", collection_status)
print("Status            :", status)
print("Passed            :", counts["passed"])
print("Failed            :", counts["failed"])
print("Errors            :", counts["errors"])
print("Skipped           :", counts["skipped"])
print("XFailed           :", counts["xfailed"])
print("XPassed           :", counts["xpassed"])
print("Runtime           :", round(runtime, 3), "s")
print("pip check         :", pip_check_status)

print("\nLocked optional backends:")
for pkg, ver in locked_versions.items():
    if ver:
        print(f"  {pkg:12s} {ver}")

print("\nSaved:")
print(json_path)
print(test_log)
print(collect_log)

if status == "PASS" and counts["failed"] == 0 and counts["errors"] == 0:
    print("\nRESULT: CATTRS 24.1.0 HISTORICAL FUNCTIONAL BASELINE RESOLVED.")
    print("Next step: freeze the complete Tier-3 12-release baseline matrix.")
else:
    print("\nRESULT: One or more failures remain.")
    print("Do NOT install arbitrary packages or exclude tests.")
    print("Send the final summary and failed-test section for diagnosis.")

In [ ]:
# ============================================================
# EMSE TIER-3 — PER-TEST COVERAGE PILOT
# Subject: attrs 24.1.0
#
# PURPOSE
#   1. Locate the frozen attrs 24.1.0 source/environment
#   2. Verify pytest/coverage instrumentation
#   3. Verify native test collection
#   4. Execute tests with dynamic coverage contexts
#   5. Extract per-test implementation statement coverage
#   6. Extract per-test implementation branch/arc coverage
#   7. Save reproducible CSV/JSON/raw coverage artifacts
#
# IMPORTANT
#   - No package installation
#   - No source modification
#   - No test modification
#   - No reduction yet
# ============================================================

from pathlib import Path
import subprocess
import json
import csv
import shutil
import time
import re
import os
from datetime import datetime

ROOT = Path(r"<LOCAL_WORKSPACE>")

PROJECT = "attrs"
VERSION = "24.1.0"

OUT = (
    ROOT
    / "coverage_characterization"
    / PROJECT
    / VERSION
)

OUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. LOCATE FROZEN PYTHON ENVIRONMENT
# ============================================================

print("=" * 100)
print("EMSE TIER-3 PER-TEST COVERAGE PILOT")
print("SUBJECT: attrs 24.1.0")
print("=" * 100)

candidate_pythons = [
    ROOT / "_baseline_py312" / f"{PROJECT}-{VERSION}" / "Scripts" / "python.exe",
    ROOT / "_venvs" / f"{PROJECT}-{VERSION}" / "Scripts" / "python.exe",
]

PYTHON = None

for p in candidate_pythons:
    if p.exists():
        PYTHON = p
        break

if PYTHON is None:

    hits = list(
        ROOT.glob(
            f"**/{PROJECT}-{VERSION}/Scripts/python.exe"
        )
    )

    hits = [
        p for p in hits
        if "_historical_venvs" not in str(p)
        and "_git_venvs" not in str(p)
        and "coverage_characterization" not in str(p)
    ]

    if len(hits) == 1:
        PYTHON = hits[0]

    elif len(hits) > 1:

        print("\nMultiple Python environments found:")

        for p in hits:
            print(" ", p)

        raise RuntimeError(
            "Cannot uniquely identify the frozen attrs 24.1.0 environment."
        )

if PYTHON is None or not PYTHON.exists():

    raise FileNotFoundError(
        "Frozen Python environment for attrs 24.1.0 was not found."
    )

print("\nFrozen Python:")
print(PYTHON)


# ============================================================
# 2. LOCATE FROZEN SOURCE TREE
# ============================================================

def plausible_source_root(path):

    return (
        (path / "tests").exists()
        and (path / "src").exists()
        and (path / "pyproject.toml").exists()
    )


source_candidates = []

for pyproject in ROOT.glob("**/pyproject.toml"):

    s = str(pyproject).lower()

    blocked = [
        "_baseline_py312",
        "_historical_venvs",
        "_git_venvs",
        "site-packages",
        "coverage_characterization",
    ]

    if any(x in s for x in blocked):
        continue

    parent = pyproject.parent

    if VERSION not in str(parent):
        continue

    if not plausible_source_root(parent):
        continue

    text = pyproject.read_text(
        encoding="utf-8",
        errors="ignore"
    ).lower()

    if re.search(
        r'name\s*=\s*["\']attrs["\']',
        text
    ):
        source_candidates.append(parent)


source_candidates = sorted(
    set(source_candidates)
)


if len(source_candidates) == 0:

    raise RuntimeError(
        "Could not automatically locate the frozen attrs 24.1.0 source tree."
    )


if len(source_candidates) > 1:

    print("\nCandidate source trees:")

    for x in source_candidates:
        print(" ", x)

    preferred = [
        x for x in source_candidates
        if "attrs" in x.name.lower()
        or "24.1.0" in str(x)
    ]

    if len(preferred) == 1:
        SRC = preferred[0]

    else:
        raise RuntimeError(
            "More than one plausible source tree was found. "
            "Do not guess."
        )

else:

    SRC = source_candidates[0]


print("\nFrozen source:")
print(SRC)


# ============================================================
# 3. PREFLIGHT
# ============================================================

print("\n" + "=" * 100)
print("1. PREFLIGHT")
print("=" * 100)

preflight_code = r'''
import sys

print("Python:", sys.version)

for name in ["pytest", "coverage", "pytest_cov"]:

    try:
        module = __import__(name)

        version = getattr(
            module,
            "__version__",
            "unknown"
        )

        print(f"{name}: {version}")

    except Exception as e:

        print(
            f"{name}: MISSING ({e})"
        )

        raise
'''

preflight = subprocess.run(
    [
        str(PYTHON),
        "-c",
        preflight_code
    ],
    text=True,
    capture_output=True
)

print(preflight.stdout)

if preflight.stderr:
    print(preflight.stderr)

if preflight.returncode != 0:

    raise RuntimeError(
        "Coverage instrumentation is not available in the "
        "frozen environment. STOP. Do not install anything."
    )


# ============================================================
# 4. COLLECTION CHECK
# ============================================================

print("\n" + "=" * 100)
print("2. NATIVE TEST COLLECTION CHECK")
print("=" * 100)

collect = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pytest",
        "--collect-only",
        "-q",
    ],
    cwd=str(SRC),
    text=True,
    capture_output=True,
)

collect_text = (
    (collect.stdout or "")
    + "\n"
    + (collect.stderr or "")
)

collect_log = (
    OUT
    / "collection.log"
)

collect_log.write_text(
    collect_text,
    encoding="utf-8"
)

print(collect_text)

print(
    "\nCollection return code:",
    collect.returncode
)

if collect.returncode != 0:

    raise RuntimeError(
        "attrs 24.1.0 collection failed. "
        "Do not continue to coverage."
    )


# ============================================================
# 5. RUN TEST SUITE WITH DYNAMIC COVERAGE CONTEXTS
# ============================================================

print("\n" + "=" * 100)
print("3. PER-TEST STATEMENT + BRANCH COVERAGE EXECUTION")
print("=" * 100)

coverage_file = (
    OUT
    / ".coverage"
)

if coverage_file.exists():
    coverage_file.unlink()


env = dict(os.environ)

env["COVERAGE_FILE"] = str(
    coverage_file
)


cmd = [
    str(PYTHON),
    "-m",
    "pytest",
    "-q",

    # Implementation source only
    "--cov=src",

    # Branch/arc information
    "--cov-branch",

    # Associate execution with individual pytest tests
    "--cov-context=test",

    # Do not print conventional coverage report here
    "--cov-report=",
]


print("\nCommand:")
print(" ".join(cmd))


t0 = time.perf_counter()


run = subprocess.run(
    cmd,
    cwd=str(SRC),
    text=True,
    capture_output=True,
    env=env,
)


runtime = (
    time.perf_counter()
    - t0
)


run_text = (
    (run.stdout or "")
    + "\n"
    + (run.stderr or "")
)


run_log = (
    OUT
    / "coverage_run.log"
)


run_log.write_text(
    run_text,
    encoding="utf-8"
)


print(run_text)

print(
    "\nCoverage return code:",
    run.returncode
)

print(
    "Coverage runtime:",
    round(runtime, 3),
    "seconds"
)


if run.returncode != 0:

    raise RuntimeError(
        "Coverage execution failed. "
        "Inspect coverage_run.log before continuing."
    )


if not coverage_file.exists():

    raise RuntimeError(
        "Tests passed but the .coverage database was not created."
    )


# ============================================================
# 6. ARCHIVE RAW COVERAGE DATABASE
# ============================================================

print("\n" + "=" * 100)
print("4. ARCHIVING RAW COVERAGE DATABASE")
print("=" * 100)

coverage_archive = (
    OUT
    / "attrs_24.1.0_per_test.coverage"
)


shutil.copy2(
    coverage_file,
    coverage_archive
)


print(
    "Coverage database:",
    coverage_archive
)


# ============================================================
# 7. CREATE EXTRACTION SCRIPT
# ============================================================

print("\n" + "=" * 100)
print("5. EXTRACTING PER-TEST COVERAGE")
print("=" * 100)


extract_script = (
    OUT
    / "_extract_per_test_coverage.py"
)


extract_code = r'''
from pathlib import Path
import coverage
import csv
import json
import os

coverage_file = Path(
    os.environ["EMSE_COVERAGE_FILE"]
)

source_root = Path(
    os.environ["EMSE_SOURCE_ROOT"]
).resolve()

output_dir = Path(
    os.environ["EMSE_OUTPUT_DIR"]
).resolve()


# ------------------------------------------------------------
# READ COVERAGE DATABASE
# ------------------------------------------------------------

data = coverage.CoverageData(
    basename=str(coverage_file)
)

data.read()


all_contexts = sorted(
    data.measured_contexts()
)

all_files = sorted(
    data.measured_files()
)


# ------------------------------------------------------------
# PRODUCTION FILE FILTER
# ------------------------------------------------------------

src_dir = (
    source_root
    / "src"
).resolve()


prod_files = []


for filename in all_files:

    p = Path(
        filename
    ).resolve()

    try:

        p.relative_to(
            src_dir
        )

        prod_files.append(
            str(p)
        )

    except Exception:

        pass


print(
    "Measured contexts :",
    len(all_contexts)
)

print(
    "Measured files    :",
    len(all_files)
)

print(
    "Production files  :",
    len(prod_files)
)


# ------------------------------------------------------------
# MAP COVERAGE CONTEXT -> PYTEST TEST ID
# ------------------------------------------------------------

def context_to_test_id(ctx):

    if not ctx:
        return None

    for suffix in [
        "|run",
        "|setup",
        "|teardown"
    ]:

        if ctx.endswith(
            suffix
        ):

            return ctx[
                :-len(suffix)
            ]

    if "|" in ctx:

        left, right = ctx.rsplit(
            "|",
            1
        )

        if right in {
            "run",
            "setup",
            "teardown"
        }:

            return left

    return None


test_to_contexts = {}


for ctx in all_contexts:

    test_id = context_to_test_id(
        ctx
    )

    if test_id is None:
        continue

    test_to_contexts.setdefault(
        test_id,
        []
    ).append(
        ctx
    )


print(
    "Unique pytest tests :",
    len(test_to_contexts)
)


# ------------------------------------------------------------
# EXTRACT COVERAGE ELEMENTS PER TEST
# ------------------------------------------------------------

rows = []

detail = {}


for test_id in sorted(
    test_to_contexts
):

    contexts = sorted(
        test_to_contexts[
            test_id
        ]
    )

    data.set_query_contexts(
        contexts
    )

    line_elements = set()

    branch_elements = set()


    for filename in prod_files:

        rel = (
            Path(filename)
            .resolve()
            .relative_to(source_root)
            .as_posix()
        )


        # ----------------------------
        # STATEMENT / LINE ELEMENTS
        # ----------------------------

        lines = (
            data.lines(filename)
            or []
        )


        for line in lines:

            line_elements.add(
                f"{rel}:{line}"
            )


        # ----------------------------
        # BRANCH / ARC ELEMENTS
        # ----------------------------

        arcs = (
            data.arcs(filename)
            or []
        )


        for a, b in arcs:

            branch_elements.add(
                f"{rel}:{a}->{b}"
            )


    rows.append({

        "test_id":
            test_id,

        "contexts":
            len(contexts),

        "statement_elements":
            len(line_elements),

        "branch_arc_elements":
            len(branch_elements),
    })


    detail[test_id] = {

        "contexts":
            contexts,

        "statement_elements":
            sorted(
                line_elements
            ),

        "branch_arc_elements":
            sorted(
                branch_elements
            ),
    }


# ------------------------------------------------------------
# REMOVE CONTEXT FILTER
# ------------------------------------------------------------

data.set_query_contexts(
    None
)


# ------------------------------------------------------------
# FULL-SUITE OBSERVED UNION
# ------------------------------------------------------------

suite_lines = set()

suite_arcs = set()


for filename in prod_files:

    rel = (
        Path(filename)
        .resolve()
        .relative_to(source_root)
        .as_posix()
    )


    lines = (
        data.lines(filename)
        or []
    )


    for line in lines:

        suite_lines.add(
            f"{rel}:{line}"
        )


    arcs = (
        data.arcs(filename)
        or []
    )


    for a, b in arcs:

        suite_arcs.add(
            f"{rel}:{a}->{b}"
        )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary = {

    "unique_test_contexts":
        len(test_to_contexts),

    "production_files":
        len(prod_files),

    "suite_statement_elements_observed":
        len(suite_lines),

    "suite_branch_arc_elements_observed":
        len(suite_arcs),

    "measured_contexts_total":
        len(all_contexts),
}


# ------------------------------------------------------------
# SAVE PER-TEST CSV
# ------------------------------------------------------------

csv_path = (
    output_dir
    / "per_test_coverage.csv"
)


with csv_path.open(
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(

        f,

        fieldnames=[
            "test_id",
            "contexts",
            "statement_elements",
            "branch_arc_elements",
        ]
    )

    writer.writeheader()

    writer.writerows(
        rows
    )


# ------------------------------------------------------------
# SAVE EXACT COVERAGE ELEMENTS
# ------------------------------------------------------------

json_path = (
    output_dir
    / "per_test_coverage_elements.json"
)


json_path.write_text(

    json.dumps(
        detail,
        indent=2,
        ensure_ascii=False
    ),

    encoding="utf-8"
)


# ------------------------------------------------------------
# SAVE COVERAGE SUMMARY
# ------------------------------------------------------------

summary_path = (
    output_dir
    / "coverage_summary.json"
)


summary_path.write_text(

    json.dumps(
        summary,
        indent=2
    ),

    encoding="utf-8"
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print()

print("=" * 80)

print(
    "PER-TEST COVERAGE EXTRACTION COMPLETE"
)

print("=" * 80)

print(
    "Tests with contexts :",
    summary[
        "unique_test_contexts"
    ]
)

print(
    "Production files    :",
    summary[
        "production_files"
    ]
)

print(
    "Observed statements :",
    summary[
        "suite_statement_elements_observed"
    ]
)

print(
    "Observed branch arcs:",
    summary[
        "suite_branch_arc_elements_observed"
    ]
)

print()

print("Saved:")

print(csv_path)

print(json_path)

print(summary_path)
'''


extract_script.write_text(
    extract_code,
    encoding="utf-8"
)


# ============================================================
# 8. EXECUTE EXTRACTION INSIDE FROZEN ENVIRONMENT
# ============================================================

extract_env = dict(
    env
)

extract_env[
    "EMSE_COVERAGE_FILE"
] = str(
    coverage_archive
)

extract_env[
    "EMSE_SOURCE_ROOT"
] = str(
    SRC
)

extract_env[
    "EMSE_OUTPUT_DIR"
] = str(
    OUT
)


extract = subprocess.run(

    [
        str(PYTHON),
        str(extract_script),
    ],

    cwd=str(SRC),

    text=True,

    capture_output=True,

    env=extract_env,
)


print(
    extract.stdout
)


if extract.stderr:

    print(
        extract.stderr
    )


if extract.returncode != 0:

    raise RuntimeError(
        "Coverage execution passed, but "
        "per-test extraction failed."
    )


# ============================================================
# 9. LOAD SUMMARY
# ============================================================

summary_path = (
    OUT
    / "coverage_summary.json"
)


summary = json.loads(

    summary_path.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 10. VALIDATION
# ============================================================

print("\n" + "=" * 100)
print("6. COVERAGE VALIDATION")
print("=" * 100)


tests_with_contexts = (
    summary[
        "unique_test_contexts"
    ]
)


production_files = (
    summary[
        "production_files"
    ]
)


observed_statements = (
    summary[
        "suite_statement_elements_observed"
    ]
)


observed_arcs = (
    summary[
        "suite_branch_arc_elements_observed"
    ]
)


if tests_with_contexts <= 0:

    raise RuntimeError(
        "No pytest test contexts were extracted."
    )


if production_files <= 0:

    raise RuntimeError(
        "No implementation files were measured."
    )


if observed_statements <= 0:

    raise RuntimeError(
        "No implementation statement coverage was observed."
    )


if observed_arcs <= 0:

    raise RuntimeError(
        "No implementation branch/arc coverage was observed."
    )


# ============================================================
# 11. SAVE EXPERIMENT METADATA
# ============================================================

metadata = {

    "project":
        PROJECT,

    "version":
        VERSION,

    "phase":
        "per_test_coverage_pilot",

    "status":
        "PASS",

    "source_root":
        str(SRC),

    "python":
        str(PYTHON),

    "coverage_database":
        str(coverage_archive),

    "implementation_scope":
        "src/",

    "branch_coverage":
        True,

    "dynamic_context":
        "pytest test",

    "test_execution_returncode":
        run.returncode,

    "runtime_seconds":
        runtime,

    "unique_test_contexts":
        tests_with_contexts,

    "production_files":
        production_files,

    "observed_statement_elements":
        observed_statements,

    "observed_branch_arc_elements":
        observed_arcs,

    "timestamp":
        datetime.now().isoformat(),
}


metadata_path = (
    OUT
    / "coverage_characterization_metadata.json"
)


metadata_path.write_text(

    json.dumps(
        metadata,
        indent=2
    ),

    encoding="utf-8"
)


# ============================================================
# 12. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 100)

print(
    "ATTRS 24.1.0 PER-TEST COVERAGE PILOT"
)

print("=" * 100)

print(
    "Status                  : PASS"
)

print(
    "Tests with contexts     :",
    tests_with_contexts
)

print(
    "Production files        :",
    production_files
)

print(
    "Observed statements     :",
    observed_statements
)

print(
    "Observed branch arcs    :",
    observed_arcs
)

print(
    "Coverage runtime        :",
    round(runtime, 3),
    "s"
)


print("\nSaved:")

print(
    OUT
    / "per_test_coverage.csv"
)

print(
    OUT
    / "per_test_coverage_elements.json"
)

print(
    OUT
    / "coverage_summary.json"
)

print(
    coverage_archive
)

print(
    metadata_path
)


print("\n" + "=" * 100)

print(
    "PILOT COMPLETE — DO NOT RUN REDUCTION YET"
)

print(
    "Send the final ATTRS 24.1.0 PER-TEST COVERAGE PILOT block for validation."
)

print("=" * 100)

In [ ]:
# ============================================================
# ATTRS 24.1.0 — COVERAGE EXTRACTION RECOVERY
#
# IMPORTANT:
#   - DOES NOT rerun pytest
#   - DOES NOT modify source/tests/environment
#   - Reads the already-created coverage database
#   - Uses exact SQLite context IDs instead of regex context queries
# ============================================================

from pathlib import Path
import sqlite3
import json
import csv
import os
import re
import traceback
from collections import defaultdict
from datetime import datetime

ROOT = Path(r"<LOCAL_WORKSPACE>")

PROJECT = "attrs"
VERSION = "24.1.0"

OUT = (
    ROOT
    / "coverage_characterization"
    / PROJECT
    / VERSION
)

print("=" * 100)
print("ATTRS 24.1.0 — PER-TEST COVERAGE EXTRACTION RECOVERY")
print("=" * 100)

# ------------------------------------------------------------------
# 1. FIND EXISTING COVERAGE DATABASE
# ------------------------------------------------------------------

candidate_dbs = [
    OUT / "attrs_24.1.0_per_test.coverage",
    OUT / ".coverage",
]

COVERAGE_DB = None

for p in candidate_dbs:
    if p.exists() and p.stat().st_size > 0:
        COVERAGE_DB = p
        break

if COVERAGE_DB is None:
    raise FileNotFoundError(
        "Existing coverage database was not found.\n"
        "Expected one of:\n"
        f"  {candidate_dbs[0]}\n"
        f"  {candidate_dbs[1]}"
    )

print("\nCoverage database:")
print(COVERAGE_DB)

print("Size:", COVERAGE_DB.stat().st_size, "bytes")

# ------------------------------------------------------------------
# 2. FIND THE ORIGINAL ATTRS SOURCE ROOT
# ------------------------------------------------------------------

source_candidates = []

for pyproject in ROOT.glob("**/pyproject.toml"):

    s = str(pyproject).lower()

    blocked = [
        "_baseline_py312",
        "_historical_venvs",
        "_git_venvs",
        "site-packages",
        "coverage_characterization",
    ]

    if any(x in s for x in blocked):
        continue

    parent = pyproject.parent

    if VERSION not in str(parent):
        continue

    if not (parent / "src").exists():
        continue

    if not (parent / "tests").exists():
        continue

    try:
        text = pyproject.read_text(
            encoding="utf-8",
            errors="ignore"
        ).lower()
    except Exception:
        continue

    if re.search(r'name\s*=\s*["\']attrs["\']', text):
        source_candidates.append(parent.resolve())

source_candidates = sorted(set(source_candidates))

if len(source_candidates) == 0:
    raise RuntimeError(
        "Could not locate attrs 24.1.0 source root."
    )

if len(source_candidates) > 1:
    print("\nSource candidates:")
    for x in source_candidates:
        print(" ", x)

    preferred = [
        x for x in source_candidates
        if VERSION in str(x)
    ]

    if len(preferred) == 1:
        SRC = preferred[0]
    else:
        raise RuntimeError(
            "Multiple attrs 24.1.0 source trees found. "
            "Cannot safely choose one."
        )
else:
    SRC = source_candidates[0]

SRC_DIR = (SRC / "src").resolve()

print("\nSource root:")
print(SRC)

print("\nProduction source directory:")
print(SRC_DIR)

# ------------------------------------------------------------------
# 3. OPEN COVERAGE SQLITE DATABASE
# ------------------------------------------------------------------

conn = sqlite3.connect(str(COVERAGE_DB))
conn.row_factory = sqlite3.Row
cur = conn.cursor()

tables = {
    row["name"]
    for row in cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )
}

print("\nCoverage database tables:")
print(sorted(tables))

required = {"file", "context"}

if not required.issubset(tables):
    raise RuntimeError(
        f"Coverage database schema is unexpected. "
        f"Missing: {required - tables}"
    )

# ------------------------------------------------------------------
# 4. READ FILE TABLE AND RESTRICT TO src/
# ------------------------------------------------------------------

file_rows = cur.execute(
    "SELECT id, path FROM file ORDER BY id"
).fetchall()

prod_file_ids = {}
prod_file_rel = {}

for row in file_rows:

    file_id = int(row["id"])
    raw_path = row["path"]

    p = Path(raw_path)

    if not p.is_absolute():
        p = (SRC / p).resolve()
    else:
        p = p.resolve()

    try:
        rel = p.relative_to(SRC_DIR)
    except Exception:
        continue

    # Store relative to project root as src/...
    project_rel = "src/" + rel.as_posix()

    prod_file_ids[file_id] = p
    prod_file_rel[file_id] = project_rel

print("\nProduction files recorded in coverage DB:",
      len(prod_file_ids))

if len(prod_file_ids) == 0:
    raise RuntimeError(
        "Coverage database contains no files under attrs src/."
    )

# ------------------------------------------------------------------
# 5. READ EXACT COVERAGE CONTEXTS
# ------------------------------------------------------------------

context_rows = cur.execute(
    "SELECT id, context FROM context ORDER BY id"
).fetchall()

context_id_to_raw = {
    int(r["id"]): r["context"]
    for r in context_rows
}

print("Measured raw contexts:", len(context_id_to_raw))

# pytest-cov format normally:
# tests/test_x.py::test_name|run
# tests/test_x.py::test_name|setup
# tests/test_x.py::test_name|teardown

def parse_test_context(ctx):

    if not ctx:
        return None

    for suffix in ("|run", "|setup", "|teardown"):

        if ctx.endswith(suffix):
            return ctx[:-len(suffix)]

    if "|" in ctx:
        left, right = ctx.rsplit("|", 1)

        if right in {"run", "setup", "teardown"}:
            return left

    return None


context_id_to_test = {}

test_to_context_ids = defaultdict(set)

for context_id, raw in context_id_to_raw.items():

    test_id = parse_test_context(raw)

    if test_id is None:
        continue

    context_id_to_test[context_id] = test_id
    test_to_context_ids[test_id].add(context_id)

print("Unique pytest test IDs:", len(test_to_context_ids))

if len(test_to_context_ids) == 0:
    print("\nFirst 20 contexts for diagnosis:")
    for x in list(context_id_to_raw.values())[:20]:
        print(repr(x))

    raise RuntimeError(
        "No pytest test contexts could be recognized."
    )

# ------------------------------------------------------------------
# 6. PREPARE PER-TEST ELEMENT SETS
# ------------------------------------------------------------------

test_lines = defaultdict(set)
test_arcs = defaultdict(set)

suite_lines = set()
suite_arcs = set()

# ------------------------------------------------------------------
# 7. EXTRACT LINE COVERAGE
#
# coverage.py stores line numbers in compressed line_bits.
# Use coverage.numbits only for decoding that official format.
# ------------------------------------------------------------------

if "line_bits" in tables:

    try:
        from coverage.numbits import numbits_to_nums
    except Exception as e:
        raise RuntimeError(
            "Could not import coverage.numbits.numbits_to_nums"
        ) from e

    line_rows = cur.execute(
        """
        SELECT file_id, context_id, numbits
        FROM line_bits
        """
    ).fetchall()

    for row in line_rows:

        file_id = int(row["file_id"])
        context_id = int(row["context_id"])

        if file_id not in prod_file_ids:
            continue

        rel = prod_file_rel[file_id]

        nums = numbits_to_nums(row["numbits"])

        for lineno in nums:

            element = f"{rel}:{lineno}"

            suite_lines.add(element)

            test_id = context_id_to_test.get(context_id)

            if test_id is not None:
                test_lines[test_id].add(element)

# ------------------------------------------------------------------
# 8. EXTRACT ARC COVERAGE
#
# Branch-mode coverage stores executed control-flow arcs here.
# ------------------------------------------------------------------

if "arc" in tables:

    arc_rows = cur.execute(
        """
        SELECT file_id, context_id, fromno, tono
        FROM arc
        """
    ).fetchall()

    for row in arc_rows:

        file_id = int(row["file_id"])
        context_id = int(row["context_id"])

        if file_id not in prod_file_ids:
            continue

        rel = prod_file_rel[file_id]

        fromno = int(row["fromno"])
        tono = int(row["tono"])

        element = f"{rel}:{fromno}->{tono}"

        suite_arcs.add(element)

        test_id = context_id_to_test.get(context_id)

        if test_id is not None:
            test_arcs[test_id].add(element)

# ------------------------------------------------------------------
# 9. IF BRANCH MODE STORES ONLY ARCS, DERIVE EXECUTED LINES
#
# Some coverage.py branch-mode databases primarily populate arc data.
# Positive arc endpoints correspond to executable source line numbers.
# ------------------------------------------------------------------

if len(suite_lines) == 0 and len(suite_arcs) > 0:

    print(
        "\nNOTE: line_bits contains no production line records; "
        "deriving executed statement-line elements from positive arc endpoints."
    )

    # Suite union
    for arc_element in suite_arcs:

        file_part, arc_part = arc_element.rsplit(":", 1)
        a, b = arc_part.split("->")

        for n in (int(a), int(b)):
            if n > 0:
                suite_lines.add(
                    f"{file_part}:{n}"
                )

    # Individual tests
    for test_id in test_to_context_ids:

        for arc_element in test_arcs.get(test_id, set()):

            file_part, arc_part = arc_element.rsplit(":", 1)
            a, b = arc_part.split("->")

            for n in (int(a), int(b)):
                if n > 0:
                    test_lines[test_id].add(
                        f"{file_part}:{n}"
                    )

# ------------------------------------------------------------------
# 10. BUILD PER-TEST TABLE
# ------------------------------------------------------------------

rows = []
detail = {}

all_test_ids = sorted(test_to_context_ids)

for test_id in all_test_ids:

    contexts = sorted(
        context_id_to_raw[cid]
        for cid in test_to_context_ids[test_id]
    )

    lines = sorted(
        test_lines.get(test_id, set())
    )

    arcs = sorted(
        test_arcs.get(test_id, set())
    )

    rows.append({
        "test_id": test_id,
        "contexts": len(contexts),
        "statement_elements": len(lines),
        "branch_arc_elements": len(arcs),
    })

    detail[test_id] = {
        "contexts": contexts,
        "statement_elements": lines,
        "branch_arc_elements": arcs,
    }

# ------------------------------------------------------------------
# 11. VALIDATE NONZERO TEST COVERAGE
# ------------------------------------------------------------------

tests_with_statement_coverage = sum(
    1 for r in rows
    if r["statement_elements"] > 0
)

tests_with_arc_coverage = sum(
    1 for r in rows
    if r["branch_arc_elements"] > 0
)

tests_with_no_prod_coverage = [
    r["test_id"]
    for r in rows
    if r["statement_elements"] == 0
    and r["branch_arc_elements"] == 0
]

# ------------------------------------------------------------------
# 12. SAVE OUTPUTS
# ------------------------------------------------------------------

csv_path = OUT / "per_test_coverage.csv"

with csv_path.open(
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "test_id",
            "contexts",
            "statement_elements",
            "branch_arc_elements",
        ]
    )

    writer.writeheader()
    writer.writerows(rows)


json_path = OUT / "per_test_coverage_elements.json"

json_path.write_text(
    json.dumps(
        detail,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


summary = {
    "project": PROJECT,
    "version": VERSION,
    "status": "PASS",
    "coverage_database": str(COVERAGE_DB),
    "source_root": str(SRC),
    "production_files": len(prod_file_ids),
    "raw_contexts": len(context_id_to_raw),
    "unique_test_contexts": len(all_test_ids),
    "tests_with_statement_coverage": tests_with_statement_coverage,
    "tests_with_branch_arc_coverage": tests_with_arc_coverage,
    "tests_with_no_production_coverage":
        len(tests_with_no_prod_coverage),
    "suite_statement_elements_observed":
        len(suite_lines),
    "suite_branch_arc_elements_observed":
        len(suite_arcs),
    "timestamp": datetime.now().isoformat(),
}

summary_path = OUT / "coverage_summary.json"

summary_path.write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)


no_cov_path = OUT / "tests_with_no_production_coverage.json"

no_cov_path.write_text(
    json.dumps(
        tests_with_no_prod_coverage,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------------
# 13. FINAL SANITY CHECKS
# ------------------------------------------------------------------

if summary["unique_test_contexts"] <= 0:
    raise RuntimeError(
        "No test contexts extracted."
    )

if summary["production_files"] <= 0:
    raise RuntimeError(
        "No production files extracted."
    )

if summary["suite_statement_elements_observed"] <= 0:
    raise RuntimeError(
        "No production statement elements extracted."
    )

if summary["suite_branch_arc_elements_observed"] <= 0:
    raise RuntimeError(
        "No production arc elements extracted."
    )

# ------------------------------------------------------------------
# 14. DISPLAY RESULT
# ------------------------------------------------------------------

print("\n" + "=" * 100)
print("ATTRS 24.1.0 PER-TEST COVERAGE PILOT — RECOVERED")
print("=" * 100)

print(
    "Status                       :",
    summary["status"]
)

print(
    "Tests with contexts          :",
    summary["unique_test_contexts"]
)

print(
    "Production files             :",
    summary["production_files"]
)

print(
    "Tests with statement coverage:",
    summary["tests_with_statement_coverage"]
)

print(
    "Tests with branch coverage   :",
    summary["tests_with_branch_arc_coverage"]
)

print(
    "Tests with no prod coverage  :",
    summary["tests_with_no_production_coverage"]
)

print(
    "Observed statements          :",
    summary["suite_statement_elements_observed"]
)

print(
    "Observed branch arcs         :",
    summary["suite_branch_arc_elements_observed"]
)

print("\nSaved:")
print(csv_path)
print(json_path)
print(summary_path)
print(no_cov_path)

print("\n" + "=" * 100)
print(
    "NO PYTEST RERUN WAS PERFORMED — EXISTING COVERAGE DATA ONLY"
)
print("=" * 100)

conn.close()

In [ ]:
# ============================================================
# ATTRS 24.1.0 — COVERAGE UNIVERSE CONSISTENCY CHECK
#
# Purpose:
#   Compare:
#       all collected native pytest tests
#       vs.
#       tests represented in coverage contexts
#
# This DOES NOT rerun the functional test suite.
# Only pytest --collect-only is executed.
# ============================================================

from pathlib import Path
import subprocess
import json
import re
from datetime import datetime

ROOT = Path(r"<LOCAL_WORKSPACE>")

PROJECT = "attrs"
VERSION = "24.1.0"

OUT = (
    ROOT
    / "coverage_characterization"
    / PROJECT
    / VERSION
)

# ------------------------------------------------------------
# 1. LOCATE PYTHON
# ------------------------------------------------------------

candidate_pythons = [
    ROOT / "_baseline_py312" / f"{PROJECT}-{VERSION}" / "Scripts" / "python.exe",
    ROOT / "_venvs" / f"{PROJECT}-{VERSION}" / "Scripts" / "python.exe",
]

PYTHON = None

for p in candidate_pythons:
    if p.exists():
        PYTHON = p
        break

if PYTHON is None:
    hits = list(
        ROOT.glob(
            f"**/{PROJECT}-{VERSION}/Scripts/python.exe"
        )
    )

    hits = [
        p for p in hits
        if "_historical_venvs" not in str(p)
        and "_git_venvs" not in str(p)
        and "coverage_characterization" not in str(p)
    ]

    if len(hits) == 1:
        PYTHON = hits[0]

if PYTHON is None:
    raise RuntimeError(
        "Could not identify attrs 24.1.0 Python environment."
    )

# ------------------------------------------------------------
# 2. LOCATE SOURCE
# ------------------------------------------------------------

source_candidates = []

for pyproject in ROOT.glob("**/pyproject.toml"):

    s = str(pyproject).lower()

    if any(
        x in s
        for x in [
            "_baseline_py312",
            "_historical_venvs",
            "_git_venvs",
            "site-packages",
            "coverage_characterization",
        ]
    ):
        continue

    parent = pyproject.parent

    if VERSION not in str(parent):
        continue

    if not (parent / "tests").exists():
        continue

    if not (parent / "src").exists():
        continue

    txt = pyproject.read_text(
        encoding="utf-8",
        errors="ignore"
    ).lower()

    if re.search(
        r'name\s*=\s*["\']attrs["\']',
        txt
    ):
        source_candidates.append(
            parent.resolve()
        )

source_candidates = sorted(
    set(source_candidates)
)

if len(source_candidates) != 1:
    print("Source candidates:")
    for x in source_candidates:
        print(" ", x)

    raise RuntimeError(
        "Could not uniquely identify attrs 24.1.0 source."
    )

SRC = source_candidates[0]

print("=" * 100)
print("ATTRS 24.1.0 — COVERAGE UNIVERSE CONSISTENCY CHECK")
print("=" * 100)

print("Python:", PYTHON)
print("Source:", SRC)

# ------------------------------------------------------------
# 3. QUICK COLLECTION ONLY
# ------------------------------------------------------------

print("\nCollecting pytest node IDs only...")

collect = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pytest",
        "--collect-only",
        "-q",
    ],
    cwd=str(SRC),
    text=True,
    capture_output=True,
)

collect_text = (
    (collect.stdout or "")
    + "\n"
    + (collect.stderr or "")
)

if collect.returncode != 0:
    print(collect_text)
    raise RuntimeError(
        "Collection failed."
    )

# ------------------------------------------------------------
# 4. EXTRACT NODE IDS
# ------------------------------------------------------------

collected_ids = set()

for raw in collect.stdout.splitlines():

    line = raw.strip()

    # pytest -q collection normally emits node IDs containing ::
    if "::" not in line:
        continue

    # Remove accidental surrounding whitespace only.
    collected_ids.add(line)

print("Collected pytest node IDs:", len(collected_ids))

# ------------------------------------------------------------
# 5. LOAD ALREADY-EXTRACTED COVERAGE TEST IDS
# ------------------------------------------------------------

coverage_json = (
    OUT
    / "per_test_coverage_elements.json"
)

if not coverage_json.exists():
    raise FileNotFoundError(
        coverage_json
    )

coverage_data = json.loads(
    coverage_json.read_text(
        encoding="utf-8"
    )
)

coverage_ids = set(
    coverage_data.keys()
)

print("Coverage-context test IDs:", len(coverage_ids))

# ------------------------------------------------------------
# 6. COMPARE
# ------------------------------------------------------------

missing_from_coverage = sorted(
    collected_ids
    - coverage_ids
)

unexpected_coverage_ids = sorted(
    coverage_ids
    - collected_ids
)

represented = sorted(
    collected_ids
    & coverage_ids
)

print("\nRepresented in coverage :", len(represented))
print("No coverage context     :", len(missing_from_coverage))
print("Unexpected context IDs  :", len(unexpected_coverage_ids))

# ------------------------------------------------------------
# 7. SAVE MISSING TESTS
# ------------------------------------------------------------

missing_path = (
    OUT
    / "collected_tests_without_coverage_context.json"
)

missing_path.write_text(
    json.dumps(
        missing_from_coverage,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

unexpected_path = (
    OUT
    / "coverage_contexts_not_in_collection.json"
)

unexpected_path.write_text(
    json.dumps(
        unexpected_coverage_ids,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. READ COVERAGE TEST RUN SUMMARY
# ------------------------------------------------------------

run_log = (
    OUT
    / "coverage_run.log"
)

run_summary_lines = []

if run_log.exists():

    txt = run_log.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    for line in txt.splitlines():

        low = line.lower()

        if (
            " passed" in low
            or " failed" in low
            or " skipped" in low
            or " xfailed" in low
            or " xpassed" in low
        ):
            run_summary_lines.append(
                line.strip()
            )

# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

result = {
    "project": PROJECT,
    "version": VERSION,
    "collected_tests": len(collected_ids),
    "coverage_context_tests": len(coverage_ids),
    "represented_tests": len(represented),
    "tests_without_coverage_context": len(missing_from_coverage),
    "unexpected_coverage_contexts": len(unexpected_coverage_ids),
    "coverage_run_summary_lines": run_summary_lines,
    "timestamp": datetime.now().isoformat(),
}

result_path = (
    OUT
    / "coverage_universe_consistency.json"
)

result_path.write_text(
    json.dumps(
        result,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 100)
print("ATTRS 24.1.0 — COVERAGE UNIVERSE CONSISTENCY")
print("=" * 100)

print(
    "Collected tests              :",
    result["collected_tests"]
)

print(
    "Tests with coverage contexts :",
    result["coverage_context_tests"]
)

print(
    "Tests represented            :",
    result["represented_tests"]
)

print(
    "Tests without context        :",
    result["tests_without_coverage_context"]
)

print(
    "Unexpected context IDs       :",
    result["unexpected_coverage_contexts"]
)

if run_summary_lines:
    print("\nOriginal coverage-run summary:")
    for x in run_summary_lines[-5:]:
        print(" ", x)

print("\nFirst 20 tests without coverage context:")

for x in missing_from_coverage[:20]:
    print(" ", x)

print("\nSaved:")
print(result_path)
print(missing_path)
print(unexpected_path)

print("\n" + "=" * 100)

if len(unexpected_coverage_ids) == 0:
    print("CONTEXT MAPPING: CONSISTENT")
else:
    print("CONTEXT MAPPING: REQUIRES INVESTIGATION")

print("=" * 100)

In [ ]:
from pathlib import Path
import json

p = Path(
    r"<LOCAL_WORKSPACE>\coverage_characterization\attrs\24.1.0"
    r"\coverage_universe_consistency.json"
)

d = json.loads(p.read_text(encoding="utf-8"))

print("=" * 90)
print("ATTRS 24.1.0 — COVERAGE UNIVERSE CONSISTENCY")
print("=" * 90)

print("Collected tests              :", d["collected_tests"])
print("Tests with coverage contexts :", d["coverage_context_tests"])
print("Tests represented            :", d["represented_tests"])
print("Tests without context        :", d["tests_without_coverage_context"])
print("Unexpected context IDs       :", d["unexpected_coverage_contexts"])

print("\nCoverage-run summary:")
for line in d.get("coverage_run_summary_lines", []):
    print(" ", line)

In [ ]:
from pathlib import Path
import re

p = Path(
    r"<LOCAL_WORKSPACE>\coverage_characterization\attrs\24.1.0"
    r"\collection.log"
)

text = p.read_text(
    encoding="utf-8",
    errors="ignore"
)

lines = text.splitlines()

print("=" * 90)
print("ATTRS 24.1.0 — COLLECTION COUNT DIAGNOSTIC")
print("=" * 90)

print("\nLast 25 lines of collection.log:\n")

for line in lines[-25:]:
    print(line)

print("\n" + "-" * 90)

# Find pytest's own collection summary lines.
summary_lines = [
    line.strip()
    for line in lines
    if re.search(
        r"\b(collected|selected|deselected|error|skipped)\b",
        line,
        flags=re.I
    )
]

print("Pytest collection-summary lines:")

for line in summary_lines[-20:]:
    print(" ", line)

print("\nLines containing pytest-style node IDs:")

node_lines = [
    line.strip()
    for line in lines
    if "::" in line
]

print("Count:", len(node_lines))

# Show unusual collection lines that may represent a test item
# without the normal "::" form.
print("\nPotential non-standard collected items:")

for line in lines:
    s = line.strip()

    if (
        s
        and "::" not in s
        and (
            s.startswith("tests/")
            or s.startswith("tests\\")
            or s.endswith(".py")
        )
    ):
        print(" ", s)

print("\n" + "=" * 90)

In [ ]:
from pathlib import Path
import json
from datetime import datetime

ROOT = Path(r"<LOCAL_WORKSPACE>")
OUT = ROOT / "frozen_protocol"
OUT.mkdir(parents=True, exist_ok=True)

protocol = {
    "protocol_name": "EMSE_Tier3_PerTest_Coverage_Protocol",
    "status": "FROZEN",
    "freeze_timestamp": datetime.now().isoformat(),

    "pilot_subject": "attrs 24.1.0",

    "instrumentation": {
        "framework": "coverage.py via pytest-cov",
        "coverage_scope": "production implementation only",
        "dynamic_context": "pytest test",
        "branch_mode": True,
        "statement_representation": "executed production source lines",
        "branch_representation": "executed coverage.py control-flow arcs"
    },

    "pilot_validation": {
        "collected_test_node_ids": 1334,
        "coverage_context_tests": 1260,
        "tests_without_production_context": 74,
        "unexpected_context_ids": 0,
        "production_files": 19,
        "observed_statement_elements": 2392,
        "observed_branch_arc_elements": 3405,
        "context_mapping": "CONSISTENT"
    },

    "test_universe_policy": (
        "The native eligible regression-test universe is defined independently "
        "of production-code coverage. Tests without measured production coverage "
        "remain recorded in the test-universe provenance and are not silently "
        "treated as nonexistent."
    ),

    "zero_coverage_policy": (
        "Tests with no measured production-code coverage are retained in the "
        "universe record with an empty coverage set. They cannot contribute new "
        "coverage elements during coverage-guided set-cover selection."
    ),

    "collection_skip_policy": (
        "Collection-time skips are recorded separately because pytest may report "
        "them in execution outcome totals even though they do not appear among "
        "successfully collected node IDs."
    ),

    "scope_policy": (
        "Only production implementation files belonging to the frozen release "
        "are included. Test files, virtual environments, and external libraries "
        "are excluded."
    ),

    "integrity_policy": (
        "No production source, upstream test source, or test fixture may be "
        "modified for coverage characterization."
    ),

    "post_freeze_policy": (
        "The coverage representation and zero-coverage handling rules must remain "
        "unchanged across all 12 frozen releases and must not be altered in "
        "response to subsequent reduction results."
    ),

    "next_phase": "batch per-test coverage characterization across 12 releases"
}

path = OUT / "tier3_frozen_per_test_coverage_protocol.json"

path.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print("=" * 90)
print("TIER-3 PER-TEST COVERAGE PROTOCOL — FROZEN")
print("=" * 90)
print("Pilot                 : attrs 24.1.0")
print("Collected tests       : 1334")
print("Coverage contexts     : 1260")
print("Zero/no-context tests : 74")
print("Unexpected contexts   : 0")
print("Statements observed   : 2392")
print("Branch arcs observed  : 3405")
print()
print("Saved:")
print(path)
print()
print("NEXT: BATCH COVERAGE CHARACTERIZATION FOR REMAINING 11 RELEASES")

In [ ]:
from pathlib import Path
import json

p = Path(
    r"<LOCAL_WORKSPACE>\coverage_characterization"
    r"\batch_preflight_failures.json"
)

data = json.loads(
    p.read_text(
        encoding="utf-8"
    )
)

print("=" * 100)
print("TIER-3 COVERAGE BATCH — PREFLIGHT FAILURES")
print("=" * 100)

print("Number of failures:", len(data))

for i, item in enumerate(data, 1):

    print("\n" + "-" * 100)
    print(f"FAILURE {i}")
    print("-" * 100)

    print("Project :", item.get("project"))
    print("Version :", item.get("version"))
    print("Reason  :", item.get("reason"))

    stdout = item.get("stdout", "")
    stderr = item.get("stderr", "")

    if stdout:
        print("\nSTDOUT:")
        print(stdout)

    if stderr:
        print("\nSTDERR:")
        print(stderr)

print("\n" + "=" * 100)

In [ ]:
# ============================================================
# CATTRS 24.1.0 — ADD COVERAGE INSTRUMENTATION ONLY
#
# Purpose:
#   Add pytest-cov to the already frozen historical environment
#   WITHOUT changing pytest, coverage, or any project dependency.
#
# Method:
#   pip install --no-deps pytest-cov==5.0.0
#
# Then verify:
#   - pytest remains 8.0.0
#   - coverage remains 7.4.0
#   - pytest-cov is available
#   - pip check passes
#
# NO source/test/fixture modification
# ============================================================

from pathlib import Path
import subprocess
import json
from datetime import datetime

ROOT = Path(r"<LOCAL_WORKSPACE>")

PYTHON = (
    ROOT
    / "_historical_venvs"
    / "cattrs-24.1.0-locked"
    / "Scripts"
    / "python.exe"
)

OUT = (
    ROOT
    / "coverage_characterization"
    / "cattrs"
    / "24.1.0"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

if not PYTHON.exists():
    raise FileNotFoundError(PYTHON)

print("=" * 100)
print("CATTRS 24.1.0 — COVERAGE INSTRUMENTATION ADDITION")
print("=" * 100)

def run(cmd):
    p = subprocess.run(
        [str(x) for x in cmd],
        text=True,
        capture_output=True
    )

    if p.stdout:
        print(p.stdout)

    if p.stderr:
        print(p.stderr)

    return p


# ------------------------------------------------------------
# 1. RECORD BEFORE STATE
# ------------------------------------------------------------

print("\n1. BEFORE STATE")
print("-" * 100)

before_code = r'''
import importlib.metadata as m

for pkg in [
    "pytest",
    "coverage",
    "pytest-cov",
    "attrs",
    "hypothesis"
]:
    try:
        print(f"{pkg}={m.version(pkg)}")
    except Exception:
        print(f"{pkg}=NOT_INSTALLED")
'''

before = run([
    PYTHON,
    "-c",
    before_code
])

if before.returncode != 0:
    raise RuntimeError(
        "Could not inspect frozen environment."
    )


# ------------------------------------------------------------
# 2. INSTALL ONLY PYTEST-COV
# ------------------------------------------------------------

print("\n2. INSTALLING INSTRUMENTATION ONLY")
print("-" * 100)

install = run([
    PYTHON,
    "-m",
    "pip",
    "install",
    "--no-deps",
    "pytest-cov==5.0.0"
])

if install.returncode != 0:
    raise RuntimeError(
        "pytest-cov installation failed. "
        "Do not install another version yet."
    )


# ------------------------------------------------------------
# 3. VERIFY AFTER STATE
# ------------------------------------------------------------

print("\n3. AFTER STATE")
print("-" * 100)

after = run([
    PYTHON,
    "-c",
    before_code
])

if after.returncode != 0:
    raise RuntimeError(
        "Could not inspect environment after instrumentation."
    )


# ------------------------------------------------------------
# 4. EXPLICIT VERSION ASSERTIONS
# ------------------------------------------------------------

print("\n4. VERSION INTEGRITY CHECK")
print("-" * 100)

check_code = r'''
import importlib.metadata as m

expected = {
    "pytest": "8.0.0",
    "coverage": "7.4.0",
    "pytest-cov": "5.0.0",
    "attrs": "23.1.0",
    "hypothesis": "6.90.0",
}

bad = []

for pkg, want in expected.items():

    got = m.version(pkg)

    print(
        f"{pkg:12s} expected={want:10s} actual={got}"
    )

    if got != want:
        bad.append(
            (pkg, want, got)
        )

if bad:
    raise SystemExit(
        "VERSION_DRIFT=" + repr(bad)
    )

print("VERSION_INTEGRITY=PASS")
'''

check = run([
    PYTHON,
    "-c",
    check_code
])

if check.returncode != 0:
    raise RuntimeError(
        "Historical dependency versions changed. "
        "STOP before coverage."
    )


# ------------------------------------------------------------
# 5. VERIFY PYTEST-COV IMPORT
# ------------------------------------------------------------

print("\n5. PYTEST-COV IMPORT CHECK")
print("-" * 100)

plugin_check = run([
    PYTHON,
    "-c",
    (
        "import pytest_cov; "
        "print('pytest_cov import: PASS')"
    )
])

if plugin_check.returncode != 0:
    raise RuntimeError(
        "pytest-cov still cannot be imported."
    )


# ------------------------------------------------------------
# 6. PIP CHECK
# ------------------------------------------------------------

print("\n6. PIP CHECK")
print("-" * 100)

pip_check = run([
    PYTHON,
    "-m",
    "pip",
    "check"
])

if pip_check.returncode != 0:
    raise RuntimeError(
        "pip check failed after instrumentation."
    )


# ------------------------------------------------------------
# 7. SAVE PROVENANCE RECORD
# ------------------------------------------------------------

record = {
    "project": "cattrs",
    "version": "24.1.0",
    "purpose": "coverage instrumentation only",
    "instrumentation_added": "pytest-cov==5.0.0",
    "installation_mode": "--no-deps",
    "historical_pytest_preserved": "8.0.0",
    "historical_coverage_preserved": "7.4.0",
    "historical_attrs_preserved": "23.1.0",
    "historical_hypothesis_preserved": "6.90.0",
    "source_modified": False,
    "tests_modified": False,
    "fixtures_modified": False,
    "pip_check": "PASS",
    "timestamp": datetime.now().isoformat(),
}

record_path = (
    OUT
    / "coverage_instrumentation_addition.json"
)

record_path.write_text(
    json.dumps(
        record,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CATTRS 24.1.0 — COVERAGE PREFLIGHT READY")
print("=" * 100)

print("pytest       : 8.0.0")
print("coverage     : 7.4.0")
print("pytest-cov   : 5.0.0")
print("attrs        : 23.1.0")
print("hypothesis   : 6.90.0")
print("pip check    : PASS")
print("source edits : NONE")
print("test edits   : NONE")

print("\nSaved:")
print(record_path)

print("\nNEXT:")
print(
    "Rerun the existing Tier-3 batch coverage cell. "
    "Its preflight should now pass and the batch will proceed."
)

## Tier-3 authoritative corrected batch

This is the retained frozen-protocol batch implementation. For **cattrs**, the versioned `src` directory is bound through `PYTHONPATH` for both pytest collection and pytest-cov execution. The existing `--cov-branch`, `--cov-context=test`, `--ignore=bench`, checkpoint/resume, and consistency checks are preserved.

**Run this batch cell when ready. Do not reinstall packages or modify source/tests.**


In [ ]:
# ============================================================
# EMSE TIER-3
# BATCH PER-TEST COVERAGE CHARACTERIZATION
# CORRECTED: cattrs source-tree binding is applied consistently to
#            pytest collection and pytest-cov execution.
#
# Frozen design:
#   4 projects × 3 releases = 12 releases
#   attrs 24.1.0 pilot already COMPLETE
#   This cell processes the remaining 11 releases
#
# Coverage representation:
#   - implementation statements = executed production source lines
#   - branch structure = executed coverage.py control-flow arcs
#   - dynamic pytest contexts
#
# Frozen special rules:
#   cattrs:
#       bench/ excluded
#
#   boltons:
#       tests/test_jsonutils.py::test_reverse_iter_lines
#       excluded consistently from all releases
#
# Safety:
#   - NO package installation
#   - NO source modification
#   - NO test modification
#   - NO fixture modification
#   - checkpoint after every release
#   - safe to rerun after interruption
# ============================================================

from pathlib import Path
import subprocess
import sqlite3
import json
import csv
import os
import re
import shutil
import time
from collections import defaultdict
from datetime import datetime


# ============================================================
# 0. GLOBAL PATHS
# ============================================================

ROOT = Path(r"<LOCAL_WORKSPACE>")

FROZEN_DIR = ROOT / "frozen_protocol"

FROZEN_MATRIX = (
    FROZEN_DIR
    / "tier3_frozen_baseline_matrix.csv"
)

FROZEN_COVERAGE_PROTOCOL = (
    FROZEN_DIR
    / "tier3_frozen_per_test_coverage_protocol.json"
)

OUT_ROOT = (
    ROOT
    / "coverage_characterization"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. VERIFY FROZEN PROTOCOL FILES
# ============================================================

print("=" * 110)
print("EMSE TIER-3 — BATCH PER-TEST COVERAGE CHARACTERIZATION")
print("=" * 110)

if not FROZEN_MATRIX.exists():
    raise FileNotFoundError(
        f"Frozen baseline matrix missing:\n{FROZEN_MATRIX}"
    )

if not FROZEN_COVERAGE_PROTOCOL.exists():
    raise FileNotFoundError(
        f"Frozen coverage protocol missing:\n"
        f"{FROZEN_COVERAGE_PROTOCOL}"
    )

print("Frozen baseline matrix   : PASS")
print("Frozen coverage protocol : PASS")


# ============================================================
# 2. FROZEN RELEASE ORDER
# ============================================================

RELEASES = [

    ("attrs", "24.1.0"),
    ("attrs", "25.3.0"),
    ("attrs", "26.1.0"),

    ("cattrs", "24.1.0"),
    ("cattrs", "25.3.0"),
    ("cattrs", "26.1.0"),

    ("boltons", "24.1.0"),
    ("boltons", "25.0.0"),
    ("boltons", "26.1.0"),

    ("more-itertools", "10.5.0"),
    ("more-itertools", "10.8.0"),
    ("more-itertools", "11.1.0"),
]


# ============================================================
# 3. PROJECT COVERAGE CONFIGURATION
# ============================================================

CONFIG = {

    "attrs": {
        "coverage_target": "src",
        "production_dirs": ["src"],
        "pytest_extra": [],
    },

    "cattrs": {
        "coverage_target": "src",
        "production_dirs": ["src"],
        "pytest_extra": [
            "--ignore=bench"
        ],
    },

    "boltons": {
        "coverage_target": "boltons",
        "production_dirs": ["boltons"],
        "pytest_extra": [
            "--deselect=tests/test_jsonutils.py::test_reverse_iter_lines"
        ],
    },

    "more-itertools": {
        "coverage_target": "more_itertools",
        "production_dirs": ["more_itertools"],
        "pytest_extra": [],
    },
}


# ============================================================
# 4. HELPERS
# ============================================================

def norm_project_name(name):
    return (
        name.lower()
        .replace("-", "")
        .replace("_", "")
    )


def run_capture(cmd, cwd=None, env=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
    )


def read_pyproject_project_name(path):

    pyproject = path / "pyproject.toml"

    if not pyproject.exists():
        return None

    text = pyproject.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    m = re.search(
        r'(?m)^\s*name\s*=\s*["\']([^"\']+)["\']',
        text
    )

    if m:
        return m.group(1)

    return None


# ============================================================
# 5. LOCATE FROZEN PYTHON ENVIRONMENT
# ============================================================

def locate_python(project, version):

    candidates = []

    # Normalized Python 3.12 environments
    candidates.extend([
        ROOT
        / "_baseline_py312"
        / f"{project}-{version}"
        / "Scripts"
        / "python.exe",

        ROOT
        / "_baseline_py312"
        / f"{project.replace('-', '_')}-{version}"
        / "Scripts"
        / "python.exe",
    ])

    # cattrs special provenance environments
    if project == "cattrs":

        if version == "24.1.0":

            candidates.insert(
                0,
                ROOT
                / "_historical_venvs"
                / "cattrs-24.1.0-locked"
                / "Scripts"
                / "python.exe"
            )

        else:

            candidates.insert(
                0,
                ROOT
                / "_git_venvs"
                / f"cattrs-{version}"
                / "Scripts"
                / "python.exe"
            )

    for p in candidates:

        if p.exists():
            return p.resolve()

    # Conservative fallback search
    all_hits = list(
        ROOT.glob("**/Scripts/python.exe")
    )

    project_key = norm_project_name(project)

    hits = []

    for p in all_hits:

        s = str(p).lower()

        if version not in s:
            continue

        if project_key not in norm_project_name(s):
            continue

        if "site-packages" in s:
            continue

        hits.append(p.resolve())

    hits = sorted(set(hits))

    if len(hits) == 1:
        return hits[0]

    print(
        f"\nCould not uniquely identify Python for "
        f"{project} {version}"
    )

    for h in hits:
        print("  ", h)

    raise RuntimeError(
        f"Frozen Python environment unresolved: "
        f"{project} {version}"
    )


# ============================================================
# 6. LOCATE FROZEN SOURCE TREE
# ============================================================

def source_has_project_layout(path, project):

    if not (path / "tests").exists():
        return False

    cfg = CONFIG[project]

    if not any(
        (path / x).exists()
        for x in cfg["production_dirs"]
    ):
        return False

    return True


def locate_source(project, version):

    # Exact cattrs provenance tree
    if project == "cattrs":

        exact = (
            ROOT
            / "_git_exec"
            / "cattrs_equivalence"
            / "20260913_115702"
            / version
        )

        if exact.exists():
            return exact.resolve()

    candidates = []

    metadata_files = (
        list(ROOT.glob("**/pyproject.toml"))
        + list(ROOT.glob("**/setup.py"))
        + list(ROOT.glob("**/setup.cfg"))
    )

    blocked_terms = [
        "_baseline_py312",
        "_historical_venvs",
        "_git_venvs",
        "site-packages",
        "coverage_characterization",
        "frozen_protocol",
    ]

    project_key = norm_project_name(project)

    for meta in metadata_files:

        parent = meta.parent.resolve()

        s = str(parent).lower()

        if any(x in s for x in blocked_terms):
            continue

        if version not in s:
            continue

        if not source_has_project_layout(
            parent,
            project
        ):
            continue

        identity_ok = (
            project_key
            in norm_project_name(str(parent))
        )

        if (
            not identity_ok
            and (parent / "pyproject.toml").exists()
        ):

            declared_name = (
                read_pyproject_project_name(parent)
            )

            if declared_name:
                identity_ok = (
                    norm_project_name(declared_name)
                    == project_key
                )

        if identity_ok:
            candidates.append(parent)

    candidates = sorted(set(candidates))

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:

        preferred = []

        for p in candidates:

            s = str(p).lower()

            if (
                "release" in s
                or "source" in s
                or "acquisition" in s
                or "_git_exec" in s
            ):
                preferred.append(p)

        preferred = sorted(set(preferred))

        if len(preferred) == 1:
            return preferred[0]

    print(
        f"\nSource candidates for "
        f"{project} {version}:"
    )

    for p in candidates:
        print("  ", p)

    raise RuntimeError(
        f"Frozen source tree could not be uniquely resolved "
        f"for {project} {version}."
    )


# ============================================================
# 7. PREFLIGHT
# ============================================================

print("\n" + "=" * 110)
print("PHASE 1 — PREFLIGHT ALL FROZEN RELEASES")
print("=" * 110)

resolved = {}
preflight_failures = []

for project, version in RELEASES:

    if project == "attrs" and version == "24.1.0":
        print(
            f"{project:16s} {version:8s} : "
            f"PILOT ALREADY COMPLETE"
        )
        continue

    try:

        python_exe = locate_python(
            project,
            version
        )

        source_root = locate_source(
            project,
            version
        )

        code = r'''
import sys
print("PYTHON=" + sys.version.replace("\n", " "))

for modname in ["pytest", "coverage", "pytest_cov"]:
    try:
        mod = __import__(modname)
        ver = getattr(mod, "__version__", "unknown")
        print(f"{modname}={ver}")
    except Exception as e:
        print(f"{modname}=MISSING:{e}")
        raise
'''

        p = run_capture(
            [
                python_exe,
                "-c",
                code
            ],
            cwd=source_root
        )

        if p.returncode != 0:

            preflight_failures.append({
                "project": project,
                "version": version,
                "reason":
                    "pytest/coverage/pytest-cov preflight failed",
                "stdout": p.stdout,
                "stderr": p.stderr,
            })

            print(
                f"{project:16s} {version:8s} : "
                f"PREFLIGHT FAIL"
            )

            continue

        resolved[
            (project, version)
        ] = {
            "python": python_exe,
            "source": source_root,
        }

        print(
            f"{project:16s} {version:8s} : "
            f"PREFLIGHT PASS"
        )

        print(
            f"    Python : {python_exe}"
        )

        print(
            f"    Source : {source_root}"
        )

    except Exception as e:

        preflight_failures.append({
            "project": project,
            "version": version,
            "reason": str(e)
        })

        print(
            f"{project:16s} {version:8s} : "
            f"PREFLIGHT FAIL — {e}"
        )


if preflight_failures:

    failure_file = (
        OUT_ROOT
        / "batch_preflight_failures.json"
    )

    failure_file.write_text(
        json.dumps(
            preflight_failures,
            indent=2
        ),
        encoding="utf-8"
    )

    print("\n" + "=" * 110)
    print("STOP — BATCH PREFLIGHT NOT CLEAN")
    print("=" * 110)

    print(
        "No long coverage batch has been started."
    )

    print(
        "Do NOT install or alter anything automatically."
    )

    print("\nSaved:")
    print(failure_file)

    raise RuntimeError(
        "One or more frozen release environments failed "
        "coverage preflight."
    )


print("\nALL REMAINING RELEASES PASSED PREFLIGHT.")


# ============================================================
# 8. EXACT SQLITE EXTRACTION
# ============================================================

def extract_coverage_database(
    coverage_db,
    source_root,
    project,
    collected_ids,
    output_dir
):

    conn = sqlite3.connect(
        str(coverage_db)
    )

    conn.row_factory = sqlite3.Row

    cur = conn.cursor()

    tables = {
        r["name"]
        for r in cur.execute(
            "SELECT name "
            "FROM sqlite_master "
            "WHERE type='table'"
        )
    }

    if "file" not in tables or "context" not in tables:

        conn.close()

        raise RuntimeError(
            "Unexpected coverage SQLite schema."
        )

    prod_roots = [
        (source_root / x).resolve()
        for x in CONFIG[project]["production_dirs"]
    ]

    prod_file_rel = {}

    for row in cur.execute(
        "SELECT id, path FROM file"
    ).fetchall():

        file_id = int(row["id"])

        p = Path(row["path"])

        if not p.is_absolute():
            p = (
                source_root
                / p
            ).resolve()
        else:
            p = p.resolve()

        accepted = False

        for prod_root in prod_roots:

            try:

                p.relative_to(
                    prod_root
                )

                accepted = True
                break

            except Exception:
                pass

        if not accepted:
            continue

        try:

            rel = (
                p.relative_to(
                    source_root
                )
                .as_posix()
            )

        except Exception:

            rel = p.as_posix()

        prod_file_rel[
            file_id
        ] = rel

    if not prod_file_rel:

        conn.close()

        raise RuntimeError(
            "No production implementation files were "
            "recorded in coverage database."
        )

    context_id_to_raw = {}

    for row in cur.execute(
        "SELECT id, context FROM context"
    ).fetchall():

        context_id_to_raw[
            int(row["id"])
        ] = row["context"]

    def context_to_test_id(ctx):

        if not ctx:
            return None

        for suffix in (
            "|run",
            "|setup",
            "|teardown"
        ):

            if ctx.endswith(suffix):
                return ctx[:-len(suffix)]

        if "|" in ctx:

            left, right = ctx.rsplit(
                "|",
                1
            )

            if right in {
                "run",
                "setup",
                "teardown"
            }:
                return left

        return None

    context_id_to_test = {}
    test_to_context_ids = defaultdict(set)

    for cid, raw in context_id_to_raw.items():

        test_id = context_to_test_id(
            raw
        )

        if test_id is None:
            continue

        context_id_to_test[cid] = test_id

        test_to_context_ids[
            test_id
        ].add(cid)

    coverage_context_ids = set(
        test_to_context_ids.keys()
    )

    unexpected_context_ids = (
        coverage_context_ids
        - collected_ids
    )

    if unexpected_context_ids:

        conn.close()

        raise RuntimeError(
            f"{project} has coverage contexts that do not "
            f"map to collected pytest node IDs: "
            f"{len(unexpected_context_ids)}"
        )

    test_lines = defaultdict(set)
    test_arcs = defaultdict(set)

    suite_lines = set()
    suite_arcs = set()

    if "line_bits" in tables:

        from coverage.numbits import (
            numbits_to_nums
        )

        for row in cur.execute(
            """
            SELECT file_id, context_id, numbits
            FROM line_bits
            """
        ).fetchall():

            file_id = int(
                row["file_id"]
            )

            context_id = int(
                row["context_id"]
            )

            if file_id not in prod_file_rel:
                continue

            rel = prod_file_rel[
                file_id
            ]

            nums = numbits_to_nums(
                row["numbits"]
            )

            for lineno in nums:

                element = (
                    f"{rel}:{lineno}"
                )

                suite_lines.add(
                    element
                )

                test_id = (
                    context_id_to_test
                    .get(context_id)
                )

                if test_id is not None:

                    test_lines[
                        test_id
                    ].add(element)

    if "arc" in tables:

        for row in cur.execute(
            """
            SELECT
                file_id,
                context_id,
                fromno,
                tono
            FROM arc
            """
        ).fetchall():

            file_id = int(
                row["file_id"]
            )

            context_id = int(
                row["context_id"]
            )

            if file_id not in prod_file_rel:
                continue

            rel = prod_file_rel[
                file_id
            ]

            a = int(
                row["fromno"]
            )

            b = int(
                row["tono"]
            )

            element = (
                f"{rel}:{a}->{b}"
            )

            suite_arcs.add(
                element
            )

            test_id = (
                context_id_to_test
                .get(context_id)
            )

            if test_id is not None:

                test_arcs[
                    test_id
                ].add(element)

    if not suite_lines and suite_arcs:

        for arc in suite_arcs:

            file_part, pair = (
                arc.rsplit(":", 1)
            )

            a, b = pair.split("->")

            for n in (
                int(a),
                int(b)
            ):

                if n > 0:

                    suite_lines.add(
                        f"{file_part}:{n}"
                    )

        for test_id, arcs in test_arcs.items():

            for arc in arcs:

                file_part, pair = (
                    arc.rsplit(":", 1)
                )

                a, b = pair.split("->")

                for n in (
                    int(a),
                    int(b)
                ):

                    if n > 0:

                        test_lines[
                            test_id
                        ].add(
                            f"{file_part}:{n}"
                        )

    detail = {}
    rows = []

    for test_id in sorted(
        collected_ids
    ):

        context_ids = (
            test_to_context_ids
            .get(
                test_id,
                set()
            )
        )

        contexts = sorted(
            context_id_to_raw[cid]
            for cid in context_ids
        )

        lines = sorted(
            test_lines.get(
                test_id,
                set()
            )
        )

        arcs = sorted(
            test_arcs.get(
                test_id,
                set()
            )
        )

        has_context = (
            test_id
            in coverage_context_ids
        )

        rows.append({
            "test_id":
                test_id,

            "has_coverage_context":
                int(has_context),

            "contexts":
                len(contexts),

            "statement_elements":
                len(lines),

            "branch_arc_elements":
                len(arcs),
        })

        detail[
            test_id
        ] = {
            "has_coverage_context":
                has_context,

            "contexts":
                contexts,

            "statement_elements":
                lines,

            "branch_arc_elements":
                arcs,
        }

    csv_path = (
        output_dir
        / "per_test_coverage.csv"
    )

    with csv_path.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=[
                "test_id",
                "has_coverage_context",
                "contexts",
                "statement_elements",
                "branch_arc_elements",
            ]
        )

        writer.writeheader()
        writer.writerows(rows)

    detail_path = (
        output_dir
        / "per_test_coverage_elements.json"
    )

    detail_path.write_text(
        json.dumps(
            detail,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    no_context = sorted(
        collected_ids
        - coverage_context_ids
    )

    no_context_path = (
        output_dir
        / "collected_tests_without_coverage_context.json"
    )

    no_context_path.write_text(
        json.dumps(
            no_context,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    unexpected_path = (
        output_dir
        / "coverage_contexts_not_in_collection.json"
    )

    unexpected_path.write_text(
        json.dumps(
            sorted(
                unexpected_context_ids
            ),
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    tests_with_lines = sum(
        1
        for x in rows
        if x["statement_elements"] > 0
    )

    tests_with_arcs = sum(
        1
        for x in rows
        if x["branch_arc_elements"] > 0
    )

    summary = {
        "collected_tests":
            len(collected_ids),

        "coverage_context_tests":
            len(coverage_context_ids),

        "tests_without_coverage_context":
            len(no_context),

        "unexpected_context_ids":
            len(unexpected_context_ids),

        "production_files":
            len(prod_file_rel),

        "tests_with_statement_coverage":
            tests_with_lines,

        "tests_with_branch_arc_coverage":
            tests_with_arcs,

        "observed_statement_elements":
            len(suite_lines),

        "observed_branch_arc_elements":
            len(suite_arcs),
    }

    conn.close()

    return summary


# ============================================================
# 9. RUN ONE RELEASE
# ============================================================

def process_release(
    project,
    version,
    python_exe,
    source_root
):

    cfg = CONFIG[
        project
    ]

    out = (
        OUT_ROOT
        / project
        / version
    )

    out.mkdir(
        parents=True,
        exist_ok=True
    )

    complete_marker = (
        out
        / "coverage_characterization_complete.json"
    )

    if complete_marker.exists():

        previous = json.loads(
            complete_marker.read_text(
                encoding="utf-8"
            )
        )

        if previous.get("status") == "PASS":

            print(
                f"\nSKIP {project} {version}: "
                f"already COMPLETE."
            )

            return previous

    print("\n" + "=" * 110)
    print(
        f"PROCESSING {project} {version}"
    )
    print("=" * 110)

    print("Python :", python_exe)
    print("Source :", source_root)

    # --------------------------------------------------------
    # FROZEN EXECUTION ENVIRONMENT
    # cattrs must execute from its versioned source tree.
    # The same environment is used for collection and coverage.
    # --------------------------------------------------------

    env = dict(os.environ)

    if project == "cattrs":

        src_dir = (
            source_root
            / "src"
        ).resolve()

        src_pkg = (
            src_dir
            / "cattrs"
        )

        if not src_pkg.exists():
            raise RuntimeError(
                f"cattrs source package not found: {src_pkg}"
            )

        old_pythonpath = env.get(
            "PYTHONPATH",
            ""
        )

        if old_pythonpath:
            env["PYTHONPATH"] = (
                str(src_dir)
                + os.pathsep
                + old_pythonpath
            )
        else:
            env["PYTHONPATH"] = str(src_dir)

        print(
            "cattrs source binding:",
            src_dir
        )

    # --------------------------------------------------------
    # A. COLLECTION
    # --------------------------------------------------------

    print("\n[1/5] Collecting eligible pytest node IDs...")

    collect_cmd = [
        str(python_exe),
        "-m",
        "pytest",
        "--collect-only",
        "-q",
    ] + cfg["pytest_extra"]

    t_collect = time.perf_counter()

    collect = run_capture(
        collect_cmd,
        cwd=source_root,
        env=env
    )

    collect_runtime = (
        time.perf_counter()
        - t_collect
    )

    collect_text = (
        (collect.stdout or "")
        + "\n"
        + (collect.stderr or "")
    )

    (
        out
        / "collection.log"
    ).write_text(
        collect_text,
        encoding="utf-8"
    )

    if collect.returncode != 0:

        raise RuntimeError(
            f"Collection failed for "
            f"{project} {version}"
        )

    collected_ids = set()

    for raw in collect.stdout.splitlines():

        line = raw.strip()

        if "::" in line:
            collected_ids.add(
                line
            )

    if not collected_ids:

        raise RuntimeError(
            f"No pytest node IDs parsed for "
            f"{project} {version}"
        )

    print(
        "Collected eligible node IDs:",
        len(collected_ids)
    )

    print(
        "Collection runtime:",
        round(
            collect_runtime,
            3
        ),
        "s"
    )

    # --------------------------------------------------------
    # B. COVERAGE RUN
    # --------------------------------------------------------

    print("\n[2/5] Running native suite with dynamic coverage contexts...")

    coverage_file = (
        out
        / ".coverage"
    )

    if coverage_file.exists():
        coverage_file.unlink()

    env["COVERAGE_FILE"] = str(
        coverage_file
    )

    coverage_cmd = [
        str(python_exe),
        "-m",
        "pytest",
        "-q",
        f"--cov={cfg['coverage_target']}",
        "--cov-branch",
        "--cov-context=test",
        "--cov-report=",
    ] + cfg["pytest_extra"]

    print(
        "Command:",
        " ".join(
            coverage_cmd
        )
    )

    print(
        "\nThis release may take several minutes."
    )

    t0 = time.perf_counter()

    coverage_log = (
        out
        / "coverage_run.log"
    )

    with coverage_log.open(
        "w",
        encoding="utf-8"
    ) as log:

        proc = subprocess.run(
            coverage_cmd,
            cwd=str(source_root),
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

    runtime = (
        time.perf_counter()
        - t0
    )

    print(
        "Coverage execution return code:",
        proc.returncode
    )

    print(
        "Coverage runtime:",
        round(runtime, 3),
        "s"
    )

    if proc.returncode != 0:

        tail = (
            coverage_log
            .read_text(
                encoding="utf-8",
                errors="ignore"
            )
            .splitlines()
        )

        print(
            "\nLast 40 log lines:"
        )

        for x in tail[-40:]:
            print(x)

        raise RuntimeError(
            f"Coverage execution failed for "
            f"{project} {version}. "
            f"Completed earlier releases remain checkpointed."
        )

    if not coverage_file.exists():

        raise RuntimeError(
            f"Coverage DB not created for "
            f"{project} {version}"
        )

    # --------------------------------------------------------
    # C. ARCHIVE DB
    # --------------------------------------------------------

    print("\n[3/5] Archiving raw coverage database...")

    archive = (
        out
        / f"{project}_{version}_per_test.coverage"
    )

    shutil.copy2(
        coverage_file,
        archive
    )

    # --------------------------------------------------------
    # D. EXTRACTION
    # --------------------------------------------------------

    print("\n[4/5] Extracting exact per-test structural elements...")

    summary = extract_coverage_database(
        coverage_db=archive,
        source_root=source_root,
        project=project,
        collected_ids=collected_ids,
        output_dir=out,
    )

    if (
        summary["unexpected_context_ids"]
        != 0
    ):

        raise RuntimeError(
            "Unexpected context IDs detected."
        )

    if (
        summary["observed_statement_elements"]
        <= 0
    ):

        raise RuntimeError(
            "No production statements observed."
        )

    if (
        summary["observed_branch_arc_elements"]
        <= 0
    ):

        raise RuntimeError(
            "No production branch arcs observed."
        )

    # --------------------------------------------------------
    # E. CHECKPOINT
    # --------------------------------------------------------

    print("\n[5/5] Saving checkpoint...")

    result = {
        "project":
            project,

        "version":
            version,

        "status":
            "PASS",

        "python":
            str(python_exe),

        "source_root":
            str(source_root),

        "coverage_target":
            cfg["coverage_target"],

        "pytest_extra":
            cfg["pytest_extra"],

        "collected_tests":
            summary[
                "collected_tests"
            ],

        "coverage_context_tests":
            summary[
                "coverage_context_tests"
            ],

        "tests_without_coverage_context":
            summary[
                "tests_without_coverage_context"
            ],

        "unexpected_context_ids":
            summary[
                "unexpected_context_ids"
            ],

        "production_files":
            summary[
                "production_files"
            ],

        "tests_with_statement_coverage":
            summary[
                "tests_with_statement_coverage"
            ],

        "tests_with_branch_arc_coverage":
            summary[
                "tests_with_branch_arc_coverage"
            ],

        "observed_statement_elements":
            summary[
                "observed_statement_elements"
            ],

        "observed_branch_arc_elements":
            summary[
                "observed_branch_arc_elements"
            ],

        "collection_runtime_seconds":
            collect_runtime,

        "coverage_runtime_seconds":
            runtime,

        "coverage_database":
            str(archive),

        "completed_at":
            datetime.now().isoformat(),
    }

    complete_marker.write_text(
        json.dumps(
            result,
            indent=2
        ),
        encoding="utf-8"
    )

    print("\nRESULT:")
    print(
        "  Collected tests       :",
        result["collected_tests"]
    )

    print(
        "  Coverage contexts     :",
        result["coverage_context_tests"]
    )

    print(
        "  No-context tests      :",
        result[
            "tests_without_coverage_context"
        ]
    )

    print(
        "  Production files      :",
        result["production_files"]
    )

    print(
        "  Observed statements   :",
        result[
            "observed_statement_elements"
        ]
    )

    print(
        "  Observed branch arcs  :",
        result[
            "observed_branch_arc_elements"
        ]
    )

    print(
        "  Runtime               :",
        round(runtime, 3),
        "s"
    )

    print(
        f"\nCHECKPOINT COMPLETE: "
        f"{project} {version}"
    )

    return result


# ============================================================
# 10. LOAD COMPLETED ATTRS PILOT
# ============================================================

all_results = []

pilot_summary_path = (
    OUT_ROOT
    / "attrs"
    / "24.1.0"
    / "coverage_summary.json"
)

pilot_consistency_path = (
    OUT_ROOT
    / "attrs"
    / "24.1.0"
    / "coverage_universe_consistency.json"
)

if (
    pilot_summary_path.exists()
    and pilot_consistency_path.exists()
):

    ps = json.loads(
        pilot_summary_path.read_text(
            encoding="utf-8"
        )
    )

    pc = json.loads(
        pilot_consistency_path.read_text(
            encoding="utf-8"
        )
    )

    pilot_result = {
        "project":
            "attrs",

        "version":
            "24.1.0",

        "status":
            "PASS",

        "pilot":
            True,

        "collected_tests":
            pc[
                "collected_tests"
            ],

        "coverage_context_tests":
            pc[
                "coverage_context_tests"
            ],

        "tests_without_coverage_context":
            pc[
                "tests_without_coverage_context"
            ],

        "unexpected_context_ids":
            pc[
                "unexpected_coverage_contexts"
            ],

        "production_files":
            ps[
                "production_files"
            ],

        "tests_with_statement_coverage":
            ps[
                "tests_with_statement_coverage"
            ],

        "tests_with_branch_arc_coverage":
            ps[
                "tests_with_branch_arc_coverage"
            ],

        "observed_statement_elements":
            ps[
                "suite_statement_elements_observed"
            ],

        "observed_branch_arc_elements":
            ps[
                "suite_branch_arc_elements_observed"
            ],
    }

    all_results.append(
        pilot_result
    )

else:

    raise RuntimeError(
        "Completed attrs 24.1.0 pilot artifacts "
        "could not be found."
    )


# ============================================================
# 11. RUN REMAINING 11 RELEASES
# ============================================================

print("\n" + "=" * 110)
print("PHASE 2 — BATCH COVERAGE EXECUTION")
print("=" * 110)

batch_start = time.perf_counter()

for index, (
    project,
    version
) in enumerate(
    RELEASES,
    start=1
):

    if (
        project == "attrs"
        and version == "24.1.0"
    ):
        continue

    info = resolved[
        (project, version)
    ]

    print(
        f"\nFrozen release "
        f"{index}/12: "
        f"{project} {version}"
    )

    result = process_release(
        project=project,
        version=version,
        python_exe=info["python"],
        source_root=info["source"],
    )

    all_results.append(
        result
    )

    checkpoint = (
        OUT_ROOT
        / "tier3_batch_coverage_checkpoint.json"
    )

    checkpoint.write_text(
        json.dumps(
            all_results,
            indent=2
        ),
        encoding="utf-8"
    )


batch_runtime = (
    time.perf_counter()
    - batch_start
)


# ============================================================
# 12. NORMALIZE ALL 12 RESULTS
# ============================================================

final_results = []

for project, version in RELEASES:

    if (
        project == "attrs"
        and version == "24.1.0"
    ):

        match = [
            x for x in all_results
            if (
                x["project"] == project
                and x["version"] == version
            )
        ]

        if not match:
            raise RuntimeError(
                "Pilot result missing from final matrix."
            )

        final_results.append(
            match[0]
        )

        continue

    marker = (
        OUT_ROOT
        / project
        / version
        / "coverage_characterization_complete.json"
    )

    if not marker.exists():

        raise RuntimeError(
            f"Missing completion marker: "
            f"{project} {version}"
        )

    d = json.loads(
        marker.read_text(
            encoding="utf-8"
        )
    )

    if d.get("status") != "PASS":

        raise RuntimeError(
            f"Incomplete result: "
            f"{project} {version}"
        )

    final_results.append(
        d
    )


# ============================================================
# 13. SAVE CONSOLIDATED CSV
# ============================================================

summary_rows = []

for x in final_results:

    summary_rows.append({

        "project":
            x["project"],

        "version":
            x["version"],

        "status":
            x["status"],

        "collected_tests":
            x["collected_tests"],

        "coverage_context_tests":
            x["coverage_context_tests"],

        "tests_without_coverage_context":
            x[
                "tests_without_coverage_context"
            ],

        "unexpected_context_ids":
            x[
                "unexpected_context_ids"
            ],

        "production_files":
            x["production_files"],

        "tests_with_statement_coverage":
            x[
                "tests_with_statement_coverage"
            ],

        "tests_with_branch_arc_coverage":
            x[
                "tests_with_branch_arc_coverage"
            ],

        "observed_statement_elements":
            x[
                "observed_statement_elements"
            ],

        "observed_branch_arc_elements":
            x[
                "observed_branch_arc_elements"
            ],

        "coverage_runtime_seconds":
            x.get(
                "coverage_runtime_seconds",
                None
            ),
    })


summary_csv = (
    OUT_ROOT
    / "tier3_per_test_coverage_summary.csv"
)

with summary_csv.open(
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=
            summary_rows[0].keys()
    )

    writer.writeheader()
    writer.writerows(
        summary_rows
    )


# ============================================================
# 14. SAVE CONSOLIDATED JSON
# ============================================================

summary_json = (
    OUT_ROOT
    / "tier3_per_test_coverage_summary.json"
)

summary_json.write_text(
    json.dumps(
        {
            "status":
                "PASS",

            "projects":
                4,

            "releases":
                12,

            "coverage_protocol":
                "FROZEN",

            "results":
                final_results,

            "batch_runtime_seconds":
                batch_runtime,

            "completed_at":
                datetime.now().isoformat(),
        },
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print("\n" + "=" * 110)
print("TIER-3 PER-TEST COVERAGE CHARACTERIZATION — COMPLETE")
print("=" * 110)

for r in summary_rows:

    print(
        f"{r['project']:16s} "
        f"{r['version']:8s} "
        f"tests={r['collected_tests']:5d} "
        f"contexts={r['coverage_context_tests']:5d} "
        f"noctx={r['tests_without_coverage_context']:4d} "
        f"files={r['production_files']:3d} "
        f"stmt={r['observed_statement_elements']:5d} "
        f"arcs={r['observed_branch_arc_elements']:5d}"
    )

print("\nTotal releases completed :", len(summary_rows))

print(
    "Unexpected context IDs  :",
    sum(
        r["unexpected_context_ids"]
        for r in summary_rows
    )
)

print(
    "Batch active runtime    :",
    round(
        batch_runtime / 60,
        2
    ),
    "minutes"
)

print("\nSaved:")
print(summary_csv)
print(summary_json)

print("\n" + "=" * 110)
print("STATUS: ALL 12 FROZEN RELEASES CHARACTERIZED")
print("NEXT: COVERAGE-GUIDED TEST-SUITE REDUCTION")
print("=" * 110)

In [ ]:
from pathlib import Path

log = Path(
    r"<LOCAL_WORKSPACE>\coverage_characterization"
    r"\attrs\25.3.0\collection.log"
)

if not log.exists():
    raise FileNotFoundError(log)

text = log.read_text(
    encoding="utf-8",
    errors="ignore"
)

lines = text.splitlines()

print("=" * 100)
print("ATTRS 25.3.0 — COLLECTION FAILURE DIAGNOSTIC")
print("=" * 100)

print("\nLast 80 lines:\n")

for line in lines[-80:]:
    print(line)

print("\n" + "-" * 100)

keywords = [
    "ERROR",
    "ImportError",
    "ModuleNotFoundError",
    "UsageError",
    "unrecognized arguments",
    "Traceback",
    "INTERNALERROR",
    "failed",
]

print("Relevant diagnostic lines:\n")

found = False

for line in lines:
    if any(
        k.lower() in line.lower()
        for k in keywords
    ):
        print(line)
        found = True

if not found:
    print("No standard error keyword detected.")

print("\n" + "=" * 100)

In [ ]:
from pathlib import Path
import sqlite3
import os
from datetime import datetime

ROOT = Path(r"<LOCAL_WORKSPACE>")
COVROOT = ROOT / "coverage_characterization"

print("=" * 110)
print("TIER-3 — ZERO-STATEMENT COVERAGE DIAGNOSTIC")
print("=" * 110)

# ------------------------------------------------------------
# 1. Find coverage databases produced by the batch
# ------------------------------------------------------------

dbs = []

for p in COVROOT.rglob("*"):

    if not p.is_file():
        continue

    if (
        p.name == ".coverage"
        or p.name.endswith(".coverage")
    ):
        try:
            dbs.append(
                (p.stat().st_mtime, p)
            )
        except OSError:
            pass

if not dbs:
    raise RuntimeError(
        "No coverage databases found under coverage_characterization."
    )

dbs.sort(reverse=True)

print("\nMost recently modified coverage databases:\n")

for ts, p in dbs[:10]:
    print(
        datetime.fromtimestamp(ts).isoformat(timespec="seconds"),
        " ",
        p
    )

# The failed release should normally own the newest DB.
DB = dbs[0][1]

print("\n" + "-" * 110)
print("DATABASE SELECTED FOR DIAGNOSIS")
print("-" * 110)
print(DB)

# ------------------------------------------------------------
# 2. Infer project/version from directory
# ------------------------------------------------------------

try:
    version = DB.parent.name
    project = DB.parent.parent.name
except Exception:
    project = "UNKNOWN"
    version = "UNKNOWN"

print("Project :", project)
print("Version :", version)

# ------------------------------------------------------------
# 3. Open coverage SQLite database
# ------------------------------------------------------------

conn = sqlite3.connect(str(DB))
conn.row_factory = sqlite3.Row
cur = conn.cursor()

tables = [
    r["name"]
    for r in cur.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type='table' ORDER BY name"
    ).fetchall()
]

print("\nTables:")
for t in tables:
    print(" ", t)

# ------------------------------------------------------------
# 4. Recorded file paths
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("RECORDED COVERAGE FILES")
print("=" * 110)

if "file" not in tables:
    conn.close()
    raise RuntimeError("Coverage DB has no file table.")

files = cur.execute(
    "SELECT id, path FROM file ORDER BY id"
).fetchall()

print("Recorded files:", len(files))

for row in files[:100]:
    print(
        f"{int(row['id']):4d}  {row['path']}"
    )

if len(files) > 100:
    print(
        f"... {len(files)-100} additional files not displayed"
    )

# ------------------------------------------------------------
# 5. Table row counts
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("COVERAGE TABLE COUNTS")
print("=" * 110)

for table in [
    "context",
    "line_bits",
    "arc",
    "file"
]:
    if table in tables:
        n = cur.execute(
            f"SELECT COUNT(*) AS n FROM {table}"
        ).fetchone()["n"]

        print(
            f"{table:15s}: {n}"
        )

# ------------------------------------------------------------
# 6. Context examples
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("CONTEXT EXAMPLES")
print("=" * 110)

if "context" in tables:

    contexts = cur.execute(
        "SELECT id, context "
        "FROM context "
        "ORDER BY id "
        "LIMIT 20"
    ).fetchall()

    for row in contexts:
        print(
            f"{int(row['id']):4d}  {row['context']}"
        )

# ------------------------------------------------------------
# 7. Determine which files actually have line/arc records
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("FILES WITH EXECUTION DATA")
print("=" * 110)

if "line_bits" in tables:

    rows = cur.execute(
        """
        SELECT
            f.path,
            COUNT(*) AS records
        FROM line_bits lb
        JOIN file f
          ON f.id = lb.file_id
        GROUP BY lb.file_id
        ORDER BY records DESC
        """
    ).fetchall()

    print("\nLINE_BITS:")

    if not rows:
        print("  NONE")

    for row in rows[:50]:
        print(
            f"{int(row['records']):6d}  {row['path']}"
        )

if "arc" in tables:

    rows = cur.execute(
        """
        SELECT
            f.path,
            COUNT(*) AS records
        FROM arc a
        JOIN file f
          ON f.id = a.file_id
        GROUP BY a.file_id
        ORDER BY records DESC
        """
    ).fetchall()

    print("\nARCS:")

    if not rows:
        print("  NONE")

    for row in rows[:50]:
        print(
            f"{int(row['records']):6d}  {row['path']}"
        )

# ------------------------------------------------------------
# 8. Coverage-run log tail
# ------------------------------------------------------------

log = DB.parent / "coverage_run.log"

print("\n" + "=" * 110)
print("COVERAGE RUN LOG — LAST 50 LINES")
print("=" * 110)

if log.exists():

    lines = log.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    for line in lines[-50:]:
        print(line)

else:
    print("coverage_run.log not found.")

conn.close()

print("\n" + "=" * 110)
print("DIAGNOSTIC COMPLETE")
print("=" * 110)
print(
    "Do not rerun the coverage suite yet. "
    "Send this output so the extractor can be corrected "
    "against the existing coverage database."
)